# 🚀 Employment Prediction Hackathon — R Testing Kit
This notebook walks you from raw data to your submission, step by step data
**The goal:** for each person in Round 9, predict the *probability* that they are **employed (1)** vs **unemployed (0)**.
Your submissions are scored on **AUC**, so always submit a probability between 0 and 1 — never just a hard 0/1.

## 🛠️ Step 1: Load tools and Data

Before we begin exploring and modelling the data, let’s load the required R packages and import the training and testing datasets. This gives us everything we need to start building our employment prediction model.



In [21]:
# install.packages(c('scales','pROC'))  # uncomment on first run if needed
suppressMessages({
  library(dplyr)
  library(ggplot2)
  library(scales)
})

# Load the competition datasets (adjust the paths to match your Kaggle input folder)
data_dir <- '../../assets/dataset'
train_data <- read.csv(file.path(data_dir, 'train.csv'), stringsAsFactors = FALSE)
test_data  <- read.csv(file.path(data_dir, 'test.csv'),  stringsAsFactors = FALSE)

cat('Training rows:', nrow(train_data), '| columns:', ncol(train_data), '\n')
cat('Testing rows :', nrow(test_data),  '| columns:', ncol(test_data),  '\n')

# The target column in train_data is `employed_status` (1 = employed, 0 = unemployed).
# It is NOT present in test_data — that is what you predict.


Training rows: 16719 | columns: 33 
Testing rows : 3230 | columns: 32 


## Step 2: Statistical Testing & Feature Selection
Before modelling, look at what the data actually contains. We use different tests to determine what we can find out about the data and how variables interact with dependable variable.
> **Sizing tip:** on Kaggle/Jupyter, control chart size with `options(repr.plot.width = ..., repr.plot.height = ...)`
> *before* the plot — not with fixed panel sizes. We set sensible sizes per chart below so nothing gets cropped.


In [2]:
# A tidy plotting frame with readable labels
plot_df <- train_data %>%
  filter(!is.na(employed_status)) %>%
  mutate(Status = factor(employed_status, levels = c(0, 1),
                         labels = c('Unemployed', 'Employed')))

pal <- c('Unemployed' = '#E15759', 'Employed' = '#59A14F')

# A reusable, clean theme
theme_hack <- theme_minimal(base_size = 13) +
  theme(plot.title = element_text(face = 'bold'),
        plot.subtitle = element_text(color = 'grey35'),
        legend.position = 'top')


### 2.1 — ANOVA Table
Test whether the mean of a numerical variable differs significantly between employed and unemployed groups.

In [4]:
# ============================================
# 2.1 ANOVA Table
# ============================================

# Create employment groups for the ANOVA
train_data$employed_status_factor <- factor(
  train_data$employed_status,
  levels = c(0, 1),
  labels = c("Unemployed", "Employed")
)

# Variables to test
numeric_vars <- c(
  "age",
  "days_since_last_obs",
  "total_historical_rounds",
  "tenure_lag",
  "work_readiness_score"
)

# Keep only variables that exist in the dataset
numeric_vars <- numeric_vars[
  numeric_vars %in% names(train_data)
]

# Run ANOVA for each numerical variable
for (variable in numeric_vars) {

  formula <- as.formula(
    paste(variable, "~ employed_status_factor")
  )

  anova_model <- aov(
    formula,
    data = train_data
  )

  cat("\n========================================\n")
  cat("ANOVA Test:", variable, "\n")
  cat("========================================\n")

  print(summary(anova_model))
}


ANOVA Test: age 
                          Df Sum Sq Mean Sq F value   Pr(>F)    
employed_status_factor     1    897   896.9   48.73 3.05e-12 ***
Residuals              16645 306379    18.4                     
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
72 observations deleted due to missingness

ANOVA Test: days_since_last_obs 
                         Df   Sum Sq Mean Sq F value Pr(>F)
employed_status_factor    1     1361    1361   0.138   0.71
Residuals              5260 51709437    9831               
11457 observations deleted due to missingness

ANOVA Test: total_historical_rounds 
                          Df Sum Sq Mean Sq F value Pr(>F)    
employed_status_factor     1     34   34.36   79.58 <2e-16 ***
Residuals              16651   7189    0.43                   
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
66 observations deleted due to missingness

ANOVA Test: tenure_lag 
                         Df    Sum Sq  Mean Sq F value 

**What it means:** the classes are imbalanced (~1 in 3 employed). A model that predicts 'unemployed' for everyone would look ~68% 'accurate' but is useless — this is exactly why the competition uses **AUC**, which rewards ranking people by their probability of employment rather than raw accuracy.

## Generic Anova table 

In [ ]:
# ============================================
# 2.1 ANOVA Table
# ============================================

# Create employment groups for the ANOVA
train_data$employed_status_factor <- factor(
  train_data$employed_status,
  levels = c(0, 1),
  labels = c("Unemployed", "Employed")
)

# Variables to test
numeric_vars <- c(
  "age",
  "days_since_last_obs",
  "total_historical_rounds",
  "tenure_lag",
  "work_readiness_score"
)

# Keep only variables that exist in the dataset
numeric_vars <- numeric_vars[
  numeric_vars %in% names(train_data)
]

# Run ANOVA for each numerical variable
for (variable in numeric_vars) {

  formula <- as.formula(
    paste(variable, "~ employed_status_factor")
  )

  anova_model <- aov(
    formula,
    data = train_data
  )

  cat("\n========================================\n")
  cat("ANOVA Test:", variable, "\n")
  cat("========================================\n")

  print(summary(anova_model))
}


ANOVA Test: age 
                          Df Sum Sq Mean Sq F value   Pr(>F)    
employed_status_factor     1    897   896.9   48.73 3.05e-12 ***
Residuals              16645 306379    18.4                     
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
6 observations deleted due to missingness

ANOVA Test: days_since_last_obs 
                         Df   Sum Sq Mean Sq F value Pr(>F)
employed_status_factor    1     1361    1361   0.138   0.71
Residuals              5260 51709437    9831               
11391 observations deleted due to missingness

ANOVA Test: total_historical_rounds 
                          Df Sum Sq Mean Sq F value Pr(>F)    
employed_status_factor     1     34   34.36   79.58 <2e-16 ***
Residuals              16651   7189    0.43                   
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

ANOVA Test: tenure_lag 
                         Df    Sum Sq  Mean Sq F value  Pr(>F)    
employed_status_factor    1 1.69

### 2.2 — Stepwise Regression
Use stepwise selection to identify variables that contribute most strongly to the model.

In [5]:
# ============================================
# 2.2 Stepwise Logistic Regression
# ============================================

# Candidate predictor variables
candidate_vars <- c(
  "age",
  "gender",
  "race",
  "sa_citizen",
  "province",
  "education_level",
  "education_schooling_grade_twelve_equivalent",
  "institution_type",
  "school_quintile",
  "education_field",
  "employed_lag",
  "status_lag",
  "status_broad_lag",
  "tenure_lag",
  "total_historical_rounds",
  "days_since_last_obs",
  "work_readiness_score"
)

# Keep only columns that exist
candidate_vars <- candidate_vars[
  candidate_vars %in% names(train_data)
]

# --------------------------------------------
# Create modelling dataset
# --------------------------------------------

model_vars <- c(
  "employed_status",
  candidate_vars
)

stepwise_data <- train_data[, model_vars, drop = FALSE]

# Remove rows with missing values
stepwise_data <- na.omit(stepwise_data)

cat("Rows available for modelling:",
    nrow(stepwise_data), "\n")

# --------------------------------------------
# Remove predictors with only one level
# AFTER removing missing values
# --------------------------------------------

valid_vars <- c()

cat("\nChecking predictor variables:\n")

for (variable in candidate_vars) {

  # Number of unique values remaining
  # after NA rows have been removed
  number_of_levels <- length(
    unique(stepwise_data[[variable]])
  )

  if (number_of_levels >= 2) {

    valid_vars <- c(valid_vars, variable)

    cat("✓", variable,
        "- ", number_of_levels, " levels\n")

  } else {

    cat("✗", variable,
        "- ONLY ", number_of_levels,
        " level; removed\n")
  }
}

# --------------------------------------------
# Check target variable
# --------------------------------------------

target_levels <- unique(stepwise_data$employed_status)

cat("\nEmployment status levels:\n")
print(target_levels)

if (length(target_levels) < 2) {

  stop(
    "Stepwise regression cannot run because ",
    "employed_status has fewer than 2 levels."
  )
}

# --------------------------------------------
# Convert categorical variables to factors
# --------------------------------------------

categorical_vars <- c(
  "gender",
  "race",
  "sa_citizen",
  "province",
  "education_level",
  "education_schooling_grade_twelve_equivalent",
  "institution_type",
  "education_field",
  "employed_lag",
  "status_lag",
  "status_broad_lag"
)

categorical_vars <- intersect(
  categorical_vars,
  valid_vars
)

for (variable in categorical_vars) {
  stepwise_data[[variable]] <- factor(
    stepwise_data[[variable]]
  )
}

# --------------------------------------------
# Build logistic regression formula
# --------------------------------------------

full_formula <- as.formula(
  paste(
    "employed_status ~",
    paste(valid_vars, collapse = " + ")
  )
)

cat("\n========================================\n")
cat("Full Logistic Regression Model\n")
cat("========================================\n")

print(full_formula)

# --------------------------------------------
# Fit full model
# --------------------------------------------

full_model <- glm(
  full_formula,
  data = stepwise_data,
  family = binomial()
)

summary(full_model)

# --------------------------------------------
# Stepwise selection
# --------------------------------------------

cat("\n========================================\n")
cat("Stepwise Regression\n")
cat("========================================\n")

stepwise_model <- step(
  full_model,
  direction = "both",
  trace = TRUE
)

# --------------------------------------------
# Final model
# --------------------------------------------

cat("\n========================================\n")
cat("Final Stepwise Model\n")
cat("========================================\n")

summary(stepwise_model)

# --------------------------------------------
# Selected variables
# --------------------------------------------

cat("\n========================================\n")
cat("Selected Variables\n")
cat("========================================\n")

print(names(coef(stepwise_model)))

Rows available for modelling: 968 

Checking predictor variables:
✓ age -  19  levels
✓ gender -  2  levels
✓ race -  4  levels
✗ sa_citizen - ONLY  1  level; removed
✓ province -  9  levels
✓ education_level -  4  levels
✗ institution_type - ONLY  1  level; removed
✓ school_quintile -  6  levels
✗ education_field - ONLY  1  level; removed
✓ employed_lag -  2  levels
✓ status_lag -  4  levels
✓ status_broad_lag -  4  levels
✓ tenure_lag -  279  levels
✓ total_historical_rounds -  4  levels
✓ days_since_last_obs -  207  levels
✓ work_readiness_score -  685  levels

Employment status levels:
[1] 1 0

Full Logistic Regression Model
employed_status ~ age + gender + race + province + education_level + 
    school_quintile + employed_lag + status_lag + status_broad_lag + 
    tenure_lag + total_historical_rounds + days_since_last_obs + 
    work_readiness_score



Call:
glm(formula = full_formula, family = binomial(), data = stepwise_data)

Coefficients: (4 not defined because of singularities)
                                         Estimate Std. Error z value Pr(>|z|)
(Intercept)                            -1.975e+00  8.527e-01  -2.317 0.020526
age                                     3.871e-02  2.170e-02   1.784 0.074409
genderMale                              7.914e-01  1.471e-01   5.380 7.44e-08
raceColoured                            8.235e-01  3.723e-01   2.212 0.026988
raceIndian                              1.047e+00  1.459e+00   0.717 0.473150
raceWhite                              -1.333e+01  5.354e+02  -0.025 0.980144
provinceFree State                     -1.405e-01  3.795e-01  -0.370 0.711158
provinceGauteng                        -2.062e-02  2.899e-01  -0.071 0.943287
provinceKwaZulu-Natal                  -4.668e-01  3.234e-01  -1.443 0.148890
provinceLimpopo                        -1.229e-02  3.730e-01  -0.033 0.973714
province


Stepwise Regression
Start:  AIC=1184.98
employed_status ~ age + gender + race + province + education_level + 
    school_quintile + employed_lag + status_lag + status_broad_lag + 
    tenure_lag + total_historical_rounds + days_since_last_obs + 
    work_readiness_score


Step:  AIC=1184.98
employed_status ~ age + gender + race + province + education_level + 
    school_quintile + employed_lag + status_lag + tenure_lag + 
    total_historical_rounds + days_since_last_obs + work_readiness_score


Step:  AIC=1184.98
employed_status ~ age + gender + race + province + education_level + 
    school_quintile + status_lag + tenure_lag + total_historical_rounds + 
    days_since_last_obs + work_readiness_score

                          Df Deviance    AIC
- province                 8   1143.0 1177.0
- education_level          3   1137.7 1181.7
- days_since_last_obs      1   1135.3 1183.3
- total_historical_rounds  1   1135.5 1183.5
- tenure_lag               1   1136.0 1184.0
- work_readiness


Call:
glm(formula = employed_status ~ gender + race + school_quintile + 
    status_lag + work_readiness_score, family = binomial(), data = stepwise_data)

Coefficients:
                                 Estimate Std. Error z value Pr(>|z|)    
(Intercept)                       -0.8771     0.3374  -2.600  0.00933 ** 
genderMale                         0.8001     0.1439   5.560 2.69e-08 ***
raceColoured                       0.9356     0.3559   2.628  0.00858 ** 
raceIndian                         0.7745     1.4385   0.538  0.59033    
raceWhite                        -12.5321   535.4112  -0.023  0.98133    
school_quintile                    0.1126     0.0579   1.945  0.05172 .  
status_lagunemployed              -1.0089     0.2576  -3.916 8.99e-05 ***
status_lagwage and self employed  -0.8504     0.7902  -1.076  0.28187    
status_lagwage employed           -0.1946     0.2894  -0.673  0.50122    
work_readiness_score               0.5447     0.3793   1.436  0.15097    
---
Signif. cod


Selected Variables
 [1] "(Intercept)"                      "genderMale"                      
 [3] "raceColoured"                     "raceIndian"                      
 [5] "raceWhite"                        "school_quintile"                 
 [7] "status_lagunemployed"             "status_lagwage and self employed"
 [9] "status_lagwage employed"          "work_readiness_score"            


In [ ]:
# Variables selected by stepwise regression

selected_variables <- names(
  coef(stepwise_model)
)

selected_variables

[1] "(Intercept)"                      "genderMale"                      
 [3] "raceColoured"                     "raceIndian"                      
 [5] "raceWhite"                        "school_quintile"                 
 [7] "status_lagunemployed"             "status_lagwage and self employed"
 [9] "status_lagwage employed"          "work_readiness_score"

**What it means:** the lag columns are powerful *when present* but empty for most people. Decide deliberately how to handle those NAs — dropping them throws away most of your data. A common trick (used in Step 5) is to treat 'missing history' as its own category rather than discarding the row.

## 📋 2.3 Item Reliability Analysis (IRA)

Item Reliability Analysis evaluates whether multiple items intended to measure the same underlying concept are consistent with one another. In this case, we are interested in whether the available work-readiness assessment items provide a reliable measure of work readiness.

We will use **Cronbach's alpha** to measure internal consistency. A higher alpha generally indicates that the items are measuring a similar underlying construct.

**Your task:** Examine the reliability output and identify whether the items appear to form a reliable scale that could be useful for employment prediction.


In [7]:
# ============================================
# 2.3 Item Reliability Analysis (IRA)
# ============================================

# Install if needed:
# install.packages("psych")

library(psych)

# ------------------------------------------------
# Find potential work-readiness item variables
# ------------------------------------------------

# Look for columns containing "work_readiness"
work_readiness_items <- names(train_data)[
  grepl("work_readiness", names(train_data), ignore.case = TRUE)
]

cat("Potential work-readiness variables:\n")
print(work_readiness_items)


# ------------------------------------------------
# Remove the overall score if it exists
# ------------------------------------------------

work_readiness_items <- work_readiness_items[
  work_readiness_items != "work_readiness_score"
]

cat("\nItems used for reliability analysis:\n")
print(work_readiness_items)


# ------------------------------------------------
# Check whether enough individual items exist
# ------------------------------------------------

if (length(work_readiness_items) >= 2) {
  
  # Select the item columns
  reliability_data <- train_data[
    , work_readiness_items,
    drop = FALSE
  ]
  
  # Convert columns to numeric where possible
  reliability_data <- data.frame(
    lapply(reliability_data, function(x) {
      as.numeric(as.character(x))
    })
  )
  
  # Run Cronbach's Alpha
  reliability_result <- psych::alpha(
    reliability_data,
    check.keys = TRUE
  )
  
  # Display the complete reliability output
  print(reliability_result)
  
  # Display Cronbach's alpha specifically
  cat("\n----------------------------------------\n")
  cat("Cronbach's Alpha:", reliability_result$total$raw_alpha, "\n")
  cat("----------------------------------------\n")
  
  # Display item-level statistics
  cat("\nItem Statistics:\n")
  print(reliability_result$item.stats)
  
  # Display alpha if each item is removed
  cat("\nAlpha if Item is Deleted:\n")
  print(reliability_result$alpha.drop)
  
} else {
  
  cat("\nNot enough individual work-readiness items were found.\n")
  cat("At least two items are required for Item Reliability Analysis.\n")
  cat("The dataset currently appears to contain only the overall\n")
  cat("work_readiness_score rather than individual assessment items.\n")
}

Installing package into 'C:/Users/27823/AppData/Local/R/win-library/4.4'
(as 'lib' is unspecified)



also installing the dependencies 'mnormt', 'GPArotation'





  There are binary versions available but the source versions are later:
              binary   source needs_compilation
GPArotation 2025.3-1 2026.8-1             FALSE
psych          2.6.3    2.6.5             FALSE

package 'mnormt' successfully unpacked and MD5 sums checked

The downloaded binary packages are in
	C:\Users\27823\AppData\Local\Temp\RtmpsLGHpL\downloaded_packages


installing the source packages 'GPArotation', 'psych'



Attaching package: 'psych'


The following objects are masked from 'package:scales':

    alpha, rescale


The following objects are masked from 'package:ggplot2':

    %+%, alpha




Potential work-readiness variables:
[1] "work_readiness_score"

Items used for reliability analysis:
character(0)

Not enough individual work-readiness items were found.
At least two items are required for Item Reliability Analysis.
The dataset currently appears to contain only the overall
work_readiness_score rather than individual assessment items.


**What it means:** someone employed last round is more likely to be employed now than someone who was unemployed. This 'stickiness' is part of the story — but as you'll see, it is not the whole picture, and turning it into leaderboard gains takes care.

### 2.4 — Chi-Square Test
Test whether categorical variables are significantly associated with employed_status.

In [8]:
# 2.4 Chi-Square Tests
# Test whether categorical variables are associated with employment status

categorical_vars <- c(
  "gender",
  "race",
  "sa_citizen",
  "province",
  "education_level",
  "education_schooling_grade_twelve_equiv",
  "institution_type",
  "education_field",
  "status_lag",
  "status_broad_lag"
)

# Keep only variables that exist
categorical_vars <- categorical_vars[
  categorical_vars %in% names(train_data)
]

for (variable in categorical_vars) {
  
  # Create contingency table
  contingency_table <- table(
    train_data[[variable]],
    train_data$employed_status
  )
  
  # Run chi-square test
  chi_test <- chisq.test(contingency_table)
  
  cat("\n========================================\n")
  cat("Chi-Square Test:", variable, "\n")
  cat("========================================\n")
  
  print(contingency_table)
  print(chi_test)
}


Chi-Square Test: gender 
        
            0    1
  Female 7077 2370
  Male   4298 2908

	Pearson's Chi-squared test with Yates' continuity correction

data:  contingency_table
X-squared = 439.46, df = 1, p-value < 2.2e-16



Warning message in chisq.test(contingency_table):
"Chi-squared approximation may be incorrect"



Chi-Square Test: race 
               
                    0     1
                   20    10
  African/Black 10731  4855
  Asian             4     1
  Coloured        544   349
  Indian           47    41
  Other            17     5
  Unspecified       0     1
  White            12    16

	Pearson's Chi-squared test

data:  contingency_table
X-squared = 45.368, df = 7, p-value = 1.16e-07



Warning message in chisq.test(contingency_table):
"Chi-squared approximation may be incorrect"



Chi-Square Test: sa_citizen 
       
            0     1
  FALSE     9     5
  TRUE  11366  5273

	Pearson's Chi-squared test with Yates' continuity correction

data:  contingency_table
X-squared = 0.0013041, df = 1, p-value = 0.9712


Chi-Square Test: province 
               
                   0    1
                  19    9
  Eastern Cape   999  416
  Free State     700  278
  Gauteng       3889 1941
  KwaZulu-Natal 1978  892
  Limpopo       1038  444
  Mpumalanga     847  382
  North West     705  277
  Northern Cape  216   92
  Western Cape   984  547

	Pearson's Chi-squared test

data:  contingency_table
X-squared = 35.427, df = 9, p-value = 5.009e-05



...


Chi-Square Test: education_schooling_grade_twelve_equiv 
       
           0    1
        2508 1139
  0        4    3
  1       33   22
  FALSE  996  403
  TRUE  7834 3711

	Pearson's Chi-squared test

data:  contingency_table
X-squared = 8.9842, df = 4, p-value = 0.0615



Warning message in chisq.test(contingency_table):
"Chi-squared approximation may be incorrect"



Chi-Square Test: institution_type 
                                      
                                          0    1
                                       9318 4386
  Comprehensive University              418  188
  Private College                       278  124
  Private Higher Education Institution  143   73
  Public College                        697  271
  Public University                     200   97
  Tech Accelerator                        2    2
  Trade School                           62   21
  University of Technology              257  116

	Pearson's Chi-squared test

data:  contingency_table
X-squared = 9.8039, df = 8, p-value = 0.2791



Warning message in chisq.test(contingency_table):
"Chi-squared approximation may be incorrect"



Chi-Square Test: education_field 
                                                         
                                                             0    1
                                                          7327 3299
  Accounting and bookkeeping                               104   51
  Administration                                           318  136
  Advertising                                               10    2
  Agriculture                                               74   34
  Aquaculture and fisheries                                  3    0
  Architecture                                               3    0
  Arts and crafts                                           17    8
  Auditing                                                  10    6
  Automation and testing                                     3    3
  Automotive and transport equipment manufacturing           5    3
  Automotive technician                                      2    1
  Aviation and maritime

## 🧪 2.5 Statistical Test Template

Use this template to investigate another relationship in the dataset.

Your test should:

1. State what relationship you are investigating.
2. Identify the variables involved.
3. Run an appropriate statistical test.
4. Record the important statistic and p-value.
5. Decide whether the result is statistically significant.
6. Explain what the result means for employment prediction.

In [9]:
# ============================================
# 2.5 Statistical Test Template
# ============================================

# Variable being investigated
variable_1 <- "REPLACE_WITH_VARIABLE"

# Target variable
variable_2 <- "employed_status"

# Your statistical test here
# Example:
#
# test_result <- chisq.test(
#   table(train_data[[variable_1]], train_data[[variable_2]])
# )
#
# print(test_result)


# Record the important result
# statistic <- ...
# p_value <- ...


# Your conclusion
cat("Variable tested:", variable_1, "\n")
cat("Target variable:", variable_2, "\n")
cat("Test statistic: [SUBMIT VALUE]\n")
cat("P-value: [SUBMIT VALUE]\n")

Variable tested: REPLACE_WITH_VARIABLE 
Target variable: employed_status 
Test statistic: [SUBMIT VALUE]
P-value: [SUBMIT VALUE]


**What it means:** if the boxes overlap heavily, the score alone is a weak separator — and adding weak features to a model can even *hurt* it. Part of the challenge is finding which combinations genuinely carry signal.

2.6 ANOVCA Table

In [10]:
# ============================================
# 2.6 ANOVCA Table
# ============================================

# Create employment groups for the ANOVA
train_data$employed_status_factor <- factor(
  train_data$employed_status,
  levels = c(0, 1),
  labels = c("Unemployed", "Employed")
)

# Variables to test
numeric_vars <- c(
  "age",
  "days_since_last_obs",
  "total_historical_rounds",
  "tenure_lag",
  "work_readiness_score"
)

# Keep only variables that exist in the dataset
numeric_vars <- numeric_vars[
  numeric_vars %in% names(train_data)
]

# Run ANOVA for each numerical variable
for (variable in numeric_vars) {

  formula <- as.formula(
    paste(variable, "~ employed_status_factor")
  )

  anova_model <- aov(
    formula,
    data = train_data
  )

  cat("\n========================================\n")
  cat("ANOVA Test:", variable, "\n")
  cat("========================================\n")

  print(summary(anova_model))
}


ANOVA Test: age 
                          Df Sum Sq Mean Sq F value   Pr(>F)    
employed_status_factor     1    897   896.9   48.73 3.05e-12 ***
Residuals              16645 306379    18.4                     
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
72 observations deleted due to missingness

ANOVA Test: days_since_last_obs 
                         Df   Sum Sq Mean Sq F value Pr(>F)
employed_status_factor    1     1361    1361   0.138   0.71
Residuals              5260 51709437    9831               
11457 observations deleted due to missingness

ANOVA Test: total_historical_rounds 
                          Df Sum Sq Mean Sq F value Pr(>F)    
employed_status_factor     1     34   34.36   79.58 <2e-16 ***
Residuals              16651   7189    0.43                   
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
66 observations deleted due to missingness

ANOVA Test: tenure_lag 
                         Df    Sum Sq  Mean Sq F value 

## Round and Observation Variables

The initial analysis suggests that `lag_round`, `days_since_last_obs`, and `current_round` do not show a statistically significant relationship with employment status. However, these variables should be tested further before being completely excluded from the predictive model.

A particularly strong relationship was observed for `employed_lag`. The very large F-statistic and extremely small p-value indicate a strong association between previous employment status and current employment status. This supports the earlier observation that **someone who was employed previously has a high likelihood of being employed in the future**.

## `employed_lag` and Missing History

`employed_lag` contains a substantial amount of missing data:

* Missing: **11,451 / 16,719 observations**
* Approximately **68.5%** of the values are missing.

This missingness is important because a missing value does not necessarily mean that the participant was unemployed. Instead, it may indicate that **there is no previous observation or employment history available**.

Therefore, rather than treating missing values as ordinary missing data, we can separate the information into two variables:

* **`employed_lag`** — the previous employment status when historical information exists.
* **`employed_lag_missing`** — a binary indicator showing whether previous employment information is missing.

For example:

```text
No previous history
    ↓
employed_lag_missing = 1
employed_lag = NA

Previous history exists
    ↓
employed_lag_missing = 0
employed_lag = 0 or 1
```

This allows the model to distinguish between **previously unemployed**, **previously employed**, and **having no previous employment history**, rather than incorrectly treating missing history as unemployment.


In [18]:
    train_data$employed_lag_missing <- ifelse(
    is.na(train_data$employed_lag), 1, 0)

    
    test_data$employed_lag_missing <- ifelse(
    is.na(test_data$employed_lag), 1, 0)


  

## `employed_lag_missing`

`employed_lag_missing` is a new binary variable that indicates whether the `employed_lag` value is missing.

* **`1`** → `employed_lag` is missing
* **`0`** → `employed_lag` is available

This can be useful for modelling because the **absence of previous employment information may itself contain predictive information**. The model can therefore distinguish between participants with a recorded employment history and those without one.


## `work_readiness_score` and `wr_filled`

`work_readiness_score` and `wr_filled` provide related information about work readiness. Using both directly as predictors may introduce redundancy.

Instead, we can use `wr_filled` as a **binary indicator** showing whether a work-readiness score is available, while `work_readiness_score` contains the actual score when it is present.

This allows us to distinguish between:

* **`wr_filled = 1`** → work-readiness score is available
* **`wr_filled = 0`** → work-readiness score is missing

The missingness itself may contain useful information for predicting employment, so we can test it separately rather than treating the two variables as identical predictors.


In [ ]:
train_data$wr_missing <- ifelse(
  is.na(train_data$work_readiness_score), 1, 0
)

test_data$wr_missing <- ifelse(
  is.na(test_data$work_readiness_score), 1, 0
)

## Investigation of the catergorical variables:

status_lag
status_broad_lag
education_level
education_field
institution_type
gender
province
race



In [17]:
# Chi-squared test for categorical variables
chisq.test(
  table(
    train_data$status_lag,
    train_data$employed_status
  )
)

# Chi-squared test for status_broad_lag vs employed_status
chisq.test(
  table(
    train_data$status_broad_lag,
    train_data$employed_status
  )
)

# Chi-squared test for education_level vs employed_status
chisq.test(
  table(
    train_data$education_level,
    train_data$employed_status
  )
)


	Pearson's Chi-squared test

data:  table(train_data$status_lag, train_data$employed_status)
X-squared = 485.05, df = 5, p-value < 2.2e-16



	Pearson's Chi-squared test

data:  table(train_data$status_broad_lag, train_data$employed_status)
X-squared = 471.29, df = 6, p-value < 2.2e-16


Warning message in chisq.test(table(train_data$education_level, train_data$employed_status)):
"Chi-squared approximation may be incorrect"



	Pearson's Chi-squared test

data:  table(train_data$education_level, train_data$employed_status)
X-squared = 44.366, df = 35, p-value = 0.1333


## Conclusion of the chi-squared test 
| Variable | χ² | df | p-value | Result |
|---|---:|---:|---:|---|
| status_lag | 485.05 | 5 | < 2.2e-16 | Extremely strong |
| status_broad_lag | 471.29 | 6 | < 2.2e-16 | Extremely strong |
| education_level | 44.37 | 35 | 0.1333 | Not significant |


**education_level**
We don't have enough evidence to say education level is associated with employment status in this dataset.

However, I wouldn't immediately remove it. As the p-value is not too high above 0.1 so it still play a factor as logically education still a deciding factor especially NQF levels.

## 🏗️ Step 3: A first baseline model
We predict a binary outcome, so we start with **logistic regression**. For this first attempt we deliberately
This sets a (low) floor — and shows that simply throwing variables at the model is not enough.

In [20]:
# Fill missing work readiness with the training median so every row has a value
wr_median <- median(train_data$work_readiness_score, na.rm = TRUE)
train_data$wr_filled <- ifelse(is.na(train_data$work_readiness_score), wr_median, train_data$work_readiness_score)
test_data$wr_filled  <- ifelse(is.na(test_data$work_readiness_score),  wr_median, test_data$work_readiness_score)

# Treat citizenship as a category
train_data$sa_citizen_chr <- as.character(train_data$sa_citizen)
test_data$sa_citizen_chr  <- as.character(test_data$sa_citizen)

baseline_model <- glm(employed_status ~ wr_filled + sa_citizen_chr,
                      data = train_data, family = binomial)

summary(baseline_model)



Call:
glm(formula = employed_status ~ wr_filled + sa_citizen_chr, family = binomial, 
    data = train_data)

Coefficients:
                   Estimate Std. Error z value Pr(>|z|)    
(Intercept)         -1.0516     0.5610  -1.875   0.0609 .  
wr_filled            0.9161     0.1061   8.632   <2e-16 ***
sa_citizen_chrTRUE  -0.1441     0.5586  -0.258   0.7964    
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

(Dispersion parameter for binomial family taken to be 1)

    Null deviance: 20801  on 16652  degrees of freedom
Residual deviance: 20726  on 16650  degrees of freedom
  (66 observations deleted due to missingness)
AIC: 20732

Number of Fisher Scoring iterations: 4


Description of all the variables

In [ ]:
variable_check <- data.frame(
  variable = names(train_data),
  type = sapply(train_data, function(x) class(x)[1]),
  unique_values = sapply(train_data, function(x) length(unique(x))),
  missing = sapply(train_data, function(x) sum(is.na(x))),
  missing_percent = sapply(train_data, function(x) mean(is.na(x)) * 100)
)

variable_check

,variable,type,unique_values,missing,missing_percent
,<chr>,<chr>,<int>,<int>,<dbl>
anonymised_id,anonymised_id,character,16719,0,0.00000000
lag_round,lag_round,integer,8,11449,68.47897602
current_round,current_round,integer,8,0,0.00000000
days_since_last_obs,days_since_last_obs,integer,287,11449,68.47897602
total_historical_rounds,total_historical_rounds,integer,5,0,0.00000000
survey_date,survey_date,character,204,0,0.00000000
sample,sample,character,24,0,0.00000000
sample_first,sample_first,character,23,0,0.00000000
employed_status,employed_status,integer,3,66,0.39476045


## 🔮 Step 4: Predict and build a submission
For AUC we submit **probabilities**, not hard labels. The block below writes a valid `submission_baseline.csv`.
Upload it on the competition's Submit page to see your public leaderboard score.

In [ ]:
# Make sure the test set has the same prepared column the model was trained on
test_data$sa_citizen_chr <- as.character(test_data$sa_citizen)

# Predicted PROBABILITY of employment (0-1)
test_prob <- predict(baseline_model, newdata = test_data, type = 'response')

submission <- data.frame(anonymised_id = test_data$anonymised_id,
                         employed_status = test_prob)
write.csv(submission, 'submission_baseline.csv', row.names = FALSE, quote = FALSE)
cat('Saved submission_baseline.csv with', nrow(submission), 'rows\n')
head(submission)


Saved submission_baseline.csv with 3230 rows


,anonymised_id,employed_status
,<chr>,<dbl>
1,ID_10,0.3147682
2,ID_100,0.3147682
3,ID_1000,0.3147682
4,ID_1001,0.3147682
5,ID_1002,0.3147682
6,ID_1003,0.3147682


## 🔄 Step 5: Choose better features and watch the score improve
The baseline features barely helped. Now we swap in features that actually carry signal:
`gender` and `education_level`, while keeping `work_readiness_score`. Same model, better inputs — and the score improves.
This is the core lesson: **which** variables you choose matters more than how many.

In [ ]:
# Fill missing work readiness (recompute to be safe)
wr_median <- median(train_data$work_readiness_score, na.rm = TRUE)
train_data$wr_filled <- ifelse(is.na(train_data$work_readiness_score), wr_median, train_data$work_readiness_score)
test_data$wr_filled  <- ifelse(is.na(test_data$work_readiness_score),  wr_median, test_data$work_readiness_score)

# Align education_level factor levels between train and test
train_data$education_level <- factor(train_data$education_level)
test_data$education_level  <- factor(test_data$education_level, levels = levels(train_data$education_level))

upgraded_model <- glm(employed_status ~ gender + education_level + wr_filled,
                      data = train_data, family = binomial)

up_prob <- predict(upgraded_model, newdata = test_data, type = 'response')
up_prob[is.na(up_prob)] <- mean(train_data$employed_status, na.rm = TRUE)

submission2 <- data.frame(anonymised_id = test_data$anonymised_id, employed_status = up_prob)
write.csv(submission2, 'submission_upgraded.csv', row.names = FALSE, quote = FALSE)
cat('Saved submission_upgraded.csv\n')
head(submission2)


Saved submission_upgraded.csv


,anonymised_id,employed_status
,<chr>,<dbl>
1,ID_10,0.2354129
2,ID_100,0.2354129
3,ID_1000,0.2354129
4,ID_1001,0.2354129
5,ID_1002,0.2354129
6,ID_1003,0.2354129


## 🚀 Over to you — ideas to climb the leaderboard
You just saw that swapping weak features for better ones lifted the score. There is a lot more room to improve:

- **Use participant history:** `employed_lag`, `status_lag`, `status_broad_lag`, `tenure_lag` — a person's *previous* status is a strong clue, when it is present.
- **Handle missingness thoughtfully:** treat 'missing' as its own category; train and test differ in how complete some columns are.
- **More demographics & geography:** `age`, `race`, `province`, `district`, `municipality`.
- **Education & skills:** `school_quintile`, the `matric_*` subjects.
- **Try stronger models:** `ranger` (random forest) or `xgboost`.
- **Always submit probabilities** and validate with AUC, not accuracy.

Good luck!

# ============================================================
# EXPERIMENT 1 — HISTORICAL EMPLOYMENT MODEL
# ============================================================
#
# Goal:
# Predict Round 9 employment probability using:
#
#   - employed_lag
#   - status_lag
#   - status_broad_lag
#   - work_readiness_score
#   - age
#   - education_nqf
#   - total_historical_rounds
#   - tenure_lag
#   - days_since_last_obs
#
# Output:
#   submission_exp1_historical.csv
#
# ============================================================


# ------------------------------------------------------------
# 1. Load packages
# ------------------------------------------------------------

if (!requireNamespace("pROC", quietly = TRUE)) {
  install.packages("pROC")
}

library(pROC)


# ------------------------------------------------------------
# 2. Check that the datasets exist
# ------------------------------------------------------------

cat("Training rows:", nrow(train_data), "\n")
cat("Testing rows :", nrow(test_data), "\n")


# ------------------------------------------------------------
# 3. Remove rows where the target is missing
# ------------------------------------------------------------

train_model <- train_data[
  !is.na(train_data$employed_status),
]

cat(
  "Training rows after removing missing target:",
  nrow(train_model),
  "\n\n"
)


# ------------------------------------------------------------
# 4. Create education NQF variable
# ------------------------------------------------------------
#
# Extract the NQF level from education_level.
#
# Examples:
# "Grade 12/NQF Level 4" -> 4
# "Bachelor Degree NQF Level 7" -> 7
#
# If no NQF level can be extracted, use the median
# training NQF level.
# ------------------------------------------------------------

extract_nqf <- function(x) {

  x <- as.character(x)

  result <- rep(NA_real_, length(x))

  matches <- regmatches(
    x,
    regexpr(
      "NQF Level[[:space:]]*[0-9]+",
      x,
      ignore.case = TRUE
    )
  )

  valid <- matches != "" & !is.na(matches)

  if (any(valid)) {

    result[valid] <- suppressWarnings(
      as.numeric(
        sub(
          ".*NQF Level[[:space:]]*",
          "",
          matches[valid],
          ignore.case = TRUE
        )
      )
    )
  }

  result
}


train_model$education_nqf <- extract_nqf(
  train_model$education_level
)

test_data$education_nqf <- extract_nqf(
  test_data$education_level
)


# Fill missing education NQF using training median

nqf_median <- median(
  train_model$education_nqf,
  na.rm = TRUE
)

if (!is.finite(nqf_median)) {
  nqf_median <- 4
}

train_model$education_nqf[
  is.na(train_model$education_nqf)
] <- nqf_median

test_data$education_nqf[
  is.na(test_data$education_nqf)
] <- nqf_median


# ------------------------------------------------------------
# 5. Fill numerical missing values
# ------------------------------------------------------------
#
# Important:
# Missing previous-round information is meaningful.
#
# We therefore create explicit "has previous observation"
# information before filling the numerical variables.
# ------------------------------------------------------------

train_model$has_previous_observation <- ifelse(
  is.na(train_model$employed_lag),
  0,
  1
)

test_data$has_previous_observation <- ifelse(
  is.na(test_data$employed_lag),
  0,
  1
)


# Helper function for median imputation

fill_with_train_median <- function(
  train_x,
  test_x
) {

  med <- median(
    train_x,
    na.rm = TRUE
  )

  if (!is.finite(med)) {
    med <- 0
  }

  train_x[is.na(train_x)] <- med
  test_x[is.na(test_x)] <- med

  list(
    train = train_x,
    test = test_x
  )
}


# employed_lag

filled <- fill_with_train_median(
  train_model$employed_lag,
  test_data$employed_lag
)

train_model$employed_lag_filled <- filled$train
test_data$employed_lag_filled <- filled$test


# tenure_lag

filled <- fill_with_train_median(
  train_model$tenure_lag,
  test_data$tenure_lag
)

train_model$tenure_lag_filled <- filled$train
test_data$tenure_lag_filled <- filled$test


# days_since_last_obs

filled <- fill_with_train_median(
  train_model$days_since_last_obs,
  test_data$days_since_last_obs
)

train_model$days_since_last_obs_filled <- filled$train
test_data$days_since_last_obs_filled <- filled$test


# work readiness

filled <- fill_with_train_median(
  train_model$work_readiness_score,
  test_data$work_readiness_score
)

train_model$wr_filled <- filled$train
test_data$wr_filled <- filled$test


# age

filled <- fill_with_train_median(
  train_model$age,
  test_data$age
)

train_model$age_filled <- filled$train
test_data$age_filled <- filled$test


# ------------------------------------------------------------
# 6. Clean categorical variables
# ------------------------------------------------------------
#
# We combine train and test BEFORE converting to factors.
#
# This prevents:
#
# "factor X has new levels"
#
# during prediction.
# ------------------------------------------------------------

categorical_variables <- c(
  "status_lag",
  "status_broad_lag",
  "education_level"
)


for (variable in categorical_variables) {

  combined <- c(
    as.character(train_model[[variable]]),
    as.character(test_data[[variable]])
  )

  combined[is.na(combined)] <- "MISSING"

  combined_factor <- factor(combined)

  train_indices <- seq_len(nrow(train_model))

  test_indices <- (
    nrow(train_model) + 1
  ):(nrow(train_model) + nrow(test_data))

  train_model[[variable]] <- combined_factor[
    train_indices
  ]

  test_data[[variable]] <- combined_factor[
    test_indices
  ]
}


# ------------------------------------------------------------
# 7. Build the model formula
# ------------------------------------------------------------

model_formula <- employed_status ~
  employed_lag_filled +
  status_lag +
  status_broad_lag +
  wr_filled +
  age_filled +
  education_nqf +
  total_historical_rounds +
  tenure_lag_filled +
  days_since_last_obs_filled +
  has_previous_observation


# ------------------------------------------------------------
# 8. Train the model
# ------------------------------------------------------------

cat("\n")
cat("============================================\n")
cat("TRAINING EXPERIMENT 1\n")
cat("============================================\n")

historical_model <- glm(
  formula = model_formula,
  data = train_model,
  family = binomial(link = "logit")
)

cat("\nModel trained successfully.\n\n")

print(summary(historical_model))


# ------------------------------------------------------------
# 9. Generate Round 9 predictions
# ------------------------------------------------------------

cat("\n")
cat("============================================\n")
cat("GENERATING ROUND 9 PREDICTIONS\n")
cat("============================================\n")


test_probability <- predict(
  historical_model,
  newdata = test_data,
  type = "response"
)


# ------------------------------------------------------------
# 10. Safety checks
# ------------------------------------------------------------

# Replace impossible predictions

test_probability[
  !is.finite(test_probability)
] <- mean(
  train_model$employed_status,
  na.rm = TRUE
)


# Keep probabilities between 0 and 1

test_probability <- pmin(
  pmax(test_probability, 0.000001),
  0.999999
)


cat(
  "\nPrediction summary:\n"
)

print(
  summary(test_probability)
)


# ------------------------------------------------------------
# 11. Create submission
# ------------------------------------------------------------

submission_exp1 <- data.frame(

  anonymised_id = test_data$anonymised_id,

  employed_status = test_probability

)


# ------------------------------------------------------------
# 12. Check submission
# ------------------------------------------------------------

cat("\n")
cat("Submission rows:", nrow(submission_exp1), "\n")

cat(
  "Expected test rows:",
  nrow(test_data),
  "\n"
)

cat(
  "Missing predictions:",
  sum(is.na(submission_exp1$employed_status)),
  "\n"
)

cat("\nFirst predictions:\n")

print(
  head(submission_exp1, 10)
)


# ------------------------------------------------------------
# 13. Save submission
# ------------------------------------------------------------

output_file <- "submission_exp1_historical.csv"

write.csv(
  submission_exp1,
  output_file,
  row.names = FALSE,
  quote = FALSE
)


cat("\n")
cat("============================================\n")
cat("SUBMISSION CREATED\n")
cat("============================================\n")

cat(
  "Saved:",
  output_file,
  "\n"
)


# ------------------------------------------------------------
# 14. Final validation of file
# ------------------------------------------------------------

if (
  nrow(submission_exp1) == nrow(test_data) &&
  all(!is.na(submission_exp1$employed_status)) &&
  all(
    submission_exp1$employed_status >= 0 &
    submission_exp1$employed_status <= 1
  )
) {

  cat("\n")
  cat("✓ Submission passed all checks.\n")
  cat("✓ Correct number of rows.\n")
  cat("✓ No missing predictions.\n")
  cat("✓ All predictions are probabilities between 0 and 1.\n")

} else {

  cat("\n")
  cat("⚠ Submission failed one or more checks.\n")

}

In [22]:
# ============================================================
# EXPERIMENT 1 — HISTORICAL EMPLOYMENT MODEL
# ============================================================
#
# Goal:
# Predict Round 9 employment probability using:
#
#   - employed_lag
#   - status_lag
#   - status_broad_lag
#   - work_readiness_score
#   - age
#   - education_nqf
#   - total_historical_rounds
#   - tenure_lag
#   - days_since_last_obs
#
# Output:
#   submission_exp1_historical.csv
#
# ============================================================


# ------------------------------------------------------------
# 1. Load packages
# ------------------------------------------------------------

if (!requireNamespace("pROC", quietly = TRUE)) {
  install.packages("pROC")
}

library(pROC)


# ------------------------------------------------------------
# 2. Check that the datasets exist
# ------------------------------------------------------------

cat("Training rows:", nrow(train_data), "\n")
cat("Testing rows :", nrow(test_data), "\n")


# ------------------------------------------------------------
# 3. Remove rows where the target is missing
# ------------------------------------------------------------

train_model <- train_data[
  !is.na(train_data$employed_status),
]

cat(
  "Training rows after removing missing target:",
  nrow(train_model),
  "\n\n"
)


# ------------------------------------------------------------
# 4. Create education NQF variable
# ------------------------------------------------------------
#
# Extract the NQF level from education_level.
#
# Examples:
# "Grade 12/NQF Level 4" -> 4
# "Bachelor Degree NQF Level 7" -> 7
#
# If no NQF level can be extracted, use the median
# training NQF level.
# ------------------------------------------------------------

extract_nqf <- function(x) {

  x <- as.character(x)

  result <- rep(NA_real_, length(x))

  matches <- regmatches(
    x,
    regexpr(
      "NQF Level[[:space:]]*[0-9]+",
      x,
      ignore.case = TRUE
    )
  )

  valid <- matches != "" & !is.na(matches)

  if (any(valid)) {

    result[valid] <- suppressWarnings(
      as.numeric(
        sub(
          ".*NQF Level[[:space:]]*",
          "",
          matches[valid],
          ignore.case = TRUE
        )
      )
    )
  }

  result
}


train_model$education_nqf <- extract_nqf(
  train_model$education_level
)

test_data$education_nqf <- extract_nqf(
  test_data$education_level
)


# Fill missing education NQF using training median

nqf_median <- median(
  train_model$education_nqf,
  na.rm = TRUE
)

if (!is.finite(nqf_median)) {
  nqf_median <- 4
}

train_model$education_nqf[
  is.na(train_model$education_nqf)
] <- nqf_median

test_data$education_nqf[
  is.na(test_data$education_nqf)
] <- nqf_median


# ------------------------------------------------------------
# 5. Fill numerical missing values
# ------------------------------------------------------------
#
# Important:
# Missing previous-round information is meaningful.
#
# We therefore create explicit "has previous observation"
# information before filling the numerical variables.
# ------------------------------------------------------------

train_model$has_previous_observation <- ifelse(
  is.na(train_model$employed_lag),
  0,
  1
)

test_data$has_previous_observation <- ifelse(
  is.na(test_data$employed_lag),
  0,
  1
)


# Helper function for median imputation

fill_with_train_median <- function(
  train_x,
  test_x
) {

  med <- median(
    train_x,
    na.rm = TRUE
  )

  if (!is.finite(med)) {
    med <- 0
  }

  train_x[is.na(train_x)] <- med
  test_x[is.na(test_x)] <- med

  list(
    train = train_x,
    test = test_x
  )
}


# employed_lag

filled <- fill_with_train_median(
  train_model$employed_lag,
  test_data$employed_lag
)

train_model$employed_lag_filled <- filled$train
test_data$employed_lag_filled <- filled$test


# tenure_lag

filled <- fill_with_train_median(
  train_model$tenure_lag,
  test_data$tenure_lag
)

train_model$tenure_lag_filled <- filled$train
test_data$tenure_lag_filled <- filled$test


# days_since_last_obs

filled <- fill_with_train_median(
  train_model$days_since_last_obs,
  test_data$days_since_last_obs
)

train_model$days_since_last_obs_filled <- filled$train
test_data$days_since_last_obs_filled <- filled$test


# work readiness

filled <- fill_with_train_median(
  train_model$work_readiness_score,
  test_data$work_readiness_score
)

train_model$wr_filled <- filled$train
test_data$wr_filled <- filled$test


# age

filled <- fill_with_train_median(
  train_model$age,
  test_data$age
)

train_model$age_filled <- filled$train
test_data$age_filled <- filled$test


# ------------------------------------------------------------
# 6. Clean categorical variables
# ------------------------------------------------------------
#
# We combine train and test BEFORE converting to factors.
#
# This prevents:
#
# "factor X has new levels"
#
# during prediction.
# ------------------------------------------------------------

categorical_variables <- c(
  "status_lag",
  "status_broad_lag",
  "education_level"
)


for (variable in categorical_variables) {

  combined <- c(
    as.character(train_model[[variable]]),
    as.character(test_data[[variable]])
  )

  combined[is.na(combined)] <- "MISSING"

  combined_factor <- factor(combined)

  train_indices <- seq_len(nrow(train_model))

  test_indices <- (
    nrow(train_model) + 1
  ):(nrow(train_model) + nrow(test_data))

  train_model[[variable]] <- combined_factor[
    train_indices
  ]

  test_data[[variable]] <- combined_factor[
    test_indices
  ]
}


# ------------------------------------------------------------
# 7. Build the model formula
# ------------------------------------------------------------

model_formula <- employed_status ~
  employed_lag_filled +
  status_lag +
  status_broad_lag +
  wr_filled +
  age_filled +
  education_nqf +
  total_historical_rounds +
  tenure_lag_filled +
  days_since_last_obs_filled +
  has_previous_observation


# ------------------------------------------------------------
# 8. Train the model
# ------------------------------------------------------------

cat("\n")
cat("============================================\n")
cat("TRAINING EXPERIMENT 1\n")
cat("============================================\n")

historical_model <- glm(
  formula = model_formula,
  data = train_model,
  family = binomial(link = "logit")
)

cat("\nModel trained successfully.\n\n")

print(summary(historical_model))


# ------------------------------------------------------------
# 9. Generate Round 9 predictions
# ------------------------------------------------------------

cat("\n")
cat("============================================\n")
cat("GENERATING ROUND 9 PREDICTIONS\n")
cat("============================================\n")


test_probability <- predict(
  historical_model,
  newdata = test_data,
  type = "response"
)


# ------------------------------------------------------------
# 10. Safety checks
# ------------------------------------------------------------

# Replace impossible predictions

test_probability[
  !is.finite(test_probability)
] <- mean(
  train_model$employed_status,
  na.rm = TRUE
)


# Keep probabilities between 0 and 1

test_probability <- pmin(
  pmax(test_probability, 0.000001),
  0.999999
)


cat(
  "\nPrediction summary:\n"
)

print(
  summary(test_probability)
)


# ------------------------------------------------------------
# 11. Create submission
# ------------------------------------------------------------

submission_exp1 <- data.frame(

  anonymised_id = test_data$anonymised_id,

  employed_status = test_probability

)


# ------------------------------------------------------------
# 12. Check submission
# ------------------------------------------------------------

cat("\n")
cat("Submission rows:", nrow(submission_exp1), "\n")

cat(
  "Expected test rows:",
  nrow(test_data),
  "\n"
)

cat(
  "Missing predictions:",
  sum(is.na(submission_exp1$employed_status)),
  "\n"
)

cat("\nFirst predictions:\n")

print(
  head(submission_exp1, 10)
)


# ------------------------------------------------------------
# 13. Save submission
# ------------------------------------------------------------

output_file <- "submission_exp1_historical.csv"

write.csv(
  submission_exp1,
  output_file,
  row.names = FALSE,
  quote = FALSE
)


cat("\n")
cat("============================================\n")
cat("SUBMISSION CREATED\n")
cat("============================================\n")

cat(
  "Saved:",
  output_file,
  "\n"
)


# ------------------------------------------------------------
# 14. Final validation of file
# ------------------------------------------------------------

if (
  nrow(submission_exp1) == nrow(test_data) &&
  all(!is.na(submission_exp1$employed_status)) &&
  all(
    submission_exp1$employed_status >= 0 &
    submission_exp1$employed_status <= 1
  )
) {

  cat("\n")
  cat("✓ Submission passed all checks.\n")
  cat("✓ Correct number of rows.\n")
  cat("✓ No missing predictions.\n")
  cat("✓ All predictions are probabilities between 0 and 1.\n")

} else {

  cat("\n")
  cat("⚠ Submission failed one or more checks.\n")

}

Warning message:
"package 'pROC' was built under R version 4.4.3"
Type 'citation("pROC")' for a citation.


Attaching package: 'pROC'


The following objects are masked from 'package:stats':

    cov, smooth, var




Training rows: 16719 
Testing rows : 3230 
Training rows after removing missing target: 16653 



Warning message in result[valid] <- suppressWarnings(as.numeric(sub(".*NQF Level[[:space:]]*", :
"number of items to replace is not a multiple of replacement length"
Warning message in result[valid] <- suppressWarnings(as.numeric(sub(".*NQF Level[[:space:]]*", :
"number of items to replace is not a multiple of replacement length"



TRAINING EXPERIMENT 1

Model trained successfully.


Call:
glm(formula = model_formula, family = binomial(link = "logit"), 
    data = train_model)

Coefficients: (4 not defined because of singularities)
                                         Estimate Std. Error z value Pr(>|z|)
(Intercept)                            -1.785e+00  1.864e-01  -9.574  < 2e-16
employed_lag_filled                     1.122e+00  1.128e-01   9.949  < 2e-16
status_lagself employed                 2.399e+00  1.094e+00   2.192  0.02836
status_lagunclear                       1.304e+00  4.076e-01   3.200  0.00137
status_lagunemployed                    2.009e-01  2.045e-01   0.982  0.32601
status_lagwage and self employed        2.362e-01  2.887e-01   0.818  0.41331
status_lagwage employed                        NA         NA      NA       NA
status_broad_lagself employed          -2.622e+00  1.096e+00  -2.392  0.01674
status_broad_lagstudying               -4.869e-01  2.008e-01  -2.425  0.01529
status_broad_la

Warning message in predict.lm(object, newdata, se.fit, scale = 1, type = if (type == :
"prediction from rank-deficient fit; attr(*, "non-estim") has doubtful cases"



Prediction summary:
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
 0.1811  0.2854  0.3051  0.3310  0.3275  0.9146 

Submission rows: 3230 
Expected test rows: 3230 
Missing predictions: 0 

First predictions:
   anonymised_id employed_status
1          ID_10       0.2977006
2         ID_100       0.2935673
3        ID_1000       0.3018674
4        ID_1001       0.3004273
5        ID_1002       0.3018674
6        ID_1003       0.3018674
7        ID_1004       0.3047591
8        ID_1005       0.3018674
9        ID_1006       0.2989913
10       ID_1007       0.3018674

SUBMISSION CREATED
Saved: submission_exp1_historical.csv 

✓ Submission passed all checks.
✓ Correct number of rows.
✓ No missing predictions.
✓ All predictions are probabilities between 0 and 1.


In [24]:
# ============================================================
# EXPERIMENT 2
# Historical Employment + Labour Market Status
# ============================================================

cat("============================================\n")
cat("EXPERIMENT 2: HISTORICAL STATUS MODEL\n")
cat("============================================\n\n")

# ------------------------------------------------------------
# 1. LOAD DATA
# ------------------------------------------------------------

data_dir <- "../../assets/dataset"

train_data <- read.csv(
  file.path(data_dir, "train.csv"),
  stringsAsFactors = FALSE
)

test_data <- read.csv(
  file.path(data_dir, "test.csv"),
  stringsAsFactors = FALSE
)

cat("Training rows:", nrow(train_data), "\n")
cat("Testing rows :", nrow(test_data), "\n\n")


# ------------------------------------------------------------
# 2. REMOVE MISSING TARGET
# ------------------------------------------------------------

train_data <- train_data[
  !is.na(train_data$employed_status),
]

train_data$employed_status <- as.numeric(
  train_data$employed_status
)

cat("Training rows after target cleaning:",
    nrow(train_data), "\n\n")


# ------------------------------------------------------------
# 3. HELPER FUNCTION
# Convert categorical variables into safe factors
# ------------------------------------------------------------

prepare_factor <- function(train, test, column) {

  # Convert to character first
  train[[column]] <- as.character(train[[column]])
  test[[column]]  <- as.character(test[[column]])

  # Explicit missing category
  train[[column]][
    is.na(train[[column]]) |
    train[[column]] == ""
  ] <- "UNKNOWN"

  test[[column]][
    is.na(test[[column]]) |
    test[[column]] == ""
  ] <- "UNKNOWN"

  # Find categories shared between train and test
  train_levels <- unique(train[[column]])

  # Anything in test that training never saw
  # becomes UNKNOWN
  test[[column]][
    !test[[column]] %in% train_levels
  ] <- "UNKNOWN"

  # Make factors using training levels
  train[[column]] <- factor(
    train[[column]],
    levels = train_levels
  )

  test[[column]] <- factor(
    test[[column]],
    levels = train_levels
  )

  return(list(train = train, test = test))
}


# ------------------------------------------------------------
# 4. MISSINGNESS FEATURES
# ------------------------------------------------------------

# Previous employment
train_data$has_employed_lag <- !is.na(
  train_data$employed_lag
)

test_data$has_employed_lag <- !is.na(
  test_data$employed_lag
)

# Previous detailed status
train_data$has_status_lag <- !is.na(
  train_data$status_lag
)

test_data$has_status_lag <- !is.na(
  test_data$status_lag
)

# Previous broad status
train_data$has_status_broad_lag <- !is.na(
  train_data$status_broad_lag
)

test_data$has_status_broad_lag <- !is.na(
  test_data$status_broad_lag
)


# ------------------------------------------------------------
# 5. FILL NUMERIC VARIABLES
# ------------------------------------------------------------

numeric_variables <- c(
  "employed_lag",
  "age",
  "work_readiness_score",
  "tenure_lag",
  "days_since_last_obs",
  "total_historical_rounds",
  "lag_round"
)

for (col in numeric_variables) {

  if (col %in% names(train_data)) {

    median_value <- median(
      train_data[[col]],
      na.rm = TRUE
    )

    if (!is.finite(median_value)) {
      median_value <- 0
    }

    train_data[[paste0(col, "_filled")]] <-
      ifelse(
        is.na(train_data[[col]]),
        median_value,
        train_data[[col]]
      )

    test_data[[paste0(col, "_filled")]] <-
      ifelse(
        is.na(test_data[[col]]),
        median_value,
        test_data[[col]]
      )
  }
}


# ------------------------------------------------------------
# 6. PREPARE CATEGORICAL VARIABLES
# ------------------------------------------------------------

categorical_variables <- c(
  "gender",
  "race",
  "province",
  "district",
  "status_lag",
  "status_broad_lag"
)

for (col in categorical_variables) {

  if (col %in% names(train_data)) {

    result <- prepare_factor(
      train_data,
      test_data,
      col
    )

    train_data <- result$train
    test_data  <- result$test
  }
}


# ------------------------------------------------------------
# 7. MODEL FORMULA
# ------------------------------------------------------------

model_formula <- employed_status ~
  employed_lag_filled +
  status_lag +
  status_broad_lag +
  has_employed_lag +
  has_status_lag +
  has_status_broad_lag +
  work_readiness_score_filled +
  age_filled +
  tenure_lag_filled +
  days_since_last_obs_filled +
  total_historical_rounds_filled +
  lag_round_filled +
  gender +
  race +
  province +
  district


# ------------------------------------------------------------
# 8. TRAIN MODEL
# ------------------------------------------------------------

cat("Training model...\n\n")

model <- glm(
  formula = model_formula,
  data = train_data,
  family = binomial(link = "logit")
)

cat("Model trained successfully.\n\n")

print(summary(model))


# ------------------------------------------------------------
# 9. GENERATE PREDICTIONS
# ------------------------------------------------------------

cat("\n============================================\n")
cat("GENERATING ROUND 9 PREDICTIONS\n")
cat("============================================\n\n")

test_prob <- predict(
  model,
  newdata = test_data,
  type = "response"
)


# ------------------------------------------------------------
# 10. SAFETY CHECK
# ------------------------------------------------------------

# Replace invalid predictions
fallback_probability <- mean(
  train_data$employed_status,
  na.rm = TRUE
)

test_prob[
  is.na(test_prob) |
  !is.finite(test_prob)
] <- fallback_probability

# Keep probabilities strictly inside [0,1]
test_prob <- pmin(
  pmax(test_prob, 0.000001),
  0.999999
)


# ------------------------------------------------------------
# 11. CREATE SUBMISSION
# ------------------------------------------------------------

submission_exp2 <- data.frame(
  anonymised_id = test_data$anonymised_id,
  employed_status = test_prob
)


# ------------------------------------------------------------
# 12. VALIDATE SUBMISSION
# ------------------------------------------------------------

cat("\nPrediction summary:\n")
print(summary(submission_exp2$employed_status))

cat("\nNumber of NA predictions:",
    sum(is.na(submission_exp2$employed_status)),
    "\n")

cat("Number of invalid predictions:",
    sum(
      !is.finite(
        submission_exp2$employed_status
      )
    ),
    "\n")


# ------------------------------------------------------------
# 13. SAVE
# ------------------------------------------------------------

output_file <- "submission_exp2_historical_status_clean.csv"

write.csv(
  submission_exp2,
  output_file,
  row.names = FALSE,
  quote = FALSE
)

cat("\nSaved:", output_file, "\n")
cat("Rows:", nrow(submission_exp2), "\n\n")


# ------------------------------------------------------------
# 14. PREVIEW
# ------------------------------------------------------------

cat("First 10 predictions:\n\n")

print(
  head(
    submission_exp2,
    10
  )
)

cat("\n============================================\n")
cat("EXPERIMENT COMPLETE\n")
cat("============================================\n")

EXPERIMENT 2: HISTORICAL STATUS MODEL

Training rows: 16719 
Testing rows : 3230 

Training rows after target cleaning: 16653 

Training model...

Model trained successfully.


Call:
glm(formula = model_formula, family = binomial(link = "logit"), 
    data = train_data)

Coefficients: (6 not defined because of singularities)
                                         Estimate Std. Error z value Pr(>|z|)
(Intercept)                            -2.379e+00  4.485e-01  -5.304 1.13e-07
employed_lag_filled                     1.343e+00  3.011e-01   4.460 8.18e-06
status_lagunemployed                    6.683e-02  2.109e-01   0.317   0.7513
status_lagwage employed                -3.257e-01  2.969e-01  -1.097   0.2727
status_lagself employed                 1.854e+00  1.143e+00   1.622   0.1048
status_lagunclear                       9.861e-01  4.151e-01   2.375   0.0175
status_lagwage and self employed               NA         NA      NA       NA
status_broad_lagunemployed             -1.547e-01

Warning message in predict.lm(object, newdata, se.fit, scale = 1, type = if (type == :
"prediction from rank-deficient fit; attr(*, "non-estim") has doubtful cases"



Prediction summary:
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
0.07003 0.25304 0.28861 0.33713 0.39388 0.94196 

Number of NA predictions: 0 
Number of invalid predictions: 0 

Saved: submission_exp2_historical_status_clean.csv 
Rows: 3230 

First 10 predictions:

   anonymised_id employed_status
1          ID_10       0.2530412
2         ID_100       0.2482038
3        ID_1000       0.2579405
4        ID_1001       0.2579405
5        ID_1002       0.2579405
6        ID_1003       0.2579405
7        ID_1004       0.2579405
8        ID_1005       0.2579405
9        ID_1006       0.2579405
10       ID_1007       0.2579405

EXPERIMENT COMPLETE


In [25]:
# ============================================================
# EXPERIMENT 3
# Baseline + Clean Historical Features
# ============================================================

cat("========================================\n")
cat("EXPERIMENT 3: BASELINE + HISTORY\n")
cat("========================================\n\n")

# ------------------------------------------------------------
# 1. LOAD DATA
# ------------------------------------------------------------

data_dir <- "../../assets/dataset"

train_data <- read.csv(
  file.path(data_dir, "train.csv"),
  stringsAsFactors = FALSE
)

test_data <- read.csv(
  file.path(data_dir, "test.csv"),
  stringsAsFactors = FALSE
)

cat("Training rows:", nrow(train_data), "\n")
cat("Testing rows :", nrow(test_data), "\n\n")


# ------------------------------------------------------------
# 2. CLEAN TARGET
# ------------------------------------------------------------

train_data <- train_data[
  !is.na(train_data$employed_status),
]

train_data$employed_status <- as.numeric(
  train_data$employed_status
)

cat("Training rows after removing missing target:",
    nrow(train_data), "\n\n")

cat("Target distribution:\n")
print(table(train_data$employed_status))
cat("\n")


# ------------------------------------------------------------
# 3. HELPER FUNCTION FOR NUMERIC VARIABLES
# ------------------------------------------------------------

# Convert safely to numeric
to_numeric <- function(x) {
  suppressWarnings(as.numeric(as.character(x)))
}


# ------------------------------------------------------------
# 4. WORK READINESS
# ------------------------------------------------------------

train_data$work_readiness_score <- to_numeric(
  train_data$work_readiness_score
)

test_data$work_readiness_score <- to_numeric(
  test_data$work_readiness_score
)

# Use TRAINING median only
wr_median <- median(
  train_data$work_readiness_score,
  na.rm = TRUE
)

train_data$wr_filled <- ifelse(
  is.na(train_data$work_readiness_score),
  wr_median,
  train_data$work_readiness_score
)

test_data$wr_filled <- ifelse(
  is.na(test_data$work_readiness_score),
  wr_median,
  test_data$work_readiness_score
)

# Indicator showing whether the original value existed
train_data$has_work_readiness <- ifelse(
  is.na(train_data$work_readiness_score),
  0,
  1
)

test_data$has_work_readiness <- ifelse(
  is.na(test_data$work_readiness_score),
  0,
  1
)


# ------------------------------------------------------------
# 5. HISTORICAL NUMERIC FEATURES
# ------------------------------------------------------------

numeric_history <- c(
  "employed_lag",
  "tenure_lag",
  "days_since_last_obs",
  "lag_round",
  "total_historical_rounds",
  "age"
)

for (v in numeric_history) {

  train_data[[v]] <- to_numeric(train_data[[v]])
  test_data[[v]] <- to_numeric(test_data[[v]])

}


# ------------------------------------------------------------
# 6. MISSING-VALUE INDICATORS
# ------------------------------------------------------------

# Missing historical information is potentially meaningful.
# A person with no previous observation is different from
# someone whose previous observation exists but has a value of 0.

train_data$has_previous_observation <- ifelse(
  is.na(train_data$employed_lag),
  0,
  1
)

test_data$has_previous_observation <- ifelse(
  is.na(test_data$employed_lag),
  0,
  1
)

train_data$has_tenure <- ifelse(
  is.na(train_data$tenure_lag),
  0,
  1
)

test_data$has_tenure <- ifelse(
  is.na(test_data$tenure_lag),
  0,
  1
)


# ------------------------------------------------------------
# 7. IMPUTE HISTORICAL NUMERIC VARIABLES
# ------------------------------------------------------------

# IMPORTANT:
# Medians are calculated from TRAINING data only.

history_medians <- list()

for (v in numeric_history) {

  med <- median(
    train_data[[v]],
    na.rm = TRUE
  )

  history_medians[[v]] <- med

  train_data[[paste0(v, "_filled")]] <- ifelse(
    is.na(train_data[[v]]),
    med,
    train_data[[v]]
  )

  test_data[[paste0(v, "_filled")]] <- ifelse(
    is.na(test_data[[v]]),
    med,
    test_data[[v]]
  )
}


# ------------------------------------------------------------
# 8. CREATE A SIMPLE EDUCATION NQF FEATURE
# ------------------------------------------------------------

extract_nqf <- function(x) {

  x <- as.character(x)

  result <- rep(NA_real_, length(x))

  # Find "NQF Level X"
  matches <- regmatches(
    x,
    regexpr(
      "NQF Level [0-9]+",
      x
    )
  )

  valid <- matches != ""

  if (any(valid)) {

    result[valid] <- suppressWarnings(
      as.numeric(
        sub(
          "NQF Level ",
          "",
          matches[valid],
          fixed = TRUE
        )
      )
    )
  }

  # Handle common Grade X values
  grade_matches <- regmatches(
    x,
    regexpr(
      "Grade [0-9]+",
      x
    )
  )

  grade_valid <- grade_matches != ""

  if (any(grade_valid)) {

    grades <- suppressWarnings(
      as.numeric(
        sub(
          "Grade ",
          "",
          grade_matches[grade_valid],
          fixed = TRUE
        )
      )
    )

    # Approximate mapping:
    # Grade 9  -> NQF 1
    # Grade 10 -> NQF 2
    # Grade 11 -> NQF 3
    # Grade 12 -> NQF 4

    result[grade_valid] <- pmax(
      grades - 8,
      1
    )
  }

  result
}


train_data$education_nqf <- extract_nqf(
  train_data$education_level
)

test_data$education_nqf <- extract_nqf(
  test_data$education_level
)


# ------------------------------------------------------------
# 9. IMPUTE EDUCATION NQF
# ------------------------------------------------------------

education_median <- median(
  train_data$education_nqf,
  na.rm = TRUE
)

train_data$education_nqf <- ifelse(
  is.na(train_data$education_nqf),
  education_median,
  train_data$education_nqf
)

test_data$education_nqf <- ifelse(
  is.na(test_data$education_nqf),
  education_median,
  test_data$education_nqf
)


# ------------------------------------------------------------
# 10. BUILD MODEL DATA
# ------------------------------------------------------------

model_train <- data.frame(
  employed_status = train_data$employed_status,

  wr_filled = train_data$wr_filled,
  has_work_readiness = train_data$has_work_readiness,

  employed_lag_filled =
    train_data$employed_lag_filled,

  tenure_lag_filled =
    train_data$tenure_lag_filled,

  days_since_last_obs_filled =
    train_data$days_since_last_obs_filled,

  lag_round_filled =
    train_data$lag_round_filled,

  total_historical_rounds_filled =
    train_data$total_historical_rounds_filled,

  age_filled =
    train_data$age_filled,

  education_nqf =
    train_data$education_nqf,

  has_previous_observation =
    train_data$has_previous_observation,

  has_tenure =
    train_data$has_tenure
)


model_test <- data.frame(

  wr_filled = test_data$wr_filled,
  has_work_readiness = test_data$has_work_readiness,

  employed_lag_filled =
    test_data$employed_lag_filled,

  tenure_lag_filled =
    test_data$tenure_lag_filled,

  days_since_last_obs_filled =
    test_data$days_since_last_obs_filled,

  lag_round_filled =
    test_data$lag_round_filled,

  total_historical_rounds_filled =
    test_data$total_historical_rounds_filled,

  age_filled =
    test_data$age_filled,

  education_nqf =
    test_data$education_nqf,

  has_previous_observation =
    test_data$has_previous_observation,

  has_tenure =
    test_data$has_tenure
)


# ------------------------------------------------------------
# 11. FINAL NA SAFETY CHECK
# ------------------------------------------------------------

for (v in names(model_train)) {

  if (v != "employed_status") {

    med <- median(
      model_train[[v]],
      na.rm = TRUE
    )

    model_train[[v]][
      is.na(model_train[[v]])
    ] <- med

    model_test[[v]][
      is.na(model_test[[v]])
    ] <- med
  }
}


# ------------------------------------------------------------
# 12. CHECK DATA
# ------------------------------------------------------------

cat("========================================\n")
cat("MODEL DATA CHECK\n")
cat("========================================\n\n")

cat("Training dimensions:",
    nrow(model_train),
    "x",
    ncol(model_train),
    "\n")

cat("Testing dimensions :",
    nrow(model_test),
    "x",
    ncol(model_test),
    "\n\n")

cat("Remaining training NAs:\n")
print(sum(is.na(model_train)))

cat("\nRemaining testing NAs:\n")
print(sum(is.na(model_test)))

cat("\n")


# ------------------------------------------------------------
# 13. TRAIN LOGISTIC REGRESSION
# ------------------------------------------------------------

cat("========================================\n")
cat("TRAINING MODEL\n")
cat("========================================\n\n")

model_formula <- employed_status ~
  wr_filled +
  has_work_readiness +
  employed_lag_filled +
  tenure_lag_filled +
  days_since_last_obs_filled +
  lag_round_filled +
  total_historical_rounds_filled +
  age_filled +
  education_nqf +
  has_previous_observation +
  has_tenure


exp3_model <- glm(
  formula = model_formula,
  data = model_train,
  family = binomial(link = "logit")
)

cat("Model trained successfully.\n\n")

print(summary(exp3_model))


# ------------------------------------------------------------
# 14. GENERATE PREDICTIONS
# ------------------------------------------------------------

cat("\n========================================\n")
cat("GENERATING ROUND 9 PREDICTIONS\n")
cat("========================================\n\n")

exp3_prob <- predict(
  exp3_model,
  newdata = model_test,
  type = "response"
)


# ------------------------------------------------------------
# 15. SAFETY CHECK
# ------------------------------------------------------------

exp3_prob[is.na(exp3_prob)] <- mean(
  train_data$employed_status,
  na.rm = TRUE
)

exp3_prob <- pmin(
  pmax(exp3_prob, 0.000001),
  0.999999
)


# ------------------------------------------------------------
# 16. CREATE SUBMISSION
# ------------------------------------------------------------

submission_exp3 <- data.frame(
  anonymised_id = test_data$anonymised_id,
  employed_status = exp3_prob
)


# ------------------------------------------------------------
# 17. VALIDATE SUBMISSION
# ------------------------------------------------------------

cat("Prediction summary:\n")
print(summary(submission_exp3$employed_status))

cat("\nNumber of predictions:",
    nrow(submission_exp3),
    "\n")

cat("Missing predictions:",
    sum(is.na(submission_exp3$employed_status)),
    "\n")


# ------------------------------------------------------------
# 18. SAVE
# ------------------------------------------------------------

output_file <- "submission_exp3_baseline_history.csv"

write.csv(
  submission_exp3,
  output_file,
  row.names = FALSE,
  quote = FALSE
)

cat("\n========================================\n")
cat("SUBMISSION READY\n")
cat("========================================\n")

cat("\nSaved:", output_file, "\n")

cat("\nFirst 10 predictions:\n")
print(head(submission_exp3, 10))

EXPERIMENT 3: BASELINE + HISTORY

Training rows: 16719 
Testing rows : 3230 

Training rows after removing missing target: 16653 

Target distribution:

    0     1 
11375  5278 



Warning message in result[valid] <- suppressWarnings(as.numeric(sub("NQF Level ", :
"number of items to replace is not a multiple of replacement length"
Warning message in result[grade_valid] <- pmax(grades - 8, 1):
"number of items to replace is not a multiple of replacement length"
Warning message in result[valid] <- suppressWarnings(as.numeric(sub("NQF Level ", :
"number of items to replace is not a multiple of replacement length"
Warning message in result[grade_valid] <- pmax(grades - 8, 1):
"number of items to replace is not a multiple of replacement length"


MODEL DATA CHECK

Training dimensions: 16653 x 12 
Testing dimensions : 3230 x 11 

Remaining training NAs:
[1] 0

Remaining testing NAs:
[1] 0

TRAINING MODEL

Model trained successfully.


Call:
glm(formula = model_formula, family = binomial(link = "logit"), 
    data = model_train)

Coefficients:
                                 Estimate Std. Error z value Pr(>|z|)    
(Intercept)                    -1.902e+00  2.230e-01  -8.526  < 2e-16 ***
wr_filled                       8.939e-01  1.086e-01   8.230  < 2e-16 ***
has_work_readiness             -5.417e-02  3.595e-02  -1.507  0.13182    
employed_lag_filled             1.121e+00  6.574e-02  17.058  < 2e-16 ***
tenure_lag_filled              -1.146e-04  4.991e-05  -2.296  0.02165 *  
days_since_last_obs_filled      2.617e-04  3.109e-04   0.842  0.39990    
lag_round_filled                5.287e-02  1.906e-02   2.775  0.00553 ** 
total_historical_rounds_filled  4.306e-02  6.205e-02   0.694  0.48765    
age_filled                      2

In [26]:
# ============================================================
# EXPERIMENT 4 — CONTROLLED LOGISTIC REGRESSION
# ============================================================
#
# Goal:
# Build a simpler model using the strongest variables identified
# from our previous analysis.
#
# Benchmark:
# Original baseline = 0.64035 AUC
#
# We deliberately avoid:
# - municipality
# - district
# - tertiary qualification name
# - education_field
# - sample
# - survey_date
#
# These can introduce many levels / instability between train and
# Round 9 test data.
# ============================================================


# ------------------------------------------------------------
# 1. LOAD DATA
# ------------------------------------------------------------

data_dir <- "../../assets/dataset"

train_data <- read.csv(
  file.path(data_dir, "train.csv"),
  stringsAsFactors = FALSE
)

test_data <- read.csv(
  file.path(data_dir, "test.csv"),
  stringsAsFactors = FALSE
)

cat("Training rows:", nrow(train_data), "\n")
cat("Testing rows :", nrow(test_data), "\n")


# ------------------------------------------------------------
# 2. REMOVE ROWS WITH MISSING TARGET
# ------------------------------------------------------------

train_data <- train_data[
  !is.na(train_data$employed_status),
]

train_data$employed_status <- as.numeric(
  train_data$employed_status
)

cat("\nTarget distribution:\n")
print(table(train_data$employed_status))


# ------------------------------------------------------------
# 3. WORK READINESS
# ------------------------------------------------------------

# Calculate median ONLY from training data
wr_median <- median(
  train_data$work_readiness_score,
  na.rm = TRUE
)

train_data$wr_filled <- ifelse(
  is.na(train_data$work_readiness_score),
  wr_median,
  train_data$work_readiness_score
)

test_data$wr_filled <- ifelse(
  is.na(test_data$work_readiness_score),
  wr_median,
  test_data$work_readiness_score
)

# Missingness indicator
train_data$has_work_readiness <- as.integer(
  !is.na(train_data$work_readiness_score)
)

test_data$has_work_readiness <- as.integer(
  !is.na(test_data$work_readiness_score)
)


# ------------------------------------------------------------
# 4. EMPLOYMENT LAG
# ------------------------------------------------------------

train_data$employed_lag_filled <- ifelse(
  is.na(train_data$employed_lag),
  0,
  train_data$employed_lag
)

test_data$employed_lag_filled <- ifelse(
  is.na(test_data$employed_lag),
  0,
  test_data$employed_lag
)

# Important: whether a previous observation actually exists
train_data$has_previous_observation <- as.integer(
  !is.na(train_data$employed_lag)
)

test_data$has_previous_observation <- as.integer(
  !is.na(test_data$employed_lag)
)


# ------------------------------------------------------------
# 5. AGE
# ------------------------------------------------------------

age_median <- median(
  train_data$age,
  na.rm = TRUE
)

train_data$age_filled <- ifelse(
  is.na(train_data$age),
  age_median,
  train_data$age
)

test_data$age_filled <- ifelse(
  is.na(test_data$age),
  age_median,
  test_data$age
)


# ------------------------------------------------------------
# 6. SCHOOL QUINTILE
# ------------------------------------------------------------

# 0 means unspecified in this dataset.
# Treat it as its own meaningful value rather than NA.

train_data$school_quintile_filled <- ifelse(
  is.na(train_data$school_quintile),
  0,
  train_data$school_quintile
)

test_data$school_quintile_filled <- ifelse(
  is.na(test_data$school_quintile),
  0,
  test_data$school_quintile
)


# ------------------------------------------------------------
# 7. TOTAL HISTORICAL ROUNDS
# ------------------------------------------------------------

train_data$total_historical_rounds_filled <- ifelse(
  is.na(train_data$total_historical_rounds),
  0,
  train_data$total_historical_rounds
)

test_data$total_historical_rounds_filled <- ifelse(
  is.na(test_data$total_historical_rounds),
  0,
  test_data$total_historical_rounds
)


# ------------------------------------------------------------
# 8. NQF EDUCATION LEVEL
# ------------------------------------------------------------
#
# Extract the NQF level safely from education_level.
#
# If no NQF level is found, use 0.
# ------------------------------------------------------------

extract_nqf <- function(x) {

  x <- as.character(x)

  result <- rep(NA_real_, length(x))

  # Look for "NQF Level X"
  matches <- regmatches(
    x,
    regexpr(
      "NQF Level[[:space:]]*[0-9]+",
      x,
      ignore.case = TRUE
    )
  )

  valid <- matches != ""

  result[valid] <- suppressWarnings(
    as.numeric(
      sub(
        ".*NQF Level[[:space:]]*",
        "",
        matches[valid],
        ignore.case = TRUE
      )
    )
  )

  # Handle Grade X entries
  grade_matches <- regmatches(
    x,
    regexpr(
      "Grade[[:space:]]*[0-9]+",
      x,
      ignore.case = TRUE
    )
  )

  grade_valid <- is.na(result) & grade_matches != ""

  grades <- suppressWarnings(
    as.numeric(
      sub(
        ".*Grade[[:space:]]*",
        "",
        grade_matches[grade_valid],
        ignore.case = TRUE
      )
    )
  )

  # Approximate mapping:
  # Grade 8 -> NQF 1
  # Grade 9 -> NQF 1
  # Grade 10 -> NQF 2
  # Grade 11 -> NQF 3
  # Grade 12 -> NQF 4

  result[grade_valid] <- pmax(
    grades - 8,
    1
  )

  result
}


train_data$education_nqf <- extract_nqf(
  train_data$education_level
)

test_data$education_nqf <- extract_nqf(
  test_data$education_level
)

train_data$education_nqf[
  is.na(train_data$education_nqf)
] <- 0

test_data$education_nqf[
  is.na(test_data$education_nqf)
] <- 0


# ------------------------------------------------------------
# 9. CATEGORICAL VARIABLES
# ------------------------------------------------------------
#
# We only use low/moderate-cardinality variables.
#
# IMPORTANT:
# We create factors using the TRAINING levels and force unknown
# test levels to NA.
# ------------------------------------------------------------

categorical_vars <- c(
  "gender",
  "race",
  "province",
  "education_schooling_grade_twelve_equiv",
  "institution_type"
)


for (v in categorical_vars) {

  train_values <- as.character(train_data[[v]])
  test_values  <- as.character(test_data[[v]])

  # Replace missing values with explicit category
  train_values[is.na(train_values) | train_values == ""] <- "MISSING"
  test_values[is.na(test_values) | test_values == ""] <- "MISSING"

  # Training levels
  train_levels <- unique(train_values)

  # Any unseen test categories become MISSING
  test_values[
    !(test_values %in% train_levels)
  ] <- "MISSING"

  # Make factors with EXACTLY the same levels
  train_data[[v]] <- factor(
    train_values,
    levels = train_levels
  )

  test_data[[v]] <- factor(
    test_values,
    levels = train_levels
  )
}


# ------------------------------------------------------------
# 10. BUILD MODEL DATA
# ------------------------------------------------------------

model_vars <- c(
  "employed_status",

  # Strong numerical predictors
  "wr_filled",
  "has_work_readiness",
  "age_filled",
  "school_quintile_filled",
  "education_nqf",

  # Historical information
  "employed_lag_filled",
  "has_previous_observation",
  "total_historical_rounds_filled",

  # Demographics
  "gender",
  "race",
  "province",

  # Education
  "education_schooling_grade_twelve_equiv",
  "institution_type"
)

model_train <- train_data[
  ,
  model_vars,
  drop = FALSE
]

model_test <- test_data[
  ,
  setdiff(model_vars, "employed_status"),
  drop = FALSE
]


# ------------------------------------------------------------
# 11. REMOVE ANY ZERO-VARIANCE CATEGORICAL VARIABLES
# ------------------------------------------------------------

categorical_model_vars <- categorical_vars[
  sapply(
    model_train[categorical_vars],
    function(x) length(unique(x[!is.na(x)])) >= 2
  )
]

cat("\nCategorical variables used:\n")
print(categorical_model_vars)


# Rebuild model variables using only valid categorical variables

numeric_vars <- c(
  "wr_filled",
  "has_work_readiness",
  "age_filled",
  "school_quintile_filled",
  "education_nqf",
  "employed_lag_filled",
  "has_previous_observation",
  "total_historical_rounds_filled"
)

final_model_vars <- c(
  numeric_vars,
  categorical_model_vars
)


model_train <- train_data[
  ,
  c("employed_status", final_model_vars),
  drop = FALSE
]

model_test <- test_data[
  ,
  final_model_vars,
  drop = FALSE
]


# ------------------------------------------------------------
# 12. FIT MODEL
# ------------------------------------------------------------

cat("\nTraining Experiment 4...\n")

model_formula <- as.formula(
  paste(
    "employed_status ~",
    paste(final_model_vars, collapse = " + ")
  )
)

exp4_model <- glm(
  formula = model_formula,
  data = model_train,
  family = binomial(link = "logit")
)

cat("\nModel trained successfully.\n")

print(summary(exp4_model))


# ------------------------------------------------------------
# 13. PREDICT ROUND 9
# ------------------------------------------------------------

cat("\nGenerating Round 9 predictions...\n")

predictions <- predict(
  exp4_model,
  newdata = model_test,
  type = "response"
)


# ------------------------------------------------------------
# 14. CHECK PREDICTIONS
# ------------------------------------------------------------

cat("\nPrediction summary:\n")
print(summary(predictions))

cat(
  "\nNumber of NA predictions:",
  sum(is.na(predictions)),
  "\n"
)

cat(
  "Number of predictions:",
  length(predictions),
  "\n"
)


# ------------------------------------------------------------
# 15. FALLBACK FOR ANY NA PREDICTIONS
# ------------------------------------------------------------

if (any(is.na(predictions))) {

  cat(
    "\nWARNING: NA predictions detected.\n"
  )

  # Use training employment rate as fallback
  fallback_probability <- mean(
    model_train$employed_status
  )

  predictions[is.na(predictions)] <-
    fallback_probability

  cat(
    "Replaced NA predictions with training employment rate:",
    fallback_probability,
    "\n"
  )
}


# ------------------------------------------------------------
# 16. CREATE SUBMISSION
# ------------------------------------------------------------

submission <- data.frame(
  anonymised_id = test_data$anonymised_id,
  employed_status = predictions
)


# ------------------------------------------------------------
# 17. FINAL VALIDATION
# ------------------------------------------------------------

stopifnot(
  nrow(submission) == nrow(test_data)
)

stopifnot(
  all(!is.na(submission$employed_status))
)

stopifnot(
  all(
    submission$employed_status >= 0 &
    submission$employed_status <= 1
  )
)


# ------------------------------------------------------------
# 18. SAVE SUBMISSION
# ------------------------------------------------------------

output_file <- "submission_exp4_controlled_glm.csv"

write.csv(
  submission,
  output_file,
  row.names = FALSE,
  quote = FALSE
)

cat("\n============================================\n")
cat("EXPERIMENT 4 COMPLETE\n")
cat("============================================\n")
cat("Saved:", output_file, "\n")
cat("Rows:", nrow(submission), "\n")
cat("\nFirst 10 predictions:\n")

print(
  head(submission, 10)
)

Training rows: 16719 
Testing rows : 3230 

Target distribution:

    0     1 
11375  5278 


Warning message in result[valid] <- suppressWarnings(as.numeric(sub(".*NQF Level[[:space:]]*", :
"number of items to replace is not a multiple of replacement length"
Warning message in is.na(result) & grade_matches != "":
"longer object length is not a multiple of shorter object length"
Warning message in result[valid] <- suppressWarnings(as.numeric(sub(".*NQF Level[[:space:]]*", :
"number of items to replace is not a multiple of replacement length"
Warning message in is.na(result) & grade_matches != "":
"longer object length is not a multiple of shorter object length"



Categorical variables used:
[1] "gender"                                
[2] "race"                                  
[3] "province"                              
[4] "education_schooling_grade_twelve_equiv"
[5] "institution_type"                      

Training Experiment 4...

Model trained successfully.

Call:
glm(formula = model_formula, family = binomial(link = "logit"), 
    data = model_train)

Coefficients:
                                                       Estimate Std. Error
(Intercept)                                           -2.210320   0.447455
wr_filled                                              0.767062   0.111789
has_work_readiness                                     0.032758   0.040901
age_filled                                             0.027603   0.004096
school_quintile_filled                                 0.008533   0.013137
education_nqf                                          0.016434   0.029902
employed_lag_filled                                    

In [27]:
# ============================================================
# EXPERIMENT 5 — REGULARISED LOGISTIC REGRESSION
# ============================================================
#
# Goal:
# Test whether regularisation can improve on the original
# GLM by handling many correlated/numerous features more robustly.
#
# Model:
# Elastic-net / Ridge-style logistic regression using glmnet.
#
# Output:
# submission_exp5_regularised.csv
#
# ============================================================


# ------------------------------------------------------------
# 1. INSTALL / LOAD PACKAGE
# ------------------------------------------------------------

if (!requireNamespace("glmnet", quietly = TRUE)) {
  install.packages("glmnet")
}

library(glmnet)


# ------------------------------------------------------------
# 2. LOAD DATA
# ------------------------------------------------------------

data_dir <- "../../assets/dataset"

train_path <- file.path(data_dir, "train.csv")
test_path  <- file.path(data_dir, "test.csv")

if (!file.exists(train_path)) {
  stop(
    "Training file not found:\n",
    normalizePath(train_path, mustWork = FALSE)
  )
}

if (!file.exists(test_path)) {
  stop(
    "Testing file not found:\n",
    normalizePath(test_path, mustWork = FALSE)
  )
}

train_data <- read.csv(
  train_path,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

test_data <- read.csv(
  test_path,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

cat("Training rows:", nrow(train_data), "\n")
cat("Testing rows :", nrow(test_data), "\n\n")


# ------------------------------------------------------------
# 3. REMOVE ROWS WITH MISSING TARGET
# ------------------------------------------------------------

train_data <- train_data[
  !is.na(train_data$employed_status),
]

train_data$employed_status <- as.integer(
  train_data$employed_status
)

cat("Training rows after removing missing target:",
    nrow(train_data), "\n\n")

cat("Target distribution:\n")
print(table(train_data$employed_status))
cat("\n")


# ------------------------------------------------------------
# 4. FEATURE ENGINEERING
# ------------------------------------------------------------

# ---- Work readiness ----

wr_median <- median(
  train_data$work_readiness_score,
  na.rm = TRUE
)

if (!is.finite(wr_median)) {
  wr_median <- 0.5
}

train_data$wr_filled <- ifelse(
  is.na(train_data$work_readiness_score),
  wr_median,
  train_data$work_readiness_score
)

test_data$wr_filled <- ifelse(
  is.na(test_data$work_readiness_score),
  wr_median,
  test_data$work_readiness_score
)

train_data$has_work_readiness <-
  as.integer(!is.na(train_data$work_readiness_score))

test_data$has_work_readiness <-
  as.integer(!is.na(test_data$work_readiness_score))


# ---- Previous employment ----

train_data$employed_lag_filled <-
  ifelse(
    is.na(train_data$employed_lag),
    -1,
    train_data$employed_lag
  )

test_data$employed_lag_filled <-
  ifelse(
    is.na(test_data$employed_lag),
    -1,
    test_data$employed_lag
  )

train_data$has_previous_observation <-
  as.integer(!is.na(train_data$employed_lag))

test_data$has_previous_observation <-
  as.integer(!is.na(test_data$employed_lag))


# ---- Tenure ----

tenure_median <- median(
  train_data$tenure_lag,
  na.rm = TRUE
)

if (!is.finite(tenure_median)) {
  tenure_median <- 0
}

train_data$tenure_lag_filled <-
  ifelse(
    is.na(train_data$tenure_lag),
    tenure_median,
    train_data$tenure_lag
  )

test_data$tenure_lag_filled <-
  ifelse(
    is.na(test_data$tenure_lag),
    tenure_median,
    test_data$tenure_lag
  )

train_data$has_tenure <-
  as.integer(!is.na(train_data$tenure_lag))

test_data$has_tenure <-
  as.integer(!is.na(test_data$tenure_lag))


# ---- Days since previous observation ----

days_median <- median(
  train_data$days_since_last_obs,
  na.rm = TRUE
)

if (!is.finite(days_median)) {
  days_median <- 0
}

train_data$days_since_last_obs_filled <-
  ifelse(
    is.na(train_data$days_since_last_obs),
    days_median,
    train_data$days_since_last_obs
  )

test_data$days_since_last_obs_filled <-
  ifelse(
    is.na(test_data$days_since_last_obs),
    days_median,
    test_data$days_since_last_obs
  )


# ---- Lag round ----

lag_round_median <- median(
  train_data$lag_round,
  na.rm = TRUE
)

if (!is.finite(lag_round_median)) {
  lag_round_median <- 0
}

train_data$lag_round_filled <-
  ifelse(
    is.na(train_data$lag_round),
    lag_round_median,
    train_data$lag_round
  )

test_data$lag_round_filled <-
  ifelse(
    is.na(test_data$lag_round),
    lag_round_median,
    test_data$lag_round
  )


# ------------------------------------------------------------
# 5. SIMPLE EDUCATION NQF EXTRACTION
# ------------------------------------------------------------

extract_nqf <- function(x) {

  x <- as.character(x)

  result <- rep(NA_real_, length(x))

  # Look for "NQF Level X"
  matches <- regmatches(
    x,
    regexpr(
      "NQF Level [0-9]+",
      x,
      ignore.case = TRUE
    )
  )

  valid <- matches != ""

  result[valid] <- suppressWarnings(
    as.numeric(
      sub(
        ".*NQF Level ",
        "",
        matches[valid],
        ignore.case = TRUE
      )
    )
  )

  # Handle Grade X where possible
  grade_matches <- regmatches(
    x,
    regexpr(
      "Grade [0-9]+",
      x,
      ignore.case = TRUE
    )
  )

  grade_valid <- (
    is.na(result) &
    grade_matches != ""
  )

  if (any(grade_valid)) {

    grades <- suppressWarnings(
      as.numeric(
        sub(
          ".*Grade ",
          "",
          grade_matches[grade_valid],
          ignore.case = TRUE
        )
      )
    )

    result[grade_valid] <- pmax(
      grades - 8,
      1
    )
  }

  result
}


train_data$education_nqf <-
  extract_nqf(train_data$education_level)

test_data$education_nqf <-
  extract_nqf(test_data$education_level)


# Fill missing NQF using training median

nqf_median <- median(
  train_data$education_nqf,
  na.rm = TRUE
)

if (!is.finite(nqf_median)) {
  nqf_median <- 4
}

train_data$education_nqf <-
  ifelse(
    is.na(train_data$education_nqf),
    nqf_median,
    train_data$education_nqf
  )

test_data$education_nqf <-
  ifelse(
    is.na(test_data$education_nqf),
    nqf_median,
    test_data$education_nqf
  )


# ------------------------------------------------------------
# 6. CONVERT VARIABLES TO CONSISTENT CHARACTER VALUES
# ------------------------------------------------------------

# We deliberately do this BEFORE model.matrix().
# This prevents the "new factor level" errors encountered
# in previous experiments.

categorical_vars <- c(
  "gender",
  "race",
  "province",
  "district",
  "municipality",
  "status_lag",
  "status_broad_lag",
  "education_level",
  "education_schooling_grade_twelve_equiv",
  "institution_type",
  "education_field",
  "seta",
  "sample",
  "sample_first"
)

categorical_vars <- categorical_vars[
  categorical_vars %in% names(train_data)
]


for (v in categorical_vars) {

  train_data[[v]] <- as.character(
    train_data[[v]]
  )

  test_data[[v]] <- as.character(
    test_data[[v]]
  )

  # Missing values become an explicit category.
  train_data[[v]][
    is.na(train_data[[v]]) |
    train_data[[v]] == ""
  ] <- "__MISSING__"

  test_data[[v]][
    is.na(test_data[[v]]) |
    test_data[[v]] == ""
  ] <- "__MISSING__"
}


# ------------------------------------------------------------
# 7. KEEP ONLY FEATURES WE WANT
# ------------------------------------------------------------

numeric_features <- c(
  "current_round",
  "total_historical_rounds",
  "age",
  "school_quintile",
  "wr_filled",
  "employed_lag_filled",
  "tenure_lag_filled",
  "days_since_last_obs_filled",
  "lag_round_filled",
  "education_nqf",
  "has_previous_observation",
  "has_work_readiness",
  "has_tenure"
)

numeric_features <- numeric_features[
  numeric_features %in% names(train_data)
]

feature_names <- c(
  numeric_features,
  categorical_vars
)

feature_names <- feature_names[
  feature_names %in% names(train_data) &
  feature_names %in% names(test_data)
]

cat("Number of features before encoding:",
    length(feature_names), "\n")

cat("Categorical variables:\n")
print(categorical_vars)

cat("\n")


# ------------------------------------------------------------
# 8. BUILD COMBINED DATA FOR IDENTICAL ENCODING
# ------------------------------------------------------------

# Combining train + test before model.matrix ensures that
# categories appearing only in Round 9 are not treated as
# completely unknown columns.

combined <- rbind(
  train_data[, feature_names, drop = FALSE],
  test_data[, feature_names, drop = FALSE]
)


# ------------------------------------------------------------
# 9. CLEAN NUMERIC VARIABLES
# ------------------------------------------------------------

for (v in numeric_features) {

  combined[[v]] <- suppressWarnings(
    as.numeric(combined[[v]])
  )

  med <- median(
    combined[[v]],
    na.rm = TRUE
  )

  if (!is.finite(med)) {
    med <- 0
  }

  combined[[v]][
    is.na(combined[[v]])
  ] <- med
}


# ------------------------------------------------------------
# 10. BUILD MODEL MATRIX
# ------------------------------------------------------------

# Convert categorical variables to one-hot encoded columns.

for (v in categorical_vars) {
  combined[[v]] <- factor(
    combined[[v]]
  )
}

model_matrix <- model.matrix(
  ~ . - 1,
  data = combined
)

cat("Encoded feature count:",
    ncol(model_matrix), "\n\n")


# ------------------------------------------------------------
# 11. SPLIT BACK INTO TRAIN / TEST
# ------------------------------------------------------------

n_train <- nrow(train_data)

x_train <- model_matrix[
  1:n_train,
  ,
  drop = FALSE
]

x_test <- model_matrix[
  (n_train + 1):nrow(model_matrix),
  ,
  drop = FALSE
]

y_train <- train_data$employed_status


# ------------------------------------------------------------
# 12. REMOVE ZERO-VARIANCE COLUMNS
# ------------------------------------------------------------

feature_sd <- apply(
  x_train,
  2,
  sd
)

keep_features <- (
  !is.na(feature_sd) &
  feature_sd > 0
)

x_train <- x_train[
  ,
  keep_features,
  drop = FALSE
]

x_test <- x_test[
  ,
  keep_features,
  drop = FALSE
]

cat("Features after zero-variance removal:",
    ncol(x_train), "\n\n")


# ------------------------------------------------------------
# 13. TRAIN REGULARISED LOGISTIC MODELS
# ------------------------------------------------------------

cat("Training regularised logistic regression...\n\n")

set.seed(42)

# We use cross-validation to find the best lambda.
#
# alpha = 0   -> Ridge
# alpha = 1   -> Lasso
#
# We test alpha = 0.5, which gives Elastic Net behaviour.

cv_model <- cv.glmnet(
  x = x_train,
  y = y_train,
  family = "binomial",
  alpha = 0.5,
  nfolds = 5,
  type.measure = "auc",
  standardize = TRUE,
  maxit = 100000
)


# ------------------------------------------------------------
# 14. SHOW CROSS-VALIDATION RESULT
# ------------------------------------------------------------

cat("Best lambda:", cv_model$lambda.min, "\n")
cat("1-SE lambda:", cv_model$lambda.1se, "\n\n")

cat("Best cross-validation AUC:",
    max(cv_model$cvm),
    "\n\n")


# ------------------------------------------------------------
# 15. GENERATE ROUND 9 PREDICTIONS
# ------------------------------------------------------------

cat("# GENERATING ROUND 9 PREDICTIONS\n\n")

test_prob <- predict(
  cv_model,
  newx = x_test,
  s = "lambda.min",
  type = "response"
)

test_prob <- as.numeric(
  test_prob
)


# ------------------------------------------------------------
# 16. SAFETY CHECK
# ------------------------------------------------------------

test_prob[is.na(test_prob)] <- 0.5

test_prob <- pmin(
  pmax(test_prob, 0),
  1
)

cat("Prediction summary:\n")
print(
  summary(test_prob)
)

cat("\n")


# ------------------------------------------------------------
# 17. CREATE SUBMISSION
# ------------------------------------------------------------

submission <- data.frame(
  anonymised_id = test_data$anonymised_id,
  employed_status = test_prob
)


# ------------------------------------------------------------
# 18. FINAL VALIDATION
# ------------------------------------------------------------

if (nrow(submission) != nrow(test_data)) {
  stop("Submission row count does not match test data.")
}

if (any(is.na(submission$employed_status))) {
  stop("Submission contains NA predictions.")
}

if (any(
  submission$employed_status < 0 |
  submission$employed_status > 1
)) {
  stop("Predictions must be between 0 and 1.")
}

if (anyDuplicated(submission$anonymised_id)) {
  stop("Duplicate anonymised_id values detected.")
}


# ------------------------------------------------------------
# 19. SAVE SUBMISSION
# ------------------------------------------------------------

output_file <- "submission_exp5_regularised.csv"

write.csv(
  submission,
  output_file,
  row.names = FALSE,
  quote = FALSE
)


# ------------------------------------------------------------
# 20. DISPLAY RESULT
# ------------------------------------------------------------

cat("============================================\n")
cat("EXPERIMENT 5 COMPLETE\n")
cat("============================================\n")
cat("Saved:", output_file, "\n")
cat("Rows:", nrow(submission), "\n")
cat("Features:", ncol(x_train), "\n")
cat("Lambda:", cv_model$lambda.min, "\n\n")

cat("First 10 predictions:\n")
print(
  head(submission, 10)
)

Warning message:
"package 'glmnet' was built under R version 4.4.3"
Loading required package: Matrix

Loaded glmnet 4.1-10



Training rows: 16719 
Testing rows : 3230 

Training rows after removing missing target: 16653 

Target distribution:

    0     1 
11375  5278 



Warning message in result[valid] <- suppressWarnings(as.numeric(sub(".*NQF Level ", :
"number of items to replace is not a multiple of replacement length"
Warning message in is.na(result) & grade_matches != "":
"longer object length is not a multiple of shorter object length"
Warning message in result[valid] <- suppressWarnings(as.numeric(sub(".*NQF Level ", :
"number of items to replace is not a multiple of replacement length"
Warning message in is.na(result) & grade_matches != "":
"longer object length is not a multiple of shorter object length"


Number of features before encoding: 27 
Categorical variables:
 [1] "gender"                                
 [2] "race"                                  
 [3] "province"                              
 [4] "district"                              
 [5] "municipality"                          
 [6] "status_lag"                            
 [7] "status_broad_lag"                      
 [8] "education_level"                       
 [9] "education_schooling_grade_twelve_equiv"
[10] "institution_type"                      
[11] "education_field"                       
[12] "seta"                                  
[13] "sample"                                
[14] "sample_first"                          

Encoded feature count: 592 

Features after zero-variance removal: 584 

Training regularised logistic regression...

Best lambda: 0.01227164 
1-SE lambda: 0.01953992 

Best cross-validation AUC: 0.6633061 

# GENERATING ROUND 9 PREDICTIONS

Prediction summary:
   Min. 1st Qu.  Median    Mea

In [28]:
# ============================================================
# EXPERIMENT 6 — NONLINEAR GLM
# ============================================================
#
# Goal:
# Test whether nonlinear relationships improve employment
# prediction compared with the regularised model.
#
# New features:
#   - age_squared
#   - wr_squared
#   - age × work readiness
#
# We keep the model controlled to avoid the high-cardinality
# categorical variables that caused instability previously.
# ============================================================

cat("============================================\n")
cat("EXPERIMENT 6 — NONLINEAR GLM\n")
cat("============================================\n\n")

# ------------------------------------------------------------
# 1. LOAD DATA
# ------------------------------------------------------------

data_dir <- "../../assets/dataset"

train_data <- read.csv(
  file.path(data_dir, "train.csv"),
  stringsAsFactors = FALSE
)

test_data <- read.csv(
  file.path(data_dir, "test.csv"),
  stringsAsFactors = FALSE
)

cat("Training rows:", nrow(train_data), "\n")
cat("Testing rows :", nrow(test_data), "\n\n")


# ------------------------------------------------------------
# 2. REMOVE ROWS WITH MISSING TARGET
# ------------------------------------------------------------

train_data <- train_data[
  !is.na(train_data$employed_status),
]

cat("Training rows after removing missing target:",
    nrow(train_data), "\n\n")

cat("Target distribution:\n")
print(table(train_data$employed_status))
cat("\n")


# ------------------------------------------------------------
# 3. WORK READINESS
# ------------------------------------------------------------

wr_median <- median(
  train_data$work_readiness_score,
  na.rm = TRUE
)

train_data$wr_filled <- ifelse(
  is.na(train_data$work_readiness_score),
  wr_median,
  train_data$work_readiness_score
)

test_data$wr_filled <- ifelse(
  is.na(test_data$work_readiness_score),
  wr_median,
  test_data$work_readiness_score
)

# Missingness indicator
train_data$has_work_readiness <- as.integer(
  !is.na(train_data$work_readiness_score)
)

test_data$has_work_readiness <- as.integer(
  !is.na(test_data$work_readiness_score)
)


# ------------------------------------------------------------
# 4. PREVIOUS EMPLOYMENT
# ------------------------------------------------------------

employed_lag_median <- median(
  train_data$employed_lag,
  na.rm = TRUE
)

train_data$employed_lag_filled <- ifelse(
  is.na(train_data$employed_lag),
  employed_lag_median,
  train_data$employed_lag
)

test_data$employed_lag_filled <- ifelse(
  is.na(test_data$employed_lag),
  employed_lag_median,
  test_data$employed_lag
)

# Explicitly record whether previous employment exists
train_data$has_previous_observation <- as.integer(
  !is.na(train_data$employed_lag)
)

test_data$has_previous_observation <- as.integer(
  !is.na(test_data$employed_lag)
)


# ------------------------------------------------------------
# 5. HISTORICAL ROUND FEATURES
# ------------------------------------------------------------

train_data$total_historical_rounds_filled <- ifelse(
  is.na(train_data$total_historical_rounds),
  1,
  train_data$total_historical_rounds
)

test_data$total_historical_rounds_filled <- ifelse(
  is.na(test_data$total_historical_rounds),
  1,
  test_data$total_historical_rounds
)


# ------------------------------------------------------------
# 6. AGE
# ------------------------------------------------------------

age_median <- median(
  train_data$age,
  na.rm = TRUE
)

train_data$age_filled <- ifelse(
  is.na(train_data$age),
  age_median,
  train_data$age
)

test_data$age_filled <- ifelse(
  is.na(test_data$age),
  age_median,
  test_data$age
)


# ------------------------------------------------------------
# 7. CREATE NONLINEAR FEATURES
# ------------------------------------------------------------

# Age squared
train_data$age_squared <- train_data$age_filled^2
test_data$age_squared <- test_data$age_filled^2

# Work readiness squared
train_data$wr_squared <- train_data$wr_filled^2
test_data$wr_squared <- test_data$wr_filled^2

# Interaction between age and work readiness
train_data$age_wr_interaction <-
  train_data$age_filled * train_data$wr_filled

test_data$age_wr_interaction <-
  test_data$age_filled * test_data$wr_filled


# ------------------------------------------------------------
# 8. TENURE
# ------------------------------------------------------------

tenure_median <- median(
  train_data$tenure_lag,
  na.rm = TRUE
)

train_data$tenure_lag_filled <- ifelse(
  is.na(train_data$tenure_lag),
  tenure_median,
  train_data$tenure_lag
)

test_data$tenure_lag_filled <- ifelse(
  is.na(test_data$tenure_lag),
  tenure_median,
  test_data$tenure_lag
)

train_data$has_tenure <- as.integer(
  !is.na(train_data$tenure_lag)
)

test_data$has_tenure <- as.integer(
  !is.na(test_data$tenure_lag)
)


# ------------------------------------------------------------
# 9. DAYS SINCE LAST OBSERVATION
# ------------------------------------------------------------

days_median <- median(
  train_data$days_since_last_obs,
  na.rm = TRUE
)

train_data$days_since_last_obs_filled <- ifelse(
  is.na(train_data$days_since_last_obs),
  days_median,
  train_data$days_since_last_obs
)

test_data$days_since_last_obs_filled <- ifelse(
  is.na(test_data$days_since_last_obs),
  days_median,
  test_data$days_since_last_obs
)


# ------------------------------------------------------------
# 10. LAG ROUND
# ------------------------------------------------------------

lag_round_median <- median(
  train_data$lag_round,
  na.rm = TRUE
)

train_data$lag_round_filled <- ifelse(
  is.na(train_data$lag_round),
  lag_round_median,
  train_data$lag_round
)

test_data$lag_round_filled <- ifelse(
  is.na(test_data$lag_round),
  lag_round_median,
  test_data$lag_round
)


# ------------------------------------------------------------
# 11. EDUCATION — SIMPLE NQF EXTRACTION
# ------------------------------------------------------------

extract_nqf <- function(x) {

  x <- as.character(x)

  result <- rep(NA_real_, length(x))

  # Extract "NQF Level X"
  matches <- regmatches(
    x,
    regexpr(
      "NQF Level [0-9]+",
      x
    )
  )

  valid <- matches != ""

  result[valid] <- suppressWarnings(
    as.numeric(
      sub(
        "NQF Level ",
        "",
        matches[valid]
      )
    )
  )

  # Handle Grade 12
  grade12 <- grepl(
    "Grade 12",
    x,
    ignore.case = TRUE
  )

  result[
    grade12 & is.na(result)
  ] <- 4

  # Handle Grade 11
  grade11 <- grepl(
    "Grade 11",
    x,
    ignore.case = TRUE
  )

  result[
    grade11 & is.na(result)
  ] <- 3

  # Handle Grade 10
  grade10 <- grepl(
    "Grade 10",
    x,
    ignore.case = TRUE
  )

  result[
    grade10 & is.na(result)
  ] <- 2

  # Handle Grade 9
  grade9 <- grepl(
    "Grade 9",
    x,
    ignore.case = TRUE
  )

  result[
    grade9 & is.na(result)
  ] <- 1

  result
}


train_data$education_nqf <-
  extract_nqf(train_data$education_level)

test_data$education_nqf <-
  extract_nqf(test_data$education_level)

education_median <- median(
  train_data$education_nqf,
  na.rm = TRUE
)

train_data$education_nqf <- ifelse(
  is.na(train_data$education_nqf),
  education_median,
  train_data$education_nqf
)

test_data$education_nqf <- ifelse(
  is.na(test_data$education_nqf),
  education_median,
  test_data$education_nqf
)


# ------------------------------------------------------------
# 12. MODEL FORMULA
# ------------------------------------------------------------

model_formula <- employed_status ~
  age_filled +
  age_squared +
  wr_filled +
  wr_squared +
  age_wr_interaction +
  employed_lag_filled +
  total_historical_rounds_filled +
  tenure_lag_filled +
  days_since_last_obs_filled +
  lag_round_filled +
  education_nqf +
  has_previous_observation +
  has_work_readiness +
  has_tenure


# ------------------------------------------------------------
# 13. TRAIN MODEL
# ------------------------------------------------------------

cat("Training nonlinear logistic regression...\n\n")

model <- glm(
  formula = model_formula,
  data = train_data,
  family = binomial(link = "logit")
)

cat("Model trained successfully.\n\n")

print(summary(model))


# ------------------------------------------------------------
# 14. GENERATE ROUND 9 PREDICTIONS
# ------------------------------------------------------------

cat("\n============================================\n")
cat("GENERATING ROUND 9 PREDICTIONS\n")
cat("============================================\n\n")

test_prob <- predict(
  model,
  newdata = test_data,
  type = "response"
)


# ------------------------------------------------------------
# 15. CHECK PREDICTIONS
# ------------------------------------------------------------

cat("Prediction summary:\n")
print(summary(test_prob))

cat("\nNumber of NA predictions:",
    sum(is.na(test_prob)),
    "\n\n")


# ------------------------------------------------------------
# 16. HANDLE ANY UNEXPECTED NA PREDICTIONS
# ------------------------------------------------------------

if (any(is.na(test_prob))) {

  cat(
    "WARNING: NA predictions detected.\n"
  )

  # Replace unexpected NA predictions with
  # the mean training employment rate.
  fallback_probability <- mean(
    train_data$employed_status,
    na.rm = TRUE
  )

  test_prob[is.na(test_prob)] <-
    fallback_probability

  cat(
    "Replaced NA predictions with training employment rate:",
    fallback_probability,
    "\n"
  )
}


# ------------------------------------------------------------
# 17. CREATE SUBMISSION
# ------------------------------------------------------------

submission <- data.frame(
  anonymised_id = test_data$anonymised_id,
  employed_status = test_prob
)


# ------------------------------------------------------------
# 18. FINAL VALIDATION
# ------------------------------------------------------------

stopifnot(
  nrow(submission) == nrow(test_data)
)

stopifnot(
  all(!is.na(submission$employed_status))
)

stopifnot(
  all(
    submission$employed_status >= 0 &
    submission$employed_status <= 1
  )
)


# ------------------------------------------------------------
# 19. SAVE
# ------------------------------------------------------------

output_file <-
  "submission_exp6_nonlinear.csv"

write.csv(
  submission,
  output_file,
  row.names = FALSE,
  quote = FALSE
)

cat("\n============================================\n")
cat("SUBMISSION CREATED\n")
cat("============================================\n")

cat("File:", output_file, "\n")
cat("Rows:", nrow(submission), "\n\n")

cat("First 10 predictions:\n")
print(head(submission, 10))

EXPERIMENT 6 — NONLINEAR GLM

Training rows: 16719 
Testing rows : 3230 

Training rows after removing missing target: 16653 

Target distribution:

    0     1 
11375  5278 



Warning message in result[valid] <- suppressWarnings(as.numeric(sub("NQF Level ", :
"number of items to replace is not a multiple of replacement length"
Warning message in result[valid] <- suppressWarnings(as.numeric(sub("NQF Level ", :
"number of items to replace is not a multiple of replacement length"


Training nonlinear logistic regression...

Model trained successfully.


Call:
glm(formula = model_formula, family = binomial(link = "logit"), 
    data = train_data)

Coefficients:
                                 Estimate Std. Error z value Pr(>|z|)    
(Intercept)                    -3.661e+00  5.894e-01  -6.212 5.22e-10 ***
age_filled                      1.130e-01  3.868e-02   2.921 0.003489 ** 
age_squared                    -1.361e-03  6.965e-04  -1.954 0.050655 .  
wr_filled                       2.837e+00  7.675e-01   3.696 0.000219 ***
wr_squared                     -8.740e-01  4.436e-01  -1.970 0.048813 *  
age_wr_interaction             -4.417e-02  2.647e-02  -1.669 0.095126 .  
employed_lag_filled             1.120e+00  6.576e-02  17.026  < 2e-16 ***
total_historical_rounds_filled  3.613e-02  6.210e-02   0.582 0.560694    
tenure_lag_filled              -1.147e-04  4.995e-05  -2.296 0.021666 *  
days_since_last_obs_filled      2.382e-04  3.113e-04   0.765 0.444116    
lag_

In [29]:
# ============================================================
# EXPERIMENT 7 — STATUS_LAG + CONTROLLED GLM
# ============================================================
#
# Goal:
# Test whether previous detailed labour-market status
# provides useful predictive information when combined
# with the strongest numerical features.
#
# Important:
#   - status_lag is converted to a controlled categorical
#     variable.
#   - Rare categories are grouped into OTHER.
#   - Test-only categories are also mapped to OTHER.
#   - No nonlinear terms are used.
#
# Output:
#   submission_exp7_status_lag.csv
# ============================================================


cat("============================================\n")
cat("EXPERIMENT 7 — STATUS_LAG MODEL\n")
cat("============================================\n\n")


# ------------------------------------------------------------
# 1. LOAD DATA
# ------------------------------------------------------------

data_dir <- "../../assets/dataset"

train_data <- read.csv(
  file.path(data_dir, "train.csv"),
  stringsAsFactors = FALSE
)

test_data <- read.csv(
  file.path(data_dir, "test.csv"),
  stringsAsFactors = FALSE
)

cat("Training rows:", nrow(train_data), "\n")
cat("Testing rows :", nrow(test_data), "\n\n")


# ------------------------------------------------------------
# 2. REMOVE MISSING TARGET
# ------------------------------------------------------------

train_data <- train_data[
  !is.na(train_data$employed_status),
]

cat(
  "Training rows after removing missing target:",
  nrow(train_data),
  "\n\n"
)

cat("Target distribution:\n")
print(table(train_data$employed_status))
cat("\n")


# ------------------------------------------------------------
# 3. WORK READINESS
# ------------------------------------------------------------

wr_median <- median(
  train_data$work_readiness_score,
  na.rm = TRUE
)

train_data$wr_filled <- ifelse(
  is.na(train_data$work_readiness_score),
  wr_median,
  train_data$work_readiness_score
)

test_data$wr_filled <- ifelse(
  is.na(test_data$work_readiness_score),
  wr_median,
  test_data$work_readiness_score
)

train_data$has_work_readiness <- as.integer(
  !is.na(train_data$work_readiness_score)
)

test_data$has_work_readiness <- as.integer(
  !is.na(test_data$work_readiness_score)
)


# ------------------------------------------------------------
# 4. PREVIOUS EMPLOYMENT
# ------------------------------------------------------------

employed_lag_median <- median(
  train_data$employed_lag,
  na.rm = TRUE
)

train_data$employed_lag_filled <- ifelse(
  is.na(train_data$employed_lag),
  employed_lag_median,
  train_data$employed_lag
)

test_data$employed_lag_filled <- ifelse(
  is.na(test_data$employed_lag),
  employed_lag_median,
  test_data$employed_lag
)

train_data$has_previous_observation <- as.integer(
  !is.na(train_data$employed_lag)
)

test_data$has_previous_observation <- as.integer(
  !is.na(test_data$employed_lag)
)


# ------------------------------------------------------------
# 5. HISTORICAL ROUND COUNT
# ------------------------------------------------------------

train_data$total_historical_rounds_filled <- ifelse(
  is.na(train_data$total_historical_rounds),
  1,
  train_data$total_historical_rounds
)

test_data$total_historical_rounds_filled <- ifelse(
  is.na(test_data$total_historical_rounds),
  1,
  test_data$total_historical_rounds
)


# ------------------------------------------------------------
# 6. AGE
# ------------------------------------------------------------

age_median <- median(
  train_data$age,
  na.rm = TRUE
)

train_data$age_filled <- ifelse(
  is.na(train_data$age),
  age_median,
  train_data$age
)

test_data$age_filled <- ifelse(
  is.na(test_data$age),
  age_median,
  test_data$age
)


# ------------------------------------------------------------
# 7. TENURE
# ------------------------------------------------------------

tenure_median <- median(
  train_data$tenure_lag,
  na.rm = TRUE
)

train_data$tenure_lag_filled <- ifelse(
  is.na(train_data$tenure_lag),
  tenure_median,
  train_data$tenure_lag
)

test_data$tenure_lag_filled <- ifelse(
  is.na(test_data$tenure_lag),
  tenure_median,
  test_data$tenure_lag
)

train_data$has_tenure <- as.integer(
  !is.na(train_data$tenure_lag)
)

test_data$has_tenure <- as.integer(
  !is.na(test_data$tenure_lag)
)


# ------------------------------------------------------------
# 8. DAYS SINCE LAST OBSERVATION
# ------------------------------------------------------------

days_median <- median(
  train_data$days_since_last_obs,
  na.rm = TRUE
)

train_data$days_since_last_obs_filled <- ifelse(
  is.na(train_data$days_since_last_obs),
  days_median,
  train_data$days_since_last_obs
)

test_data$days_since_last_obs_filled <- ifelse(
  is.na(test_data$days_since_last_obs),
  days_median,
  test_data$days_since_last_obs
)


# ------------------------------------------------------------
# 9. LAG ROUND
# ------------------------------------------------------------

lag_round_median <- median(
  train_data$lag_round,
  na.rm = TRUE
)

train_data$lag_round_filled <- ifelse(
  is.na(train_data$lag_round),
  lag_round_median,
  train_data$lag_round
)

test_data$lag_round_filled <- ifelse(
  is.na(test_data$lag_round),
  lag_round_median,
  test_data$lag_round
)


# ------------------------------------------------------------
# 10. EDUCATION NQF
# ------------------------------------------------------------

extract_nqf <- function(x) {

  x <- as.character(x)

  result <- rep(NA_real_, length(x))

  # Extract NQF Level X
  matches <- regmatches(
    x,
    regexpr(
      "NQF Level [0-9]+",
      x
    )
  )

  valid <- matches != ""

  if (any(valid)) {

    result[valid] <- suppressWarnings(
      as.numeric(
        sub(
          "NQF Level ",
          "",
          matches[valid]
        )
      )
    )

  }

  # Grade 12
  grade12 <- grepl(
    "Grade 12",
    x,
    ignore.case = TRUE
  )

  result[
    grade12 & is.na(result)
  ] <- 4

  # Grade 11
  grade11 <- grepl(
    "Grade 11",
    x,
    ignore.case = TRUE
  )

  result[
    grade11 & is.na(result)
  ] <- 3

  # Grade 10
  grade10 <- grepl(
    "Grade 10",
    x,
    ignore.case = TRUE
  )

  result[
    grade10 & is.na(result)
  ] <- 2

  # Grade 9
  grade9 <- grepl(
    "Grade 9",
    x,
    ignore.case = TRUE
  )

  result[
    grade9 & is.na(result)
  ] <- 1

  result
}


train_data$education_nqf <-
  extract_nqf(train_data$education_level)

test_data$education_nqf <-
  extract_nqf(test_data$education_level)


education_median <- median(
  train_data$education_nqf,
  na.rm = TRUE
)

train_data$education_nqf <- ifelse(
  is.na(train_data$education_nqf),
  education_median,
  train_data$education_nqf
)

test_data$education_nqf <- ifelse(
  is.na(test_data$education_nqf),
  education_median,
  test_data$education_nqf
)


# ------------------------------------------------------------
# 11. CLEAN STATUS_LAG
# ------------------------------------------------------------

train_status <- as.character(
  train_data$status_lag
)

test_status <- as.character(
  test_data$status_lag
)

# Convert missing values to explicit category
train_status[
  is.na(train_status) |
  train_status == ""
] <- "MISSING"

test_status[
  is.na(test_status) |
  test_status == ""
] <- "MISSING"


# ------------------------------------------------------------
# 12. GROUP RARE TRAINING CATEGORIES
# ------------------------------------------------------------

status_counts <- table(train_status)

# Categories appearing fewer than 50 times
rare_status <- names(
  status_counts[
    status_counts < 50
  ]
)

train_status[
  train_status %in% rare_status
] <- "OTHER"

# Anything in test that does not exist in training
# will also become OTHER.
valid_status_levels <- unique(
  train_status
)

test_status[
  !(test_status %in% valid_status_levels)
] <- "OTHER"


# Make sure OTHER exists in training if needed
if ("OTHER" %in% test_status &&
    !("OTHER" %in% train_status)) {

  train_status[
    length(train_status)
  ] <- "OTHER"

  valid_status_levels <- unique(
    train_status
  )
}


# Convert to factors using EXACT same levels
all_status_levels <- unique(
  c(
    train_status,
    "OTHER"
  )
)

train_data$status_lag_clean <- factor(
  train_status,
  levels = all_status_levels
)

test_data$status_lag_clean <- factor(
  test_status,
  levels = all_status_levels
)


cat("Status categories used:\n")
print(levels(train_data$status_lag_clean))
cat("\n")


# ------------------------------------------------------------
# 13. MODEL FORMULA
# ------------------------------------------------------------

model_formula <- employed_status ~
  status_lag_clean +
  age_filled +
  wr_filled +
  employed_lag_filled +
  total_historical_rounds_filled +
  tenure_lag_filled +
  days_since_last_obs_filled +
  lag_round_filled +
  education_nqf +
  has_previous_observation +
  has_work_readiness +
  has_tenure


cat("Training model...\n\n")


# ------------------------------------------------------------
# 14. TRAIN MODEL
# ------------------------------------------------------------

model <- glm(
  formula = model_formula,
  data = train_data,
  family = binomial(link = "logit")
)

cat("Model trained successfully.\n\n")

print(summary(model))


# ------------------------------------------------------------
# 15. PREDICT
# ------------------------------------------------------------

cat("\n============================================\n")
cat("GENERATING ROUND 9 PREDICTIONS\n")
cat("============================================\n\n")


test_prob <- predict(
  model,
  newdata = test_data,
  type = "response"
)


# ------------------------------------------------------------
# 16. CHECK PREDICTIONS
# ------------------------------------------------------------

cat("Prediction summary:\n")
print(summary(test_prob))

cat(
  "\nNumber of NA predictions:",
  sum(is.na(test_prob)),
  "\n"
)


# ------------------------------------------------------------
# 17. FALLBACK FOR ANY NA PREDICTIONS
# ------------------------------------------------------------

if (any(is.na(test_prob))) {

  cat(
    "\nWARNING: NA predictions detected.\n"
  )

  fallback_probability <- mean(
    train_data$employed_status,
    na.rm = TRUE
  )

  test_prob[
    is.na(test_prob)
  ] <- fallback_probability

  cat(
    "Replaced NA predictions with:",
    fallback_probability,
    "\n"
  )
}


# ------------------------------------------------------------
# 18. CREATE SUBMISSION
# ------------------------------------------------------------

submission <- data.frame(
  anonymised_id =
    test_data$anonymised_id,

  employed_status =
    test_prob
)


# ------------------------------------------------------------
# 19. VALIDATE
# ------------------------------------------------------------

stopifnot(
  nrow(submission) == nrow(test_data)
)

stopifnot(
  all(!is.na(submission$employed_status))
)

stopifnot(
  all(
    submission$employed_status >= 0 &
    submission$employed_status <= 1
  )
)


# ------------------------------------------------------------
# 20. SAVE SUBMISSION
# ------------------------------------------------------------

output_file <-
  "submission_exp7_status_lag.csv"

write.csv(
  submission,
  output_file,
  row.names = FALSE,
  quote = FALSE
)


# ------------------------------------------------------------
# 21. OUTPUT
# ------------------------------------------------------------

cat("\n============================================\n")
cat("SUBMISSION CREATED\n")
cat("============================================\n\n")

cat(
  "File:",
  output_file,
  "\n"
)

cat(
  "Rows:",
  nrow(submission),
  "\n\n"
)

cat("Prediction summary:\n")
print(summary(submission$employed_status))

cat("\nFirst 10 predictions:\n")
print(
  head(
    submission,
    10
  )
)

EXPERIMENT 7 — STATUS_LAG MODEL

Training rows: 16719 
Testing rows : 3230 

Training rows after removing missing target: 16653 

Target distribution:

    0     1 
11375  5278 



Warning message in result[valid] <- suppressWarnings(as.numeric(sub("NQF Level ", :
"number of items to replace is not a multiple of replacement length"
Warning message in result[valid] <- suppressWarnings(as.numeric(sub("NQF Level ", :
"number of items to replace is not a multiple of replacement length"


Status categories used:
[1] "MISSING"                "unemployed"             "wage employed"         
[4] "self employed"          "OTHER"                  "wage and self employed"

Training model...

Model trained successfully.


Call:
glm(formula = model_formula, family = binomial(link = "logit"), 
    data = train_data)

Coefficients: (2 not defined because of singularities)
                                         Estimate Std. Error z value Pr(>|z|)
(Intercept)                            -2.045e+00  2.094e-01  -9.766  < 2e-16
status_lag_cleanunemployed             -2.224e-01  1.204e-01  -1.847  0.06479
status_lag_cleanwage employed           9.583e-01  1.475e-01   6.499 8.10e-11
status_lag_cleanself employed           7.965e-01  1.377e-01   5.782 7.36e-09
status_lag_cleanOTHER                   8.404e-01  3.808e-01   2.207  0.02734
status_lag_cleanwage and self employed  1.268e+00  3.074e-01   4.125 3.71e-05
age_filled                              2.040e-02  3.972e-03   5.137 2.7

In [30]:
# ============================================================
# EXPERIMENT 8 — BLEND EXP 5 + BASELINE
# ============================================================
#
# Exp 5: Regularised GLM      = 0.64102
# Baseline: Original GLM     = 0.64035
#
# Blend:
#   75% Exp 5
#   25% Baseline
#
# Goal:
# Determine whether combining the two strongest models
# improves the Kaggle AUC.
# ============================================================


cat("============================================\n")
cat("EXPERIMENT 8 — MODEL BLEND\n")
cat("============================================\n\n")


# ------------------------------------------------------------
# 1. FILE NAMES
# ------------------------------------------------------------

exp5_file <- "submission_exp5_regularised.csv"

baseline_file <- "submission_baseline.csv"


# ------------------------------------------------------------
# 2. CHECK FILES EXIST
# ------------------------------------------------------------

if (!file.exists(exp5_file)) {
  stop(
    paste(
      "Could not find:",
      exp5_file,
      "\nMake sure the Exp 5 submission CSV is in",
      "the current working directory."
    )
  )
}

if (!file.exists(baseline_file)) {
  stop(
    paste(
      "Could not find:",
      baseline_file,
      "\nMake sure the baseline submission CSV is in",
      "the current working directory."
    )
  )
}


# ------------------------------------------------------------
# 3. LOAD PREDICTIONS
# ------------------------------------------------------------

exp5 <- read.csv(
  exp5_file,
  stringsAsFactors = FALSE
)

baseline <- read.csv(
  baseline_file,
  stringsAsFactors = FALSE
)


cat("Exp 5 rows     :", nrow(exp5), "\n")
cat("Baseline rows  :", nrow(baseline), "\n\n")


# ------------------------------------------------------------
# 4. CHECK COLUMN NAMES
# ------------------------------------------------------------

cat("Exp 5 columns:\n")
print(names(exp5))

cat("\nBaseline columns:\n")
print(names(baseline))

cat("\n")


# ------------------------------------------------------------
# 5. BASIC VALIDATION
# ------------------------------------------------------------

if (!"anonymised_id" %in% names(exp5)) {
  stop("Exp 5 submission does not contain anonymised_id.")
}

if (!"employed_status" %in% names(exp5)) {
  stop("Exp 5 submission does not contain employed_status.")
}

if (!"anonymised_id" %in% names(baseline)) {
  stop("Baseline submission does not contain anonymised_id.")
}

if (!"employed_status" %in% names(baseline)) {
  stop("Baseline submission does not contain employed_status.")
}


# ------------------------------------------------------------
# 6. MAKE SURE BOTH FILES CONTAIN THE SAME PARTICIPANTS
# ------------------------------------------------------------

if (nrow(exp5) != nrow(baseline)) {
  stop(
    "Exp 5 and baseline submissions have different numbers of rows."
  )
}


if (!all(exp5$anonymised_id == baseline$anonymised_id)) {

  cat(
    "IDs are not in the same order.\n"
  )

  cat(
    "Aligning baseline predictions using anonymised_id...\n\n"
  )

  baseline <- baseline[
    match(
      exp5$anonymised_id,
      baseline$anonymised_id
    ),
  ]

}


# ------------------------------------------------------------
# 7. CHECK FOR MISSING PREDICTIONS
# ------------------------------------------------------------

if (any(is.na(exp5$employed_status))) {
  stop("Exp 5 contains NA predictions.")
}

if (any(is.na(baseline$employed_status))) {
  stop("Baseline contains NA predictions.")
}


# ------------------------------------------------------------
# 8. CHECK PREDICTION RANGES
# ------------------------------------------------------------

if (any(
  exp5$employed_status < 0 |
  exp5$employed_status > 1
)) {
  stop("Exp 5 predictions are outside the 0-1 range.")
}

if (any(
  baseline$employed_status < 0 |
  baseline$employed_status > 1
)) {
  stop("Baseline predictions are outside the 0-1 range.")
}


# ------------------------------------------------------------
# 9. CREATE 75/25 BLEND
# ------------------------------------------------------------

exp5_weight <- 0.75
baseline_weight <- 0.25


blended_prediction <-
  (exp5_weight * exp5$employed_status) +
  (baseline_weight * baseline$employed_status)


# ------------------------------------------------------------
# 10. CREATE SUBMISSION
# ------------------------------------------------------------

submission_exp8 <- data.frame(
  anonymised_id = exp5$anonymised_id,

  employed_status = blended_prediction
)


# ------------------------------------------------------------
# 11. FINAL VALIDATION
# ------------------------------------------------------------

stopifnot(
  nrow(submission_exp8) == nrow(exp5)
)

stopifnot(
  all(!is.na(submission_exp8$employed_status))
)

stopifnot(
  all(
    submission_exp8$employed_status >= 0 &
    submission_exp8$employed_status <= 1
  )
)


# ------------------------------------------------------------
# 12. SAVE
# ------------------------------------------------------------

output_file <- "submission_exp8_blend_75_25.csv"

write.csv(
  submission_exp8,
  output_file,
  row.names = FALSE,
  quote = FALSE
)


# ------------------------------------------------------------
# 13. DISPLAY RESULTS
# ------------------------------------------------------------

cat("\n============================================\n")
cat("EXP 8 SUBMISSION CREATED\n")
cat("============================================\n\n")

cat("Exp 5 weight     :", exp5_weight, "\n")
cat("Baseline weight  :", baseline_weight, "\n")
cat("Rows             :", nrow(submission_exp8), "\n")
cat("Output file      :", output_file, "\n\n")


cat("Prediction summary:\n")

print(
  summary(
    submission_exp8$employed_status
  )
)


cat("\nFirst 10 predictions:\n")

print(
  head(
    submission_exp8,
    10
  )
)


cat("\n============================================\n")
cat("READY FOR KAGGLE\n")
cat("============================================\n")

EXPERIMENT 8 — MODEL BLEND

Exp 5 rows     : 3230 
Baseline rows  : 3230 

Exp 5 columns:
[1] "anonymised_id"   "employed_status"

Baseline columns:
[1] "anonymised_id"   "employed_status"


EXP 8 SUBMISSION CREATED

Exp 5 weight     : 0.75 
Baseline weight  : 0.25 
Rows             : 3230 
Output file      : submission_exp8_blend_75_25.csv 

Prediction summary:
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
 0.1634  0.2773  0.3052  0.3235  0.3540  0.6202 

First 10 predictions:
   anonymised_id employed_status
1          ID_10       0.2874687
2         ID_100       0.2840309
3        ID_1000       0.2909416
4        ID_1001       0.2909416
5        ID_1002       0.2909416
6        ID_1003       0.2909416
7        ID_1004       0.2909416
8        ID_1005       0.2909416
9        ID_1006       0.2909416
10       ID_1007       0.2909416

READY FOR KAGGLE


In [35]:
# ============================================================
# EXPERIMENT 9 — XGBOOST CLASSIFICATION
# NUMERIC + HISTORICAL FEATURES
# ============================================================

cat("============================================\n")
cat("EXPERIMENT 9 — XGBOOST CLASSIFICATION\n")
cat("============================================\n\n")


# ------------------------------------------------------------
# 1. LOAD PACKAGE
# ------------------------------------------------------------

if (!requireNamespace("xgboost", quietly = TRUE)) {
  install.packages(
    "xgboost",
    repos = "https://cloud.r-project.org"
  )
}

library(xgboost)


# ------------------------------------------------------------
# 2. LOAD DATA
# ------------------------------------------------------------

data_dir <- "../../assets/dataset"

train_data <- read.csv(
  file.path(data_dir, "train.csv"),
  stringsAsFactors = FALSE
)

test_data <- read.csv(
  file.path(data_dir, "test.csv"),
  stringsAsFactors = FALSE
)


cat("Training rows:", nrow(train_data), "\n")
cat("Testing rows :", nrow(test_data), "\n\n")


# ------------------------------------------------------------
# 3. REMOVE MISSING TARGET
# ------------------------------------------------------------

train_data <- train_data[
  !is.na(train_data$employed_status),
]


cat(
  "Training rows after removing missing target:",
  nrow(train_data),
  "\n\n"
)


# ------------------------------------------------------------
# 4. SAFE NQF EXTRACTION
# ------------------------------------------------------------

extract_nqf <- function(x) {

  x <- as.character(x)

  result <- rep(
    NA_real_,
    length(x)
  )


  for (i in seq_along(x)) {

    value <- x[i]

    if (is.na(value)) {
      next
    }


    # NQF Level

    match <- regexec(
      "NQF Level ([0-9]+)",
      value,
      ignore.case = TRUE
    )

    parts <- regmatches(
      value,
      match
    )


    if (length(parts[[1]]) >= 2) {

      result[i] <- suppressWarnings(
        as.numeric(parts[[1]][2])
      )

      next

    }


    # Grade fallback

    match_grade <- regexec(
      "Grade ([0-9]+)",
      value,
      ignore.case = TRUE
    )

    grade_parts <- regmatches(
      value,
      match_grade
    )


    if (length(grade_parts[[1]]) >= 2) {

      grade <- suppressWarnings(
        as.numeric(grade_parts[[1]][2])
      )

      if (!is.na(grade)) {

        result[i] <- max(
          grade - 8,
          1
        )

      }

    }

  }


  result

}


# ------------------------------------------------------------
# 5. FEATURE CREATION
# ------------------------------------------------------------

create_features <- function(data) {

  x <- data.frame(

    current_round =
      as.numeric(data$current_round),

    total_historical_rounds =
      as.numeric(data$total_historical_rounds),

    age =
      as.numeric(data$age),

    school_quintile =
      as.numeric(data$school_quintile),

    work_readiness_score =
      as.numeric(data$work_readiness_score),

    employed_lag =
      as.numeric(data$employed_lag),

    tenure_lag =
      as.numeric(data$tenure_lag),

    days_since_last_obs =
      as.numeric(data$days_since_last_obs),

    lag_round =
      as.numeric(data$lag_round)

  )


  # ----------------------------------------------------------
  # Missingness indicators
  # ----------------------------------------------------------

  x$has_previous_observation <-
    as.numeric(
      !is.na(data$lag_round)
    )


  x$has_work_readiness <-
    as.numeric(
      !is.na(data$work_readiness_score)
    )


  x$has_employed_lag <-
    as.numeric(
      !is.na(data$employed_lag)
    )


  x$has_tenure <-
    as.numeric(
      !is.na(data$tenure_lag)
    )


  # ----------------------------------------------------------
  # Historical employment
  # ----------------------------------------------------------

  x$employment_history <-
    ifelse(
      is.na(data$employed_lag),
      0,
      as.numeric(data$employed_lag)
    )


  # ----------------------------------------------------------
  # Observation recency
  # ----------------------------------------------------------

  x$recent_observation <-
    ifelse(
      is.na(data$days_since_last_obs),
      0,
      as.numeric(data$days_since_last_obs)
    )


  # ----------------------------------------------------------
  # Education
  # ----------------------------------------------------------

  x$education_nqf <-
    extract_nqf(
      data$education_level
    )


  # ----------------------------------------------------------
  # Make sure everything is numeric
  # ----------------------------------------------------------

  for (j in seq_along(x)) {

    x[[j]] <- as.numeric(x[[j]])

  }


  return(x)

}


# ------------------------------------------------------------
# 6. CREATE FEATURES
# ------------------------------------------------------------

x_train_df <- create_features(
  train_data
)

x_test_df <- create_features(
  test_data
)


cat(
  "Number of features:",
  ncol(x_train_df),
  "\n\n"
)


cat(
  "Train feature rows:",
  nrow(x_train_df),
  "\n"
)

cat(
  "Test feature rows :",
  nrow(x_test_df),
  "\n\n"
)


# ------------------------------------------------------------
# 7. VERIFY FEATURES
# ------------------------------------------------------------

if (
  nrow(x_train_df) != nrow(train_data)
) {

  stop(
    "Training feature row count does not match training data."
  )

}


if (
  nrow(x_test_df) != nrow(test_data)
) {

  stop(
    "Testing feature row count does not match testing data."
  )

}


if (
  !identical(
    names(x_train_df),
    names(x_test_df)
  )
) {

  stop(
    "Training and testing features do not match."
  )

}


# ------------------------------------------------------------
# 8. TARGET
# ------------------------------------------------------------

y_train <- factor(
  train_data$employed_status,
  levels = c(0, 1)
)


cat("Target distribution:\n")

print(
  table(y_train)
)

cat("\n")


# ------------------------------------------------------------
# 9. CONVERT FEATURES TO MATRICES
# ------------------------------------------------------------

x_train <- as.matrix(
  x_train_df
)

x_test <- as.matrix(
  x_test_df
)


storage.mode(x_train) <- "double"
storage.mode(x_test) <- "double"


cat(
  "Training matrix:",
  nrow(x_train),
  "x",
  ncol(x_train),
  "\n"
)

cat(
  "Testing matrix :",
  nrow(x_test),
  "x",
  ncol(x_test),
  "\n\n"
)


# ------------------------------------------------------------
# 10. TRAIN XGBOOST
# ------------------------------------------------------------

cat("============================================\n")
cat("TRAINING XGBOOST\n")
cat("============================================\n\n")


set.seed(42)


xgb_model <- xgboost(

  x = x_train,

  y = y_train,

  objective = "binary:logistic",

  eval_metric = "auc",

  nrounds = 300,

  max_depth = 4,

  learning_rate = 0.03,

  subsample = 0.8,

  colsample_bytree = 0.9,

  min_child_weight = 5,

  min_split_loss = 0,

  reg_lambda = 2,

  reg_alpha = 0

)


cat("\nModel trained successfully.\n\n")


# ------------------------------------------------------------
# 11. PREDICT
# ------------------------------------------------------------

cat("============================================\n")
cat("GENERATING ROUND 9 PREDICTIONS\n")
cat("============================================\n\n")


test_prob <- predict(
  xgb_model,
  x_test
)


# ------------------------------------------------------------
# 12. CHECK PREDICTIONS
# ------------------------------------------------------------

cat("Prediction summary:\n")

print(
  summary(test_prob)
)

cat("\n")


if (any(is.na(test_prob))) {

  stop(
    "XGBoost produced NA predictions."
  )

}


if (
  length(test_prob) != nrow(test_data)
) {

  stop(
    paste(
      "Prediction count:",
      length(test_prob),
      "Expected:",
      nrow(test_data)
    )
  )

}


# ------------------------------------------------------------
# 13. CREATE SUBMISSION
# ------------------------------------------------------------

submission_exp9 <- data.frame(

  anonymised_id =
    test_data$anonymised_id,

  employed_status =
    test_prob

)


# ------------------------------------------------------------
# 14. SAVE
# ------------------------------------------------------------

output_file <-
  "submission_exp9_xgboost_numeric.csv"


write.csv(
  submission_exp9,
  output_file,
  row.names = FALSE,
  quote = FALSE
)


# ------------------------------------------------------------
# 15. FINAL OUTPUT
# ------------------------------------------------------------

cat("============================================\n")
cat("SUBMISSION CREATED\n")
cat("============================================\n\n")

cat(
  "File:",
  output_file,
  "\n"
)

cat(
  "Rows:",
  nrow(submission_exp9),
  "\n\n"
)

cat("First 10 predictions:\n")

print(
  head(
    submission_exp9,
    10
  )
)

cat("\nPrediction summary:\n")

print(
  summary(
    submission_exp9$employed_status
  )
)

cat("\n============================================\n")
cat("READY FOR KAGGLE\n")
cat("============================================\n")

EXPERIMENT 9 — XGBOOST CLASSIFICATION

Training rows: 16719 
Testing rows : 3230 

Training rows after removing missing target: 16653 

Number of features: 16 

Train feature rows: 16653 
Test feature rows : 3230 

Target distribution:
y_train
    0     1 
11375  5278 

Training matrix: 16653 x 16 
Testing matrix : 3230 x 16 

TRAINING XGBOOST


Model trained successfully.

GENERATING ROUND 9 PREDICTIONS

Prediction summary:
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
0.07892 0.32542 0.37030 0.37293 0.37921 0.78761 

SUBMISSION CREATED

File: submission_exp9_xgboost_numeric.csv 
Rows: 3230 

First 10 predictions:
   anonymised_id employed_status
1          ID_10       0.3699555
2         ID_100       0.3702955
3        ID_1000       0.3699315
4        ID_1001       0.3699315
5        ID_1002       0.3699315
6        ID_1003       0.3699315
7        ID_1004       0.3699315
8        ID_1005       0.3699315
9        ID_1006       0.3699315
10       ID_1007       0.3699315

Prediction

In [36]:
# ============================================================
# EXPERIMENT 10 — REGULARISED LOGISTIC REGRESSION
# HISTORICAL + BEHAVIOURAL FEATURES
# FINAL EXPERIMENT
# ============================================================

cat("============================================\n")
cat("EXPERIMENT 10 — REGULARISED LOGISTIC MODEL\n")
cat("============================================\n\n")


# ------------------------------------------------------------
# 1. PACKAGE
# ------------------------------------------------------------

if (!requireNamespace("glmnet", quietly = TRUE)) {
  install.packages(
    "glmnet",
    repos = "https://cloud.r-project.org"
  )
}

library(glmnet)


# ------------------------------------------------------------
# 2. LOAD DATA
# ------------------------------------------------------------

data_dir <- "../../assets/dataset"

train_data <- read.csv(
  file.path(data_dir, "train.csv"),
  stringsAsFactors = FALSE
)

test_data <- read.csv(
  file.path(data_dir, "test.csv"),
  stringsAsFactors = FALSE
)

cat("Training rows:", nrow(train_data), "\n")
cat("Testing rows :", nrow(test_data), "\n\n")


# ------------------------------------------------------------
# 3. REMOVE MISSING TARGET
# ------------------------------------------------------------

train_data <- train_data[
  !is.na(train_data$employed_status),
]

cat(
  "Training rows after removing missing target:",
  nrow(train_data),
  "\n\n"
)


# ------------------------------------------------------------
# 4. SAFE NUMERIC CONVERSION
# ------------------------------------------------------------

safe_numeric <- function(x) {

  suppressWarnings(
    as.numeric(as.character(x))
  )

}


# ------------------------------------------------------------
# 5. EDUCATION / NQF EXTRACTION
# ------------------------------------------------------------

extract_nqf <- function(x) {

  x <- as.character(x)

  result <- rep(
    NA_real_,
    length(x)
  )

  for (i in seq_along(x)) {

    value <- x[i]

    if (is.na(value) || value == "") {
      next
    }

    # Try NQF Level

    match_nqf <- regexec(
      "NQF Level ([0-9]+)",
      value,
      ignore.case = TRUE
    )

    parts <- regmatches(
      value,
      match_nqf
    )

    if (
      length(parts[[1]]) >= 2 &&
      !is.na(parts[[1]][2])
    ) {

      result[i] <- suppressWarnings(
        as.numeric(parts[[1]][2])
      )

      next
    }


    # Try Grade

    match_grade <- regexec(
      "Grade ([0-9]+)",
      value,
      ignore.case = TRUE
    )

    grade_parts <- regmatches(
      value,
      match_grade
    )

    if (
      length(grade_parts[[1]]) >= 2
    ) {

      grade <- suppressWarnings(
        as.numeric(grade_parts[[1]][2])
      )

      if (!is.na(grade)) {

        result[i] <- max(
          grade - 8,
          1
        )

      }

    }

  }

  result

}


# ------------------------------------------------------------
# 6. FEATURE ENGINEERING
# ------------------------------------------------------------

create_features <- function(data) {

  x <- data.frame(

    # ----------------------------------------
    # Core demographics
    # ----------------------------------------

    age =
      safe_numeric(data$age),

    school_quintile =
      safe_numeric(data$school_quintile),

    # ----------------------------------------
    # Work readiness
    # ----------------------------------------

    work_readiness_score =
      safe_numeric(data$work_readiness_score),

    # ----------------------------------------
    # Historical information
    # ----------------------------------------

    total_historical_rounds =
      safe_numeric(data$total_historical_rounds),

    lag_round =
      safe_numeric(data$lag_round),

    days_since_last_obs =
      safe_numeric(data$days_since_last_obs),

    tenure_lag =
      safe_numeric(data$tenure_lag),

    employed_lag =
      safe_numeric(data$employed_lag),

    # ----------------------------------------
    # Education
    # ----------------------------------------

    education_nqf =
      extract_nqf(data$education_level)

  )


  # ----------------------------------------------------------
  # Missingness indicators
  # ----------------------------------------------------------

  x$has_previous_observation <-
    as.numeric(
      !is.na(data$lag_round)
    )


  x$has_employed_history <-
    as.numeric(
      !is.na(data$employed_lag)
    )


  x$has_tenure_history <-
    as.numeric(
      !is.na(data$tenure_lag)
    )


  x$has_work_readiness <-
    as.numeric(
      !is.na(data$work_readiness_score)
    )


  x$has_days_since_observation <-
    as.numeric(
      !is.na(data$days_since_last_obs)
    )


  # ----------------------------------------------------------
  # Historical employment interaction
  # ----------------------------------------------------------

  x$employment_history_strength <-
    ifelse(
      is.na(data$employed_lag),
      0,
      safe_numeric(data$employed_lag) *
        pmin(
          safe_numeric(data$total_historical_rounds),
          5
        )
    )


  # ----------------------------------------------------------
  # Recent observation indicator
  # ----------------------------------------------------------

  days <- safe_numeric(
    data$days_since_last_obs
  )

  x$recent_observation <-
    ifelse(
      is.na(days),
      0,
      as.numeric(days <= 180)
    )


  # ----------------------------------------------------------
  # Long-term participant indicator
  # ----------------------------------------------------------

  rounds <- safe_numeric(
    data$total_historical_rounds
  )

  x$long_history <-
    ifelse(
      is.na(rounds),
      0,
      as.numeric(rounds >= 3)
    )


  # ----------------------------------------------------------
  # Age transformations
  # ----------------------------------------------------------

  age <- safe_numeric(
    data$age
  )

  x$age_squared <-
    ifelse(
      is.na(age),
      NA,
      age^2
    )


  # ----------------------------------------------------------
  # Work readiness transformation
  # ----------------------------------------------------------

  wr <- safe_numeric(
    data$work_readiness_score
  )

  x$work_readiness_squared <-
    ifelse(
      is.na(wr),
      NA,
      wr^2
    )


  # ----------------------------------------------------------
  # Replace impossible values with NA
  # ----------------------------------------------------------

  x$age[
    x$age < 15 |
      x$age > 80
  ] <- NA


  x$school_quintile[
    x$school_quintile < 0 |
      x$school_quintile > 5
  ] <- NA


  x$work_readiness_score[
    x$work_readiness_score < 0 |
      x$work_readiness_score > 1
  ] <- NA


  # ----------------------------------------------------------
  # Return
  # ----------------------------------------------------------

  x

}


# ------------------------------------------------------------
# 7. CREATE FEATURES
# ------------------------------------------------------------

x_train_df <- create_features(
  train_data
)

x_test_df <- create_features(
  test_data
)


cat(
  "Number of engineered features:",
  ncol(x_train_df),
  "\n\n"
)


# ------------------------------------------------------------
# 8. MAKE TRAIN/TEST COLUMN STRUCTURE IDENTICAL
# ------------------------------------------------------------

if (
  !identical(
    names(x_train_df),
    names(x_test_df)
  )
) {

  stop(
    "Training and testing feature names do not match."
  )

}


# ------------------------------------------------------------
# 9. IMPUTE MISSING NUMERIC VALUES
# ------------------------------------------------------------

# IMPORTANT:
# Calculate medians ONLY from training data.

for (column in names(x_train_df)) {

  median_value <- median(
    x_train_df[[column]],
    na.rm = TRUE
  )

  if (
    is.na(median_value) ||
    is.infinite(median_value)
  ) {

    median_value <- 0

  }


  x_train_df[[column]][
    is.na(x_train_df[[column]])
  ] <- median_value


  x_test_df[[column]][
    is.na(x_test_df[[column]])
  ] <- median_value

}


# ------------------------------------------------------------
# 10. SCALE FEATURES
# ------------------------------------------------------------

# Scaling is important for regularisation.

train_means <- sapply(
  x_train_df,
  mean
)

train_sds <- sapply(
  x_train_df,
  sd
)


train_sds[
  is.na(train_sds) |
    train_sds == 0
] <- 1


x_train_scaled <- sweep(
  as.matrix(x_train_df),
  2,
  train_means,
  "-"
)

x_train_scaled <- sweep(
  x_train_scaled,
  2,
  train_sds,
  "/"
)


x_test_scaled <- sweep(
  as.matrix(x_test_df),
  2,
  train_means,
  "-"
)

x_test_scaled <- sweep(
  x_test_scaled,
  2,
  train_sds,
  "/"
)


# ------------------------------------------------------------
# 11. TARGET
# ------------------------------------------------------------

y_train <- as.numeric(
  train_data$employed_status
)


cat("Target distribution:\n")

print(
  table(y_train)
)

cat("\n")


# ------------------------------------------------------------
# 12. CROSS-VALIDATED REGULARISATION
# ------------------------------------------------------------

cat("============================================\n")
cat("CROSS-VALIDATING REGULARISATION\n")
cat("============================================\n\n")


set.seed(42)


# Elastic Net:
#
# alpha = 0   -> Ridge
# alpha = 1   -> Lasso
#
# 0.25 gives us mostly Ridge with some Lasso
# behaviour.

cv_model <- cv.glmnet(

  x_train_scaled,

  y_train,

  family = "binomial",

  alpha = 0.25,

  nfolds = 5,

  type.measure = "auc",

  standardize = FALSE,

  parallel = FALSE

)


cat(
  "\nBest lambda:",
  cv_model$lambda.min,
  "\n"
)

cat(
  "1-SE lambda:",
  cv_model$lambda.1se,
  "\n\n"
)


# ------------------------------------------------------------
# 13. TRAIN FINAL MODEL
# ------------------------------------------------------------

cat("============================================\n")
cat("TRAINING FINAL REGULARISED MODEL\n")
cat("============================================\n\n")


final_model <- glmnet(

  x_train_scaled,

  y_train,

  family = "binomial",

  alpha = 0.25,

  lambda = cv_model$lambda.min,

  standardize = FALSE

)


cat("Final model trained successfully.\n\n")


# ------------------------------------------------------------
# 14. DISPLAY IMPORTANT FEATURES
# ------------------------------------------------------------

cat("Model coefficients:\n\n")

coef_table <- as.matrix(
  coef(final_model)
)

coef_table <- data.frame(
  feature = rownames(coef_table),
  coefficient = as.numeric(
    coef_table[, 1]
  ),
  row.names = NULL
)

coef_table <- coef_table[
  order(
    abs(coef_table$coefficient),
    decreasing = TRUE
  ),
]

print(
  head(
    coef_table,
    20
  )
)

cat("\n")


# ------------------------------------------------------------
# 15. GENERATE PREDICTIONS
# ------------------------------------------------------------

cat("============================================\n")
cat("GENERATING ROUND 9 PREDICTIONS\n")
cat("============================================\n\n")


test_prob <- as.numeric(
  predict(
    final_model,
    newx = x_test_scaled,
    type = "response"
  )
)


# ------------------------------------------------------------
# 16. CHECK PREDICTIONS
# ------------------------------------------------------------

cat("Prediction summary:\n")

print(
  summary(test_prob)
)

cat("\n")


if (
  any(is.na(test_prob))
) {

  stop(
    "Model produced NA predictions."
  )

}


if (
  length(test_prob) != nrow(test_data)
) {

  stop(
    paste(
      "Wrong number of predictions.",
      "Expected:",
      nrow(test_data),
      "Got:",
      length(test_prob)
    )
  )

}


# ------------------------------------------------------------
# 17. CREATE SUBMISSION
# ------------------------------------------------------------

submission_exp10 <- data.frame(

  anonymised_id =
    test_data$anonymised_id,

  employed_status =
    test_prob

)


# ------------------------------------------------------------
# 18. SAVE SUBMISSION
# ------------------------------------------------------------

output_file <-
  "submission_exp10_elastic_net.csv"


write.csv(
  submission_exp10,
  output_file,
  row.names = FALSE,
  quote = FALSE
)


# ------------------------------------------------------------
# 19. FINAL CHECK
# ------------------------------------------------------------

cat("============================================\n")
cat("EXPERIMENT 10 COMPLETE\n")
cat("============================================\n\n")

cat(
  "Saved:",
  output_file,
  "\n"
)

cat(
  "Rows:",
  nrow(submission_exp10),
  "\n\n"
)

cat("First 10 predictions:\n")

print(
  head(
    submission_exp10,
    10
  )
)

cat("\n")

cat("Prediction summary:\n")

print(
  summary(
    submission_exp10$employed_status
  )
)

cat("\n============================================\n")
cat("READY FOR KAGGLE SUBMISSION\n")
cat("============================================\n")

EXPERIMENT 10 — REGULARISED LOGISTIC MODEL

Training rows: 16719 
Testing rows : 3230 

Training rows after removing missing target: 16653 

Number of engineered features: 19 

Target distribution:
y_train
    0     1 
11375  5278 

CROSS-VALIDATING REGULARISATION


Best lambda: 0.0004645243 
1-SE lambda: 0.05340901 

TRAINING FINAL REGULARISED MODEL

Final model trained successfully.

Model coefficients:

                       feature  coefficient
1                  (Intercept) -0.787851232
4         work_readiness_score  0.247129642
2                          age  0.227525342
16 employment_history_strength  0.215978586
19                 age_squared -0.139482079
9                 employed_lag  0.116407822
20      work_readiness_squared -0.114735082
3              school_quintile  0.075029829
5      total_historical_rounds -0.067188254
13          has_tenure_history  0.045924086
6                    lag_round  0.045823489
8                   tenure_lag -0.037920763
17          recent

In [1]:
# ============================================================
# EXPERIMENT 11 — CONSERVATIVE ENSEMBLE
# Blend the strongest previous submissions
# ============================================================

cat("============================================\n")
cat("EXPERIMENT 11 — CONSERVATIVE ENSEMBLE\n")
cat("============================================\n\n")


# ------------------------------------------------------------
# 1. LOAD SUBMISSIONS
# ------------------------------------------------------------

files <- c(
  "submission_exp5_regularised.csv",
  "submission_exp2_historical_status_clean.csv",
  "submission_baseline.csv",
  "submission_exp8_blend_75_25.csv"
)

weights <- c(
  0.45,   # Exp 5 — strongest
  0.20,   # Exp 2
  0.20,   # Baseline
  0.15    # Exp 8
)


# ------------------------------------------------------------
# 2. CHECK FILES
# ------------------------------------------------------------

cat("Checking submission files...\n\n")

for (f in files) {

  if (!file.exists(f)) {

    stop(
      paste(
        "Could not find:",
        f,
        "\nMake sure the CSV files are in the current working directory."
      )
    )

  }

  cat("Found:", f, "\n")
}


# ------------------------------------------------------------
# 3. LOAD SUBMISSIONS
# ------------------------------------------------------------

submissions <- lapply(
  files,
  function(f) {
    read.csv(
      f,
      stringsAsFactors = FALSE
    )
  }
)


# ------------------------------------------------------------
# 4. CHECK STRUCTURE
# ------------------------------------------------------------

cat("\nChecking submission structure...\n\n")

for (i in seq_along(submissions)) {

  s <- submissions[[i]]

  cat(
    basename(files[i]),
    "->",
    nrow(s),
    "rows\n"
  )

  if (!all(
    c("anonymised_id", "employed_status")
    %in%
    names(s)
  )) {

    stop(
      paste(
        "Invalid columns in:",
        files[i]
      )
    )

  }

}


# ------------------------------------------------------------
# 5. CHECK IDS
# ------------------------------------------------------------

reference_ids <-
  submissions[[1]]$anonymised_id


for (i in seq_along(submissions)) {

  if (
    !identical(
      submissions[[i]]$anonymised_id,
      reference_ids
    )
  ) {

    stop(
      paste(
        "ID ordering does not match in:",
        files[i]
      )
    )

  }

}


# ------------------------------------------------------------
# 6. EXTRACT PREDICTIONS
# ------------------------------------------------------------

predictions <- lapply(
  submissions,
  function(s) {

    as.numeric(
      s$employed_status
    )

  }
)


# ------------------------------------------------------------
# 7. CHECK PREDICTIONS
# ------------------------------------------------------------

cat("\nChecking prediction ranges...\n\n")

for (i in seq_along(predictions)) {

  p <- predictions[[i]]

  cat(
    basename(files[i]),
    "\n"
  )

  cat(
    "  Min:",
    min(p, na.rm = TRUE),
    "\n"
  )

  cat(
    "  Max:",
    max(p, na.rm = TRUE),
    "\n"
  )

  cat(
    "  NA:",
    sum(is.na(p)),
    "\n\n"
  )


  if (any(is.na(p))) {

    stop(
      paste(
        "NA predictions found in:",
        files[i]
      )
    )

  }

}


# ------------------------------------------------------------
# 8. NORMALISE WEIGHTS
# ------------------------------------------------------------

weights <-
  weights / sum(weights)


cat("Weights:\n\n")

for (i in seq_along(files)) {

  cat(
    basename(files[i]),
    ":",
    weights[i],
    "\n"
  )

}


# ------------------------------------------------------------
# 9. CREATE ENSEMBLE
# ------------------------------------------------------------

cat("\nGenerating ensemble predictions...\n\n")


ensemble_prediction <-
  rep(
    0,
    length(predictions[[1]])
  )


for (i in seq_along(predictions)) {

  ensemble_prediction <-
    ensemble_prediction +
    weights[i] *
    predictions[[i]]

}


# ------------------------------------------------------------
# 10. CLIP TO VALID PROBABILITY RANGE
# ------------------------------------------------------------

ensemble_prediction <-
  pmin(
    pmax(
      ensemble_prediction,
      0
    ),
    1
  )


# ------------------------------------------------------------
# 11. CREATE SUBMISSION
# ------------------------------------------------------------

submission_exp11 <-
  data.frame(

    anonymised_id =
      reference_ids,

    employed_status =
      ensemble_prediction

  )


# ------------------------------------------------------------
# 12. CHECK OUTPUT
# ------------------------------------------------------------

cat("============================================\n")
cat("ENSEMBLE SUMMARY\n")
cat("============================================\n\n")

print(
  summary(
    submission_exp11$employed_status
  )
)

cat("\nRows:", nrow(submission_exp11), "\n")


# ------------------------------------------------------------
# 13. SAVE
# ------------------------------------------------------------

output_file <-
  "submission_exp11_ensemble.csv"


write.csv(
  submission_exp11,
  output_file,
  row.names = FALSE,
  quote = FALSE
)


# ------------------------------------------------------------
# 14. DISPLAY
# ------------------------------------------------------------

cat("\n============================================\n")
cat("EXPERIMENT 11 COMPLETE\n")
cat("============================================\n\n")

cat(
  "Saved:",
  output_file,
  "\n\n"
)

cat("First 10 predictions:\n\n")

print(
  head(
    submission_exp11,
    10
  )
)

cat("\n============================================\n")
cat("READY FOR KAGGLE SUBMISSION\n")
cat("============================================\n")

EXPERIMENT 11 — CONSERVATIVE ENSEMBLE

Checking submission files...

Found: submission_exp5_regularised.csv 
Found: submission_exp2_historical_status_clean.csv 
Found: submission_baseline.csv 
Found: submission_exp8_blend_75_25.csv 

Checking submission structure...

submission_exp5_regularised.csv -> 3230 rows
submission_exp2_historical_status_clean.csv -> 3230 rows
submission_baseline.csv -> 3230 rows
submission_exp8_blend_75_25.csv -> 3230 rows

Checking prediction ranges...

submission_exp5_regularised.csv 
  Min: 0.1347562 
  Max: 0.7212198 
  NA: 0 

submission_exp2_historical_status_clean.csv 
  Min: 0.07002563 
  Max: 0.9419643 
  NA: 0 

submission_baseline.csv 
  Min: 0.2485566 
  Max: 0.4285285 
  NA: 0 

submission_exp8_blend_75_25.csv 
  Min: 0.1633674 
  Max: 0.6202133 
  NA: 0 

Weights:

submission_exp5_regularised.csv : 0.45 
submission_exp2_historical_status_clean.csv : 0.2 
submission_baseline.csv : 0.2 
submission_exp8_blend_75_25.csv : 0.15 

Generating ensemble pr

In [2]:
# ============================================================
# EXPERIMENT 12 — HISTORICAL × BEHAVIOURAL INTERACTIONS
# REGULARISED LOGISTIC REGRESSION
# ============================================================

cat("============================================\n")
cat("EXPERIMENT 12 — HISTORY INTERACTIONS\n")
cat("============================================\n\n")


# ------------------------------------------------------------
# 1. PACKAGE
# ------------------------------------------------------------

if (!requireNamespace("glmnet", quietly = TRUE)) {
  install.packages(
    "glmnet",
    repos = "https://cloud.r-project.org"
  )
}

library(glmnet)


# ------------------------------------------------------------
# 2. LOAD DATA
# ------------------------------------------------------------

data_dir <- "../../assets/dataset"

train_data <- read.csv(
  file.path(data_dir, "train.csv"),
  stringsAsFactors = FALSE
)

test_data <- read.csv(
  file.path(data_dir, "test.csv"),
  stringsAsFactors = FALSE
)

cat("Training rows:", nrow(train_data), "\n")
cat("Testing rows :", nrow(test_data), "\n\n")


# ------------------------------------------------------------
# 3. REMOVE MISSING TARGET
# ------------------------------------------------------------

train_data <- train_data[
  !is.na(train_data$employed_status),
]

cat(
  "Training rows after removing missing target:",
  nrow(train_data),
  "\n\n"
)


# ------------------------------------------------------------
# 4. SAFE NUMERIC
# ------------------------------------------------------------

safe_numeric <- function(x) {
  suppressWarnings(
    as.numeric(as.character(x))
  )
}


# ------------------------------------------------------------
# 5. NQF EXTRACTION
# ------------------------------------------------------------

extract_nqf <- function(x) {

  x <- as.character(x)

  result <- rep(
    NA_real_,
    length(x)
  )

  for (i in seq_along(x)) {

    value <- x[i]

    if (is.na(value) || value == "") {
      next
    }

    match_nqf <- regexec(
      "NQF Level ([0-9]+)",
      value,
      ignore.case = TRUE
    )

    parts <- regmatches(
      value,
      match_nqf
    )

    if (
      length(parts[[1]]) >= 2 &&
      !is.na(parts[[1]][2])
    ) {

      result[i] <-
        suppressWarnings(
          as.numeric(parts[[1]][2])
        )

      next
    }


    match_grade <- regexec(
      "Grade ([0-9]+)",
      value,
      ignore.case = TRUE
    )

    grade_parts <- regmatches(
      value,
      match_grade
    )

    if (
      length(grade_parts[[1]]) >= 2
    ) {

      grade <-
        suppressWarnings(
          as.numeric(grade_parts[[1]][2])
        )

      if (!is.na(grade)) {

        result[i] <-
          max(
            grade - 8,
            1
          )

      }

    }

  }

  result
}


# ------------------------------------------------------------
# 6. FEATURE ENGINEERING
# ------------------------------------------------------------

create_features <- function(data) {

  age <- safe_numeric(data$age)

  wr <- safe_numeric(
    data$work_readiness_score
  )

  employed <- safe_numeric(
    data$employed_lag
  )

  rounds <- safe_numeric(
    data$total_historical_rounds
  )

  days <- safe_numeric(
    data$days_since_last_obs
  )

  tenure <- safe_numeric(
    data$tenure_lag
  )

  lag_round <- safe_numeric(
    data$lag_round
  )

  school <- safe_numeric(
    data$school_quintile
  )


  x <- data.frame(

    # --------------------------------------------------------
    # Base features
    # --------------------------------------------------------

    age = age,

    school_quintile = school,

    work_readiness_score = wr,

    total_historical_rounds = rounds,

    lag_round = lag_round,

    days_since_last_obs = days,

    tenure_lag = tenure,

    employed_lag = employed,

    education_nqf =
      extract_nqf(
        data$education_level
      )

  )


  # ----------------------------------------------------------
  # Missingness indicators
  # ----------------------------------------------------------

  x$has_previous_observation <-
    as.numeric(
      !is.na(lag_round)
    )

  x$has_employed_history <-
    as.numeric(
      !is.na(employed)
    )

  x$has_work_readiness <-
    as.numeric(
      !is.na(wr)
    )

  x$has_tenure_history <-
    as.numeric(
      !is.na(tenure)
    )

  x$has_days_history <-
    as.numeric(
      !is.na(days)
    )


  # ----------------------------------------------------------
  # Historical strength
  # ----------------------------------------------------------

  x$employment_history_strength <-
    ifelse(
      is.na(employed) |
        is.na(rounds),
      0,
      employed *
        pmin(rounds, 5)
    )


  # ----------------------------------------------------------
  # Recency
  # ----------------------------------------------------------

  x$recent_observation <-
    ifelse(
      is.na(days),
      0,
      as.numeric(days <= 180)
    )

  x$very_recent_observation <-
    ifelse(
      is.na(days),
      0,
      as.numeric(days <= 90)
    )


  # ----------------------------------------------------------
  # Long history
  # ----------------------------------------------------------

  x$long_history <-
    ifelse(
      is.na(rounds),
      0,
      as.numeric(rounds >= 3)
    )

  x$very_long_history <-
    ifelse(
      is.na(rounds),
      0,
      as.numeric(rounds >= 5)
    )


  # ==========================================================
  # INTERACTION FEATURES
  # ==========================================================

  # Previous employment × work readiness

  x$employed_x_work_readiness <-
    ifelse(
      is.na(employed) |
        is.na(wr),
      0,
      employed * wr
    )


  # Previous employment × historical rounds

  x$employed_x_rounds <-
    ifelse(
      is.na(employed) |
        is.na(rounds),
      0,
      employed * pmin(rounds, 8)
    )


  # Previous employment × recency

  x$employed_x_recent <-
    ifelse(
      is.na(employed) |
        is.na(days),
      0,
      employed *
        as.numeric(days <= 180)
    )


  # Previous employment × very recent

  x$employed_x_very_recent <-
    ifelse(
      is.na(employed) |
        is.na(days),
      0,
      employed *
        as.numeric(days <= 90)
    )


  # Work readiness × participant history

  x$work_readiness_x_rounds <-
    ifelse(
      is.na(wr) |
        is.na(rounds),
      0,
      wr * pmin(rounds, 8)
    )


  # Work readiness × recency

  x$work_readiness_x_recent <-
    ifelse(
      is.na(wr) |
        is.na(days),
      0,
      wr *
        as.numeric(days <= 180)
    )


  # Tenure × previous employment

  x$tenure_x_employed <-
    ifelse(
      is.na(tenure) |
        is.na(employed),
      0,
      tenure * employed
    )


  # Tenure in years

  x$tenure_years <-
    ifelse(
      is.na(tenure),
      0,
      pmin(tenure, 3650) / 365
    )


  # ----------------------------------------------------------
  # Age nonlinear terms
  # ----------------------------------------------------------

  x$age_squared <-
    ifelse(
      is.na(age),
      NA,
      age^2
    )


  # ----------------------------------------------------------
  # Clean impossible values
  # ----------------------------------------------------------

  x$age[
    x$age < 15 |
      x$age > 80
  ] <- NA


  x$school_quintile[
    x$school_quintile < 0 |
      x$school_quintile > 5
  ] <- NA


  x$work_readiness_score[
    x$work_readiness_score < 0 |
      x$work_readiness_score > 1
  ] <- NA


  x
}


# ------------------------------------------------------------
# 7. CREATE FEATURES
# ------------------------------------------------------------

x_train_df <-
  create_features(train_data)

x_test_df <-
  create_features(test_data)


cat(
  "Number of features:",
  ncol(x_train_df),
  "\n\n"
)


# ------------------------------------------------------------
# 8. VERIFY STRUCTURE
# ------------------------------------------------------------

if (
  !identical(
    names(x_train_df),
    names(x_test_df)
  )
) {

  stop(
    "Training and testing feature names do not match."
  )

}


# ------------------------------------------------------------
# 9. IMPUTE USING TRAINING MEDIANS
# ------------------------------------------------------------

for (column in names(x_train_df)) {

  median_value <-
    median(
      x_train_df[[column]],
      na.rm = TRUE
    )

  if (
    is.na(median_value) ||
      is.infinite(median_value)
  ) {

    median_value <- 0

  }


  x_train_df[[column]][
    is.na(x_train_df[[column]])
  ] <- median_value


  x_test_df[[column]][
    is.na(x_test_df[[column]])
  ] <- median_value

}


# ------------------------------------------------------------
# 10. SCALE
# ------------------------------------------------------------

train_means <-
  sapply(
    x_train_df,
    mean
  )

train_sds <-
  sapply(
    x_train_df,
    sd
  )


train_sds[
  is.na(train_sds) |
    train_sds == 0
] <- 1


x_train_scaled <-
  sweep(
    as.matrix(x_train_df),
    2,
    train_means,
    "-"
  )

x_train_scaled <-
  sweep(
    x_train_scaled,
    2,
    train_sds,
    "/"
  )


x_test_scaled <-
  sweep(
    as.matrix(x_test_df),
    2,
    train_means,
    "-"
  )

x_test_scaled <-
  sweep(
    x_test_scaled,
    2,
    train_sds,
    "/"
  )


# ------------------------------------------------------------
# 11. TARGET
# ------------------------------------------------------------

y_train <-
  as.numeric(
    train_data$employed_status
  )


cat("Target distribution:\n")

print(
  table(y_train)
)

cat("\n")


# ------------------------------------------------------------
# 12. CROSS VALIDATION
# ------------------------------------------------------------

cat("============================================\n")
cat("CROSS-VALIDATING ELASTIC NET\n")
cat("============================================\n\n")


set.seed(42)


cv_model <-
  cv.glmnet(

    x_train_scaled,

    y_train,

    family = "binomial",

    alpha = 0.20,

    nfolds = 5,

    type.measure = "auc",

    standardize = FALSE

  )


cat(
  "Best lambda:",
  cv_model$lambda.min,
  "\n"
)

cat(
  "1-SE lambda:",
  cv_model$lambda.1se,
  "\n\n"
)


# ------------------------------------------------------------
# 13. TRAIN FINAL MODEL
# ------------------------------------------------------------

final_model <-
  glmnet(

    x_train_scaled,

    y_train,

    family = "binomial",

    alpha = 0.20,

    lambda = cv_model$lambda.min,

    standardize = FALSE

  )


cat(
  "Final model trained successfully.\n\n"
)


# ------------------------------------------------------------
# 14. COEFFICIENTS
# ------------------------------------------------------------

cat("Top model features:\n\n")


coef_matrix <-
  as.matrix(
    coef(final_model)
  )


coef_table <-
  data.frame(

    feature =
      rownames(coef_matrix),

    coefficient =
      as.numeric(
        coef_matrix[, 1]
      ),

    row.names = NULL

  )


coef_table <-
  coef_table[
    order(
      abs(
        coef_table$coefficient
      ),
      decreasing = TRUE
    ),
  ]


print(
  head(
    coef_table,
    20
  )
)

cat("\n")


# ------------------------------------------------------------
# 15. PREDICT
# ------------------------------------------------------------

cat("============================================\n")
cat("GENERATING ROUND 9 PREDICTIONS\n")
cat("============================================\n\n")


test_prob <-
  as.numeric(
    predict(
      final_model,
      newx = x_test_scaled,
      type = "response"
    )
  )


# ------------------------------------------------------------
# 16. CHECK
# ------------------------------------------------------------

if (
  any(is.na(test_prob))
) {

  stop(
    "Model produced NA predictions."
  )

}


if (
  length(test_prob) !=
    nrow(test_data)
) {

  stop(
    paste(
      "Wrong number of predictions.",
      "Expected:",
      nrow(test_data),
      "Got:",
      length(test_prob)
    )
  )

}


cat("Prediction summary:\n\n")

print(
  summary(test_prob)
)

cat("\n")


# ------------------------------------------------------------
# 17. SUBMISSION
# ------------------------------------------------------------

submission_exp12 <-
  data.frame(

    anonymised_id =
      test_data$anonymised_id,

    employed_status =
      test_prob

  )


# ------------------------------------------------------------
# 18. SAVE
# ------------------------------------------------------------

output_file <-
  "submission_exp12_history_interactions.csv"


write.csv(
  submission_exp12,
  output_file,
  row.names = FALSE,
  quote = FALSE
)


# ------------------------------------------------------------
# 19. FINAL OUTPUT
# ------------------------------------------------------------

cat("============================================\n")
cat("EXPERIMENT 12 COMPLETE\n")
cat("============================================\n\n")

cat(
  "Saved:",
  output_file,
  "\n"
)

cat(
  "Rows:",
  nrow(submission_exp12),
  "\n\n"
)

cat("First 10 predictions:\n\n")

print(
  head(
    submission_exp12,
    10
  )
)

cat("\n============================================\n")
cat("READY FOR KAGGLE SUBMISSION\n")
cat("============================================\n")

EXPERIMENT 12 — HISTORY INTERACTIONS



Warning message:
"package 'glmnet' was built under R version 4.4.3"
Loading required package: Matrix

Loaded glmnet 4.1-10



Training rows: 16719 
Testing rows : 3230 

Training rows after removing missing target: 16653 

Number of features: 28 

Target distribution:
y_train
    0     1 
11375  5278 

CROSS-VALIDATING ELASTIC NET

Best lambda: 0.003098783 
1-SE lambda: 0.03820325 

Final model trained successfully.

Top model features:

                       feature  coefficient
1                  (Intercept) -0.787674459
4         work_readiness_score  0.143015459
16 employment_history_strength  0.106180107
22           employed_x_rounds  0.097894201
2                          age  0.084775031
23           employed_x_recent  0.083513856
9                 employed_lag  0.079540709
3              school_quintile  0.070381545
17          recent_observation -0.067041283
8                   tenure_lag -0.063210673
27           tenure_x_employed  0.057442031
6                    lag_round  0.042297464
14          has_tenure_history  0.032123057
21   employed_x_work_readiness -0.030722012
19                long_h

In [3]:
# ============================================================
# EXPERIMENT 13 — RANDOM FOREST
# CONTROLLED NUMERIC + HISTORICAL FEATURES
# ============================================================

cat("============================================\n")
cat("EXPERIMENT 13 — RANDOM FOREST\n")
cat("============================================\n\n")


# ------------------------------------------------------------
# 1. PACKAGE
# ------------------------------------------------------------

if (!requireNamespace("ranger", quietly = TRUE)) {
  install.packages(
    "ranger",
    repos = "https://cloud.r-project.org"
  )
}

library(ranger)


# ------------------------------------------------------------
# 2. LOAD DATA
# ------------------------------------------------------------

data_dir <- "../../assets/dataset"

train_data <- read.csv(
  file.path(data_dir, "train.csv"),
  stringsAsFactors = FALSE
)

test_data <- read.csv(
  file.path(data_dir, "test.csv"),
  stringsAsFactors = FALSE
)

cat("Training rows:", nrow(train_data), "\n")
cat("Testing rows :", nrow(test_data), "\n\n")


# ------------------------------------------------------------
# 3. REMOVE MISSING TARGET
# ------------------------------------------------------------

train_data <- train_data[
  !is.na(train_data$employed_status),
]

cat(
  "Training rows after removing missing target:",
  nrow(train_data),
  "\n\n"
)


# ------------------------------------------------------------
# 4. SAFE NUMERIC CONVERSION
# ------------------------------------------------------------

safe_numeric <- function(x) {
  suppressWarnings(
    as.numeric(as.character(x))
  )
}


# ------------------------------------------------------------
# 5. NQF EXTRACTION
# ------------------------------------------------------------

extract_nqf <- function(x) {

  x <- as.character(x)

  result <- rep(
    NA_real_,
    length(x)
  )

  for (i in seq_along(x)) {

    value <- x[i]

    if (is.na(value) || value == "") {
      next
    }

    # NQF Level

    match_nqf <- regexec(
      "NQF Level ([0-9]+)",
      value,
      ignore.case = TRUE
    )

    parts <- regmatches(
      value,
      match_nqf
    )

    if (
      length(parts[[1]]) >= 2 &&
      !is.na(parts[[1]][2])
    ) {

      result[i] <- suppressWarnings(
        as.numeric(parts[[1]][2])
      )

      next
    }


    # Grade

    match_grade <- regexec(
      "Grade ([0-9]+)",
      value,
      ignore.case = TRUE
    )

    grade_parts <- regmatches(
      value,
      match_grade
    )

    if (
      length(grade_parts[[1]]) >= 2
    ) {

      grade <- suppressWarnings(
        as.numeric(grade_parts[[1]][2])
      )

      if (!is.na(grade)) {

        result[i] <- max(
          grade - 8,
          1
        )

      }
    }
  }

  result
}


# ------------------------------------------------------------
# 6. FEATURE ENGINEERING
# ------------------------------------------------------------

create_features <- function(data) {

  x <- data.frame(

    # Demographics
    age =
      safe_numeric(data$age),

    school_quintile =
      safe_numeric(data$school_quintile),

    # Work readiness
    work_readiness_score =
      safe_numeric(data$work_readiness_score),

    # Historical information
    total_historical_rounds =
      safe_numeric(data$total_historical_rounds),

    lag_round =
      safe_numeric(data$lag_round),

    days_since_last_obs =
      safe_numeric(data$days_since_last_obs),

    tenure_lag =
      safe_numeric(data$tenure_lag),

    employed_lag =
      safe_numeric(data$employed_lag),

    # Education
    education_nqf =
      extract_nqf(data$education_level)
  )


  # ----------------------------------------------------------
  # Missingness indicators
  # ----------------------------------------------------------

  x$has_previous_observation <-
    as.numeric(
      !is.na(data$lag_round)
    )

  x$has_employed_history <-
    as.numeric(
      !is.na(data$employed_lag)
    )

  x$has_tenure_history <-
    as.numeric(
      !is.na(data$tenure_lag)
    )

  x$has_work_readiness <-
    as.numeric(
      !is.na(data$work_readiness_score)
    )

  x$has_days_since_observation <-
    as.numeric(
      !is.na(data$days_since_last_obs)
    )


  # ----------------------------------------------------------
  # Historical employment signal
  # ----------------------------------------------------------

  rounds <- safe_numeric(
    data$total_historical_rounds
  )

  employed <- safe_numeric(
    data$employed_lag
  )

  x$employment_history_strength <-
    ifelse(
      is.na(employed),
      0,
      employed * pmin(
        ifelse(is.na(rounds), 0, rounds),
        5
      )
    )


  # ----------------------------------------------------------
  # Recent observation
  # ----------------------------------------------------------

  days <- safe_numeric(
    data$days_since_last_obs
  )

  x$recent_observation <-
    ifelse(
      is.na(days),
      0,
      as.numeric(days <= 180)
    )


  # ----------------------------------------------------------
  # Long participant history
  # ----------------------------------------------------------

  x$long_history <-
    ifelse(
      is.na(rounds),
      0,
      as.numeric(rounds >= 3)
    )


  # ----------------------------------------------------------
  # Age nonlinear features
  # ----------------------------------------------------------

  age <- safe_numeric(
    data$age
  )

  x$age_squared <-
    ifelse(
      is.na(age),
      NA,
      age^2
    )


  # ----------------------------------------------------------
  # Work readiness nonlinear feature
  # ----------------------------------------------------------

  wr <- safe_numeric(
    data$work_readiness_score
  )

  x$work_readiness_squared <-
    ifelse(
      is.na(wr),
      NA,
      wr^2
    )


  # ----------------------------------------------------------
  # Clean impossible values
  # ----------------------------------------------------------

  x$age[
    x$age < 15 |
      x$age > 80
  ] <- NA

  x$school_quintile[
    x$school_quintile < 0 |
      x$school_quintile > 5
  ] <- NA

  x$work_readiness_score[
    x$work_readiness_score < 0 |
      x$work_readiness_score > 1
  ] <- NA


  x
}


# ------------------------------------------------------------
# 7. CREATE FEATURES
# ------------------------------------------------------------

x_train <- create_features(
  train_data
)

x_test <- create_features(
  test_data
)


cat(
  "Number of features:",
  ncol(x_train),
  "\n\n"
)


# ------------------------------------------------------------
# 8. ENSURE IDENTICAL FEATURES
# ------------------------------------------------------------

if (
  !identical(
    names(x_train),
    names(x_test)
  )
) {

  stop(
    "Training and testing feature names do not match."
  )
}


# ------------------------------------------------------------
# 9. IMPUTE MISSING VALUES
# ------------------------------------------------------------

for (column in names(x_train)) {

  median_value <- median(
    x_train[[column]],
    na.rm = TRUE
  )

  if (
    is.na(median_value) ||
    is.infinite(median_value)
  ) {

    median_value <- 0
  }


  x_train[[column]][
    is.na(x_train[[column]])
  ] <- median_value


  x_test[[column]][
    is.na(x_test[[column]])
  ] <- median_value
}


# ------------------------------------------------------------
# 10. TARGET
# ------------------------------------------------------------

y_train <- factor(
  train_data$employed_status,
  levels = c(0, 1)
)


cat("Target distribution:\n")

print(
  table(y_train)
)

cat("\n")


# ------------------------------------------------------------
# 11. COMBINE FEATURES WITH TARGET
# ------------------------------------------------------------

model_train <- x_train

model_train$target <- y_train


# ------------------------------------------------------------
# 12. TRAIN RANDOM FOREST
# ------------------------------------------------------------

cat("============================================\n")
cat("TRAINING RANDOM FOREST\n")
cat("============================================\n\n")


set.seed(42)

rf_model <- ranger(

  target ~ .,

  data = model_train,

  probability = TRUE,

  num.trees = 800,

  mtry = max(
    2,
    floor(
      sqrt(ncol(x_train))
    )
  ),

  min.node.size = 20,

  sample.fraction = 0.8,

  replace = TRUE,

  importance = "impurity",

  respect.unordered.factors = "order",

  num.threads = max(
    1,
    parallel::detectCores() - 1
  )
)


cat("Random forest trained successfully.\n\n")


# ------------------------------------------------------------
# 13. FEATURE IMPORTANCE
# ------------------------------------------------------------

cat("============================================\n")
cat("TOP FEATURE IMPORTANCE\n")
cat("============================================\n\n")


importance_table <- data.frame(

  feature =
    names(rf_model$variable.importance),

  importance =
    as.numeric(
      rf_model$variable.importance
    ),

  row.names = NULL
)


importance_table <- importance_table[
  order(
    importance_table$importance,
    decreasing = TRUE
  ),
]


print(
  head(
    importance_table,
    20
  )
)

cat("\n")


# ------------------------------------------------------------
# 14. GENERATE PREDICTIONS
# ------------------------------------------------------------

cat("============================================\n")
cat("GENERATING ROUND 9 PREDICTIONS\n")
cat("============================================\n\n")


rf_prediction <- predict(
  rf_model,
  data = x_test
)


test_prob <- rf_prediction$predictions[
  ,
  "1"
]


# ------------------------------------------------------------
# 15. CHECK PREDICTIONS
# ------------------------------------------------------------

cat("Prediction summary:\n")

print(
  summary(test_prob)
)

cat("\n")


if (
  any(is.na(test_prob))
) {

  stop(
    "Model produced NA predictions."
  )
}


if (
  length(test_prob) != nrow(test_data)
) {

  stop(
    paste(
      "Wrong number of predictions.",
      "Expected:",
      nrow(test_data),
      "Got:",
      length(test_prob)
    )
  )
}


# ------------------------------------------------------------
# 16. CREATE SUBMISSION
# ------------------------------------------------------------

submission_exp13 <- data.frame(

  anonymised_id =
    test_data$anonymised_id,

  employed_status =
    test_prob
)


# ------------------------------------------------------------
# 17. SAVE
# ------------------------------------------------------------

output_file <-
  "submission_exp13_random_forest.csv"


write.csv(
  submission_exp13,
  output_file,
  row.names = FALSE,
  quote = FALSE
)


# ------------------------------------------------------------
# 18. FINAL CHECK
# ------------------------------------------------------------

cat("============================================\n")
cat("EXPERIMENT 13 COMPLETE\n")
cat("============================================\n\n")

cat(
  "Saved:",
  output_file,
  "\n"
)

cat(
  "Rows:",
  nrow(submission_exp13),
  "\n\n"
)


cat("First 10 predictions:\n")

print(
  head(
    submission_exp13,
    10
  )
)

cat("\n")


cat("Prediction summary:\n")

print(
  summary(
    submission_exp13$employed_status
  )
)

cat("\n============================================\n")
cat("READY FOR KAGGLE SUBMISSION\n")
cat("============================================\n")

EXPERIMENT 13 — RANDOM FOREST



Installing package into 'C:/Users/27823/AppData/Local/R/win-library/4.4'
(as 'lib' is unspecified)



package 'ranger' successfully unpacked and MD5 sums checked

The downloaded binary packages are in
	C:\Users\27823\AppData\Local\Temp\RtmpENgYVa\downloaded_packages


Warning message:
"package 'ranger' was built under R version 4.4.3"


Training rows: 16719 
Testing rows : 3230 

Training rows after removing missing target: 16653 

Number of features: 19 

Target distribution:
y_train
    0     1 
11375  5278 

TRAINING RANDOM FOREST

Random forest trained successfully.

TOP FEATURE IMPORTANCE

                       feature importance
19      work_readiness_squared 187.657133
3         work_readiness_score 187.258506
6          days_since_last_obs 148.717949
7                   tenure_lag 134.289889
18                 age_squared  94.877562
1                          age  93.742558
2              school_quintile  72.882495
15 employment_history_strength  72.684489
8                 employed_lag  67.800209
5                    lag_round  63.034971
9                education_nqf  46.435625
4      total_historical_rounds  17.295159
13          has_work_readiness  13.455522
12          has_tenure_history  13.433500
16          recent_observation  10.836800
17                long_history  10.093703
14  has_days_since_obse

In [4]:
# ============================================================
# EXPERIMENT 14 — STRONGER ELASTIC NET
# CONTROLLED FEATURES + L1/L2 REGULARISATION
# ============================================================

cat("============================================\n")
cat("EXPERIMENT 14 — STRONGER ELASTIC NET\n")
cat("============================================\n\n")


# ------------------------------------------------------------
# 1. PACKAGE
# ------------------------------------------------------------

if (!requireNamespace("glmnet", quietly = TRUE)) {
  install.packages(
    "glmnet",
    repos = "https://cloud.r-project.org"
  )
}

library(glmnet)


# ------------------------------------------------------------
# 2. LOAD DATA
# ------------------------------------------------------------

data_dir <- "../../assets/dataset"

train_data <- read.csv(
  file.path(data_dir, "train.csv"),
  stringsAsFactors = FALSE
)

test_data <- read.csv(
  file.path(data_dir, "test.csv"),
  stringsAsFactors = FALSE
)

cat("Training rows:", nrow(train_data), "\n")
cat("Testing rows :", nrow(test_data), "\n\n")


# ------------------------------------------------------------
# 3. REMOVE MISSING TARGET
# ------------------------------------------------------------

train_data <- train_data[
  !is.na(train_data$employed_status),
]

cat(
  "Training rows after removing missing target:",
  nrow(train_data),
  "\n\n"
)


# ------------------------------------------------------------
# 4. SAFE NUMERIC
# ------------------------------------------------------------

safe_numeric <- function(x) {

  suppressWarnings(
    as.numeric(as.character(x))
  )

}


# ------------------------------------------------------------
# 5. EXTRACT NQF
# ------------------------------------------------------------

extract_nqf <- function(x) {

  x <- as.character(x)

  result <- rep(
    NA_real_,
    length(x)
  )

  for (i in seq_along(x)) {

    value <- x[i]

    if (
      is.na(value) ||
      value == ""
    ) {
      next
    }


    # NQF

    match_nqf <- regexec(
      "NQF Level ([0-9]+)",
      value,
      ignore.case = TRUE
    )

    parts <- regmatches(
      value,
      match_nqf
    )

    if (
      length(parts[[1]]) >= 2
    ) {

      result[i] <-
        suppressWarnings(
          as.numeric(parts[[1]][2])
        )

      next
    }


    # Grade

    match_grade <- regexec(
      "Grade ([0-9]+)",
      value,
      ignore.case = TRUE
    )

    grade_parts <- regmatches(
      value,
      match_grade
    )

    if (
      length(grade_parts[[1]]) >= 2
    ) {

      grade <-
        suppressWarnings(
          as.numeric(
            grade_parts[[1]][2]
          )
        )

      if (!is.na(grade)) {

        result[i] <-
          max(
            grade - 8,
            1
          )

      }
    }
  }

  result
}


# ------------------------------------------------------------
# 6. FEATURE ENGINEERING
# ------------------------------------------------------------

create_features <- function(data) {

  x <- data.frame(

    # ------------------------------
    # Core
    # ------------------------------

    age =
      safe_numeric(data$age),

    school_quintile =
      safe_numeric(data$school_quintile),

    work_readiness_score =
      safe_numeric(
        data$work_readiness_score
      ),


    # ------------------------------
    # Historical
    # ------------------------------

    total_historical_rounds =
      safe_numeric(
        data$total_historical_rounds
      ),

    lag_round =
      safe_numeric(
        data$lag_round
      ),

    days_since_last_obs =
      safe_numeric(
        data$days_since_last_obs
      ),

    tenure_lag =
      safe_numeric(
        data$tenure_lag
      ),

    employed_lag =
      safe_numeric(
        data$employed_lag
      ),


    # ------------------------------
    # Education
    # ------------------------------

    education_nqf =
      extract_nqf(
        data$education_level
      )
  )


  # ----------------------------------------------------------
  # Missingness
  # ----------------------------------------------------------

  x$has_previous_observation <-
    as.numeric(
      !is.na(data$lag_round)
    )

  x$has_employed_history <-
    as.numeric(
      !is.na(data$employed_lag)
    )

  x$has_tenure_history <-
    as.numeric(
      !is.na(data$tenure_lag)
    )

  x$has_work_readiness <-
    as.numeric(
      !is.na(
        data$work_readiness_score
      )
    )

  x$has_days_since_observation <-
    as.numeric(
      !is.na(
        data$days_since_last_obs
      )
    )


  # ----------------------------------------------------------
  # Historical employment strength
  # ----------------------------------------------------------

  rounds <-
    safe_numeric(
      data$total_historical_rounds
    )

  employed <-
    safe_numeric(
      data$employed_lag
    )

  x$employment_history_strength <-
    ifelse(
      is.na(employed),
      0,
      employed *
        pmin(
          ifelse(
            is.na(rounds),
            0,
            rounds
          ),
          5
        )
    )


  # ----------------------------------------------------------
  # Recent observation
  # ----------------------------------------------------------

  days <-
    safe_numeric(
      data$days_since_last_obs
    )

  x$recent_observation <-
    ifelse(
      is.na(days),
      0,
      as.numeric(
        days <= 180
      )
    )


  # ----------------------------------------------------------
  # Long history
  # ----------------------------------------------------------

  x$long_history <-
    ifelse(
      is.na(rounds),
      0,
      as.numeric(
        rounds >= 3
      )
    )


  # ----------------------------------------------------------
  # Age nonlinear feature
  # ----------------------------------------------------------

  age <-
    safe_numeric(
      data$age
    )

  x$age_squared <-
    ifelse(
      is.na(age),
      NA,
      age^2
    )


  # ----------------------------------------------------------
  # Work readiness nonlinear feature
  # ----------------------------------------------------------

  wr <-
    safe_numeric(
      data$work_readiness_score
    )

  x$work_readiness_squared <-
    ifelse(
      is.na(wr),
      NA,
      wr^2
    )


  # ----------------------------------------------------------
  # Clean impossible values
  # ----------------------------------------------------------

  x$age[
    x$age < 15 |
      x$age > 80
  ] <- NA

  x$school_quintile[
    x$school_quintile < 0 |
      x$school_quintile > 5
  ] <- NA

  x$work_readiness_score[
    x$work_readiness_score < 0 |
      x$work_readiness_score > 1
  ] <- NA


  x
}


# ------------------------------------------------------------
# 7. CREATE FEATURES
# ------------------------------------------------------------

x_train_df <-
  create_features(
    train_data
  )

x_test_df <-
  create_features(
    test_data
  )


cat(
  "Number of features:",
  ncol(x_train_df),
  "\n\n"
)


# ------------------------------------------------------------
# 8. CHECK STRUCTURE
# ------------------------------------------------------------

if (
  !identical(
    names(x_train_df),
    names(x_test_df)
  )
) {

  stop(
    "Training and testing feature names do not match."
  )

}


# ------------------------------------------------------------
# 9. IMPUTE USING TRAINING MEDIANS
# ------------------------------------------------------------

for (
  column in names(x_train_df)
) {

  median_value <-
    median(
      x_train_df[[column]],
      na.rm = TRUE
    )

  if (
    is.na(median_value) ||
      is.infinite(median_value)
  ) {

    median_value <- 0

  }


  x_train_df[[column]][
    is.na(
      x_train_df[[column]]
    )
  ] <- median_value


  x_test_df[[column]][
    is.na(
      x_test_df[[column]]
    )
  ] <- median_value

}


# ------------------------------------------------------------
# 10. SCALE
# ------------------------------------------------------------

train_means <-
  sapply(
    x_train_df,
    mean
  )

train_sds <-
  sapply(
    x_train_df,
    sd
  )


train_sds[
  is.na(train_sds) |
    train_sds == 0
] <- 1


x_train_scaled <-
  sweep(
    as.matrix(x_train_df),
    2,
    train_means,
    "-"
  )

x_train_scaled <-
  sweep(
    x_train_scaled,
    2,
    train_sds,
    "/"
  )


x_test_scaled <-
  sweep(
    as.matrix(x_test_df),
    2,
    train_means,
    "-"
  )

x_test_scaled <-
  sweep(
    x_test_scaled,
    2,
    train_sds,
    "/"
  )


# ------------------------------------------------------------
# 11. TARGET
# ------------------------------------------------------------

y_train <-
  as.numeric(
    train_data$employed_status
  )


cat("Target distribution:\n")

print(
  table(y_train)
)

cat("\n")


# ------------------------------------------------------------
# 12. CROSS-VALIDATE ALPHA
# ------------------------------------------------------------

cat("============================================\n")
cat("TESTING ELASTIC NET STRENGTH\n")
cat("============================================\n\n")


set.seed(42)


# Exp 5 used alpha = 0.25.
#
# Here we test stronger L1 regularisation:
#
# 0.50
# 0.75
# 1.00
#
# The best alpha is selected using
# cross-validated AUC.


alpha_values <- c(
  0.50,
  0.75,
  1.00
)


cv_results <- data.frame(
  alpha = alpha_values,
  auc = NA_real_,
  lambda = NA_real_
)


for (
  i in seq_along(alpha_values)
) {

  current_alpha <-
    alpha_values[i]

  cat(
    "Testing alpha =",
    current_alpha,
    "\n"
  )


  cv_fit <-
    cv.glmnet(

      x_train_scaled,

      y_train,

      family = "binomial",

      alpha = current_alpha,

      nfolds = 5,

      type.measure = "auc",

      standardize = FALSE

    )


  best_index <-
    which.min(
      abs(
        cv_fit$lambda -
          cv_fit$lambda.min
      )
    )


  cv_results$alpha[i] <-
    current_alpha

  cv_results$auc[i] <-
    max(
      cv_fit$cvm
    )

  cv_results$lambda[i] <-
    cv_fit$lambda.min

}


cat("\n============================================\n")
cat("ALPHA COMPARISON\n")
cat("============================================\n\n")

print(
  cv_results
)

cat("\n")


# ------------------------------------------------------------
# 13. SELECT BEST ALPHA
# ------------------------------------------------------------

best_row <-
  which.max(
    cv_results$auc
  )

best_alpha <-
  cv_results$alpha[
    best_row
  ]

best_lambda <-
  cv_results$lambda[
    best_row
  ]


cat(
  "Selected alpha:",
  best_alpha,
  "\n"
)

cat(
  "Selected lambda:",
  best_lambda,
  "\n\n"
)


# ------------------------------------------------------------
# 14. TRAIN FINAL MODEL
# ------------------------------------------------------------

cat("============================================\n")
cat("TRAINING FINAL MODEL\n")
cat("============================================\n\n")


final_model <-
  glmnet(

    x_train_scaled,

    y_train,

    family = "binomial",

    alpha = best_alpha,

    lambda = best_lambda,

    standardize = FALSE

  )


cat(
  "Final model trained successfully.\n\n"
)


# ------------------------------------------------------------
# 15. COEFFICIENTS
# ------------------------------------------------------------

cat("============================================\n")
cat("IMPORTANT FEATURES\n")
cat("============================================\n\n")


coef_matrix <-
  as.matrix(
    coef(final_model)
  )


coef_table <-
  data.frame(

    feature =
      rownames(
        coef_matrix
      ),

    coefficient =
      as.numeric(
        coef_matrix[, 1]
      ),

    row.names = NULL

  )


coef_table <-
  coef_table[
    order(
      abs(
        coef_table$coefficient
      ),
      decreasing = TRUE
    ),
  ]


print(
  head(
    coef_table,
    20
  )
)

cat("\n")


# ------------------------------------------------------------
# 16. PREDICTIONS
# ------------------------------------------------------------

cat("============================================\n")
cat("GENERATING ROUND 9 PREDICTIONS\n")
cat("============================================\n\n")


test_prob <-
  as.numeric(
    predict(
      final_model,
      newx = x_test_scaled,
      type = "response"
    )
  )


# ------------------------------------------------------------
# 17. VALIDATION
# ------------------------------------------------------------

cat("Prediction summary:\n")

print(
  summary(test_prob)
)

cat("\n")


if (
  any(
    is.na(test_prob)
  )
) {

  stop(
    "Model produced NA predictions."
  )

}


if (
  length(test_prob) !=
    nrow(test_data)
) {

  stop(
    paste(
      "Wrong number of predictions.",
      "Expected:",
      nrow(test_data),
      "Got:",
      length(test_prob)
    )
  )

}


# ------------------------------------------------------------
# 18. CREATE SUBMISSION
# ------------------------------------------------------------

submission_exp14 <-
  data.frame(

    anonymised_id =
      test_data$anonymised_id,

    employed_status =
      test_prob

  )


# ------------------------------------------------------------
# 19. SAVE
# ------------------------------------------------------------

output_file <-
  "submission_exp14_stronger_elastic_net.csv"


write.csv(

  submission_exp14,

  output_file,

  row.names = FALSE,

  quote = FALSE

)


# ------------------------------------------------------------
# 20. FINAL OUTPUT
# ------------------------------------------------------------

cat("============================================\n")
cat("EXPERIMENT 14 COMPLETE\n")
cat("============================================\n\n")


cat(
  "Saved:",
  output_file,
  "\n"
)

cat(
  "Rows:",
  nrow(
    submission_exp14
  ),
  "\n\n"
)


cat("First 10 predictions:\n")

print(
  head(
    submission_exp14,
    10
  )
)

cat("\n")


cat("Prediction summary:\n")

print(
  summary(
    submission_exp14$employed_status
  )
)

cat("\n============================================\n")
cat("READY FOR KAGGLE SUBMISSION\n")
cat("============================================\n")

EXPERIMENT 14 — STRONGER ELASTIC NET

Training rows: 16719 
Testing rows : 3230 

Training rows after removing missing target: 16653 

Number of features: 19 

Target distribution:
y_train
    0     1 
11375  5278 

TESTING ELASTIC NET STRENGTH

Testing alpha = 0.5 
Testing alpha = 0.75 
Testing alpha = 1 

ALPHA COMPARISON

  alpha       auc       lambda
1  0.50 0.5936060 0.0003698270
2  0.75 0.5936590 0.0003925795
3  1.00 0.5945616 0.0002444448

Selected alpha: 1 
Selected lambda: 0.0002444448 

TRAINING FINAL MODEL

Final model trained successfully.

IMPORTANT FEATURES

                       feature  coefficient
1                  (Intercept) -0.787642317
4         work_readiness_score  0.234960924
16 employment_history_strength  0.221057235
2                          age  0.179261060
9                 employed_lag  0.111097359
20      work_readiness_squared -0.102593412
19                 age_squared -0.091444802
3              school_quintile  0.074315560
5      total_historical_

In [7]:
# ============================================================
# EXPERIMENT 15 — ROBUST NONLINEAR SPLINE LOGISTIC REGRESSION
# HISTORICAL + BEHAVIOURAL FEATURES
# ============================================================

cat("============================================\n")
cat("EXPERIMENT 15 — ROBUST SPLINE LOGISTIC MODEL\n")
cat("============================================\n\n")


# ------------------------------------------------------------
# 1. PACKAGE
# ------------------------------------------------------------

library(splines)


# ------------------------------------------------------------
# 2. LOAD DATA
# ------------------------------------------------------------

data_dir <- "../../assets/dataset"

train_data <- read.csv(
  file.path(data_dir, "train.csv"),
  stringsAsFactors = FALSE
)

test_data <- read.csv(
  file.path(data_dir, "test.csv"),
  stringsAsFactors = FALSE
)

cat("Training rows:", nrow(train_data), "\n")
cat("Testing rows :", nrow(test_data), "\n\n")


# ------------------------------------------------------------
# 3. REMOVE MISSING TARGET
# ------------------------------------------------------------

train_data <- train_data[
  !is.na(train_data$employed_status),
]

cat(
  "Training rows after removing missing target:",
  nrow(train_data),
  "\n\n"
)


# ------------------------------------------------------------
# 4. SAFE NUMERIC CONVERSION
# ------------------------------------------------------------

safe_numeric <- function(x) {

  suppressWarnings(
    as.numeric(as.character(x))
  )

}


# ------------------------------------------------------------
# 5. NQF EXTRACTION
# ------------------------------------------------------------

extract_nqf <- function(x) {

  x <- as.character(x)

  result <- rep(
    NA_real_,
    length(x)
  )

  for (i in seq_along(x)) {

    value <- x[i]

    if (
      is.na(value) ||
      value == ""
    ) {
      next
    }


    # NQF Level

    match_nqf <- regexec(
      "NQF Level ([0-9]+)",
      value,
      ignore.case = TRUE
    )

    parts <- regmatches(
      value,
      match_nqf
    )

    if (
      length(parts[[1]]) >= 2
    ) {

      result[i] <-
        suppressWarnings(
          as.numeric(
            parts[[1]][2]
          )
        )

      next

    }


    # Grade

    match_grade <- regexec(
      "Grade ([0-9]+)",
      value,
      ignore.case = TRUE
    )

    grade_parts <- regmatches(
      value,
      match_grade
    )

    if (
      length(grade_parts[[1]]) >= 2
    ) {

      grade <-
        suppressWarnings(
          as.numeric(
            grade_parts[[1]][2]
          )
        )

      if (!is.na(grade)) {

        result[i] <-
          max(
            grade - 8,
            1
          )

      }

    }

  }

  result
}


# ------------------------------------------------------------
# 6. FEATURE ENGINEERING
# ------------------------------------------------------------

create_features <- function(data) {

  x <- data.frame(

    age =
      safe_numeric(
        data$age
      ),

    school_quintile =
      safe_numeric(
        data$school_quintile
      ),

    work_readiness_score =
      safe_numeric(
        data$work_readiness_score
      ),

    total_historical_rounds =
      safe_numeric(
        data$total_historical_rounds
      ),

    lag_round =
      safe_numeric(
        data$lag_round
      ),

    days_since_last_obs =
      safe_numeric(
        data$days_since_last_obs
      ),

    tenure_lag =
      safe_numeric(
        data$tenure_lag
      ),

    employed_lag =
      safe_numeric(
        data$employed_lag
      ),

    education_nqf =
      extract_nqf(
        data$education_level
      )
  )


  # ----------------------------------------------------------
  # Missingness indicators
  # ----------------------------------------------------------

  x$has_previous_observation <-
    as.numeric(
      !is.na(
        data$lag_round
      )
    )

  x$has_employed_history <-
    as.numeric(
      !is.na(
        data$employed_lag
      )
    )

  x$has_tenure_history <-
    as.numeric(
      !is.na(
        data$tenure_lag
      )
    )

  x$has_work_readiness <-
    as.numeric(
      !is.na(
        data$work_readiness_score
      )
    )

  x$has_days_since_observation <-
    as.numeric(
      !is.na(
        data$days_since_last_obs
      )
    )


  # ----------------------------------------------------------
  # Historical employment strength
  # ----------------------------------------------------------

  employed <-
    safe_numeric(
      data$employed_lag
    )

  rounds <-
    safe_numeric(
      data$total_historical_rounds
    )

  x$employment_history_strength <-
    ifelse(
      is.na(employed) |
        is.na(rounds),
      0,
      employed *
        pmin(
          rounds,
          5
        )
    )


  # ----------------------------------------------------------
  # Recent observation
  # ----------------------------------------------------------

  days <-
    safe_numeric(
      data$days_since_last_obs
    )

  x$recent_observation <-
    ifelse(
      is.na(days),
      0,
      as.numeric(
        days <= 180
      )
    )


  # ----------------------------------------------------------
  # Long history
  # ----------------------------------------------------------

  x$long_history <-
    ifelse(
      is.na(rounds),
      0,
      as.numeric(
        rounds >= 3
      )
    )


  # ----------------------------------------------------------
  # Basic nonlinear terms
  # ----------------------------------------------------------

  age <-
    safe_numeric(
      data$age
    )

  x$age_squared <-
    ifelse(
      is.na(age),
      NA,
      age^2
    )


  wr <-
    safe_numeric(
      data$work_readiness_score
    )

  x$work_readiness_squared <-
    ifelse(
      is.na(wr),
      NA,
      wr^2
    )


  # ----------------------------------------------------------
  # Clean impossible values
  # ----------------------------------------------------------

  x$age[
    x$age < 15 |
      x$age > 80
  ] <- NA

  x$school_quintile[
    x$school_quintile < 0 |
      x$school_quintile > 5
  ] <- NA

  x$work_readiness_score[
    x$work_readiness_score < 0 |
      x$work_readiness_score > 1
  ] <- NA


  x
}


# ------------------------------------------------------------
# 7. CREATE FEATURES
# ------------------------------------------------------------

x_train <- create_features(
  train_data
)

x_test <- create_features(
  test_data
)


cat(
  "Initial features:",
  ncol(x_train),
  "\n\n"
)


# ------------------------------------------------------------
# 8. IMPUTE USING TRAINING MEDIANS
# ------------------------------------------------------------

for (
  column in names(x_train)
) {

  med <-
    median(
      x_train[[column]],
      na.rm = TRUE
    )

  if (
    is.na(med) ||
      is.infinite(med)
  ) {

    med <- 0

  }


  x_train[[column]][
    is.na(
      x_train[[column]]
    )
  ] <- med


  x_test[[column]][
    is.na(
      x_test[[column]]
    )
  ] <- med

}


# ------------------------------------------------------------
# 9. CHECK UNIQUE VALUES
# ------------------------------------------------------------

cat("Unique values in important variables:\n\n")

cat(
  "age:",
  length(
    unique(
      x_train$age
    )
  ),
  "\n"
)

cat(
  "work_readiness_score:",
  length(
    unique(
      x_train$work_readiness_score
    )
  ),
  "\n"
)

cat(
  "days_since_last_obs:",
  length(
    unique(
      x_train$days_since_last_obs
    )
  ),
  "\n"
)

cat(
  "tenure_lag:",
  length(
    unique(
      x_train$tenure_lag
    )
  ),
  "\n"
)

cat(
  "total_historical_rounds:",
  length(
    unique(
      x_train$total_historical_rounds
    )
  ),
  "\n\n"
)


# ------------------------------------------------------------
# 10. TARGET
# ------------------------------------------------------------

y <- as.numeric(
  as.character(
    train_data$employed_status
  )
)


cat("Target distribution:\n")

print(
  table(y)
)

cat("\n")


# ------------------------------------------------------------
# 11. CREATE SPLINES SAFELY
# ------------------------------------------------------------

cat("============================================\n")
cat("CREATING NONLINEAR FEATURES\n")
cat("============================================\n\n")


# Start with normal features.

model_train <- x_train
model_test <- x_test


# ------------------------------------------------------------
# AGE SPLINE
# ------------------------------------------------------------

if (
  length(
    unique(
      x_train$age
    )
  ) >= 5
) {

  cat(
    "Adding age spline...\n"
  )

  age_basis <- ns(
    x_train$age,
    df = 3
  )

  age_test_basis <- predict(
    age_basis,
    x_test$age
  )

  model_train$age_s1 <-
    age_basis[, 1]

  model_train$age_s2 <-
    age_basis[, 2]

  model_train$age_s3 <-
    age_basis[, 3]

  model_test$age_s1 <-
    age_test_basis[, 1]

  model_test$age_s2 <-
    age_test_basis[, 2]

  model_test$age_s3 <-
    age_test_basis[, 3]

} else {

  cat(
    "Skipping age spline — insufficient unique values.\n"
  )

}


# ------------------------------------------------------------
# WORK READINESS SPLINE
# ------------------------------------------------------------

if (
  length(
    unique(
      x_train$work_readiness_score
    )
  ) >= 5
) {

  cat(
    "Adding work readiness spline...\n"
  )

  wr_basis <- ns(
    x_train$work_readiness_score,
    df = 3
  )

  wr_test_basis <- predict(
    wr_basis,
    x_test$work_readiness_score
  )

  model_train$wr_s1 <-
    wr_basis[, 1]

  model_train$wr_s2 <-
    wr_basis[, 2]

  model_train$wr_s3 <-
    wr_basis[, 3]

  model_test$wr_s1 <-
    wr_test_basis[, 1]

  model_test$wr_s2 <-
    wr_test_basis[, 2]

  model_test$wr_s3 <-
    wr_test_basis[, 3]

} else {

  cat(
    "Skipping work readiness spline.\n"
  )

}


# ------------------------------------------------------------
# DAYS SINCE LAST OBSERVATION SPLINE
# ------------------------------------------------------------

if (
  length(
    unique(
      x_train$days_since_last_obs
    )
  ) >= 5
) {

  cat(
    "Adding observation recency spline...\n"
  )

  days_basis <- ns(
    x_train$days_since_last_obs,
    df = 3
  )

  days_test_basis <- predict(
    days_basis,
    x_test$days_since_last_obs
  )

  model_train$days_s1 <-
    days_basis[, 1]

  model_train$days_s2 <-
    days_basis[, 2]

  model_train$days_s3 <-
    days_basis[, 3]

  model_test$days_s1 <-
    days_test_basis[, 1]

  model_test$days_s2 <-
    days_test_basis[, 2]

  model_test$days_s3 <-
    days_test_basis[, 3]

} else {

  cat(
    "Skipping days spline.\n"
  )

}


# ------------------------------------------------------------
# TENURE SPLINE
# ------------------------------------------------------------

if (
  length(
    unique(
      x_train$tenure_lag
    )
  ) >= 5
) {

  cat(
    "Adding tenure spline...\n"
  )

  tenure_basis <- ns(
    x_train$tenure_lag,
    df = 3
  )

  tenure_test_basis <- predict(
    tenure_basis,
    x_test$tenure_lag
  )

  model_train$tenure_s1 <-
    tenure_basis[, 1]

  model_train$tenure_s2 <-
    tenure_basis[, 2]

  model_train$tenure_s3 <-
    tenure_basis[, 3]

  model_test$tenure_s1 <-
    tenure_test_basis[, 1]

  model_test$tenure_s2 <-
    tenure_test_basis[, 2]

  model_test$tenure_s3 <-
    tenure_test_basis[, 3]

} else {

  cat(
    "Skipping tenure spline.\n"
  )

}


# ------------------------------------------------------------
# IMPORTANT:
# total_historical_rounds is NOT splined.
#
# The previous version failed here because the variable
# does not contain enough distinct values for ns(df = 3).
# We retain it as a normal numeric feature.
# ------------------------------------------------------------


# ------------------------------------------------------------
# 12. REMOVE REDUNDANT SQUARED TERMS
# ------------------------------------------------------------

# Splines already model nonlinear behaviour.

if ("age_squared" %in% names(model_train)) {

  model_train$age_squared <- NULL
  model_test$age_squared <- NULL

}

if (
  "work_readiness_squared" %in%
    names(model_train)
) {

  model_train$work_readiness_squared <- NULL
  model_test$work_readiness_squared <- NULL

}


# ------------------------------------------------------------
# 13. REMOVE INVALID / CONSTANT FEATURES
# ------------------------------------------------------------

keep_columns <- sapply(
  model_train,
  function(x) {

    length(
      unique(x)
    ) > 1 &&
      all(
        is.finite(x)
      )

  }
)


model_train <- model_train[
  ,
  keep_columns,
  drop = FALSE
]


# ------------------------------------------------------------
# 14. FORCE TEST TO SAME FEATURES
# ------------------------------------------------------------

missing_test_columns <-
  setdiff(
    names(model_train),
    names(model_test)
  )


if (
  length(
    missing_test_columns
  ) > 0
) {

  for (
    column in missing_test_columns
  ) {

    model_test[[column]] <- 0

  }

}


extra_test_columns <-
  setdiff(
    names(model_test),
    names(model_train)
  )


if (
  length(
    extra_test_columns
  ) > 0
) {

  model_test <- model_test[
    ,
    !(
      names(model_test) %in%
        extra_test_columns
    ),
    drop = FALSE
  ]

}


model_test <- model_test[
  ,
  names(model_train),
  drop = FALSE
]


# ------------------------------------------------------------
# 15. FINAL FEATURE COUNT
# ------------------------------------------------------------

cat(
  "\nFinal model features:",
  ncol(model_train),
  "\n"
)

cat(
  "Training feature rows:",
  nrow(model_train),
  "\n"
)

cat(
  "Testing feature rows:",
  nrow(model_test),
  "\n\n"
)


# ------------------------------------------------------------
# 16. TRAIN MODEL
# ------------------------------------------------------------

cat("============================================\n")
cat("TRAINING SPLINE LOGISTIC MODEL\n")
cat("============================================\n\n")


model_formula <- as.formula(
  paste(
    "y ~",
    paste(
      names(model_train),
      collapse = " + "
    )
  )
)


spline_model <- glm(
  model_formula,
  data = model_train,
  family = binomial(
    link = "logit"
  )
)


cat(
  "Model trained successfully.\n\n"
)


cat(
  "AIC:",
  AIC(
    spline_model
  ),
  "\n\n"
)


# ------------------------------------------------------------
# 17. PREDICTIONS
# ------------------------------------------------------------

cat("============================================\n")
cat("GENERATING ROUND 9 PREDICTIONS\n")
cat("============================================\n\n")


test_prob <- predict(
  spline_model,
  newdata = model_test,
  type = "response"
)


# ------------------------------------------------------------
# 18. CHECK PREDICTIONS
# ------------------------------------------------------------

cat(
  "Prediction summary:\n"
)

print(
  summary(
    test_prob
  )
)

cat("\n")


if (
  any(
    is.na(
      test_prob
    )
  )
) {

  stop(
    "Model produced NA predictions."
  )

}


if (
  length(
    test_prob
  ) !=
    nrow(
      test_data
    )
) {

  stop(
    paste(
      "Wrong number of predictions.",
      "Expected:",
      nrow(test_data),
      "Got:",
      length(test_prob)
    )
  )

}


# ------------------------------------------------------------
# 19. CREATE SUBMISSION
# ------------------------------------------------------------

submission_exp15 <- data.frame(

  anonymised_id =
    test_data$anonymised_id,

  employed_status =
    as.numeric(
      test_prob
    )

)


# ------------------------------------------------------------
# 20. SAVE SUBMISSION
# ------------------------------------------------------------

output_file <-
  "submission_exp15_spline_logistic.csv"


write.csv(
  submission_exp15,
  output_file,
  row.names = FALSE,
  quote = FALSE
)


# ------------------------------------------------------------
# 21. FINAL OUTPUT
# ------------------------------------------------------------

cat("============================================\n")
cat("EXPERIMENT 15 COMPLETE\n")
cat("============================================\n\n")


cat(
  "Saved:",
  output_file,
  "\n"
)

cat(
  "Rows:",
  nrow(
    submission_exp15
  ),
  "\n\n"
)


cat(
  "First 10 predictions:\n"
)

print(
  head(
    submission_exp15,
    10
  )
)


cat(
  "\nPrediction summary:\n"
)

print(
  summary(
    submission_exp15$employed_status
  )
)


cat(
  "\n============================================\n"
)

cat(
  "READY FOR KAGGLE SUBMISSION\n"
)

cat(
  "============================================\n"
)

EXPERIMENT 15 — ROBUST SPLINE LOGISTIC MODEL

Training rows: 16719 
Testing rows : 3230 

Training rows after removing missing target: 16653 

Initial features: 19 

Unique values in important variables:

age: 32 
work_readiness_score: 2989 
days_since_last_obs: 286 
tenure_lag: 723 
total_historical_rounds: 5 

Target distribution:
y
    0     1 
11375  5278 

CREATING NONLINEAR FEATURES

Adding age spline...
Adding work readiness spline...
Adding observation recency spline...
Adding tenure spline...

Final model features: 29 
Training feature rows: 16653 
Testing feature rows: 3230 

TRAINING SPLINE LOGISTIC MODEL

Model trained successfully.

AIC: 20250.98 

GENERATING ROUND 9 PREDICTIONS

Prediction summary:
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
 0.1356  0.3114  0.3193  0.3530  0.3436  0.7261 

EXPERIMENT 15 COMPLETE

Saved: submission_exp15_spline_logistic.csv 
Rows: 3230 

First 10 predictions:
   anonymised_id employed_status
1          ID_10       0.3180464
2        

In [ ]:
# ============================================================
# EXPERIMENT 16 — XGBOOST BOOSTED TREES
# NUMERIC + HISTORICAL + BEHAVIOURAL FEATURES
# ============================================================

cat("============================================\n")
cat("EXPERIMENT 16 — XGBOOST BOOSTED TREES\n")
cat("============================================\n\n")


# ------------------------------------------------------------
# 1. PACKAGE
# ------------------------------------------------------------

if (!requireNamespace("xgboost", quietly = TRUE)) {
  install.packages(
    "xgboost",
    repos = "https://cloud.r-project.org"
  )
}

library(xgboost)


# ------------------------------------------------------------
# 2. LOAD DATA
# ------------------------------------------------------------

data_dir <- "../../assets/dataset"

train_data <- read.csv(
  file.path(data_dir, "train.csv"),
  stringsAsFactors = FALSE
)

test_data <- read.csv(
  file.path(data_dir, "test.csv"),
  stringsAsFactors = FALSE
)

cat("Training rows:", nrow(train_data), "\n")
cat("Testing rows :", nrow(test_data), "\n\n")


# ------------------------------------------------------------
# 3. REMOVE MISSING TARGET
# ------------------------------------------------------------

train_data <- train_data[
  !is.na(train_data$employed_status),
]

cat(
  "Training rows after removing missing target:",
  nrow(train_data),
  "\n\n"
)


# ------------------------------------------------------------
# 4. SAFE NUMERIC CONVERSION
# ------------------------------------------------------------

safe_numeric <- function(x) {
  suppressWarnings(
    as.numeric(as.character(x))
  )
}


# ------------------------------------------------------------
# 5. EDUCATION / NQF
# ------------------------------------------------------------

extract_nqf <- function(x) {

  x <- as.character(x)

  result <- rep(NA_real_, length(x))

  for (i in seq_along(x)) {

    value <- x[i]

    if (is.na(value) || value == "") {
      next
    }

    # NQF Level

    match_nqf <- regexec(
      "NQF Level ([0-9]+)",
      value,
      ignore.case = TRUE
    )

    parts <- regmatches(
      value,
      match_nqf
    )

    if (
      length(parts[[1]]) >= 2 &&
      !is.na(parts[[1]][2])
    ) {

      result[i] <- suppressWarnings(
        as.numeric(parts[[1]][2])
      )

      next
    }

    # Grade

    match_grade <- regexec(
      "Grade ([0-9]+)",
      value,
      ignore.case = TRUE
    )

    grade_parts <- regmatches(
      value,
      match_grade
    )

    if (
      length(grade_parts[[1]]) >= 2
    ) {

      grade <- suppressWarnings(
        as.numeric(grade_parts[[1]][2])
      )

      if (!is.na(grade)) {

        result[i] <- max(
          grade - 8,
          1
        )

      }

    }

  }

  result
}


# ------------------------------------------------------------
# 6. FEATURE ENGINEERING
# ------------------------------------------------------------

create_features <- function(data) {

  x <- data.frame(

    age =
      safe_numeric(data$age),

    school_quintile =
      safe_numeric(data$school_quintile),

    work_readiness_score =
      safe_numeric(data$work_readiness_score),

    total_historical_rounds =
      safe_numeric(data$total_historical_rounds),

    lag_round =
      safe_numeric(data$lag_round),

    days_since_last_obs =
      safe_numeric(data$days_since_last_obs),

    tenure_lag =
      safe_numeric(data$tenure_lag),

    employed_lag =
      safe_numeric(data$employed_lag),

    education_nqf =
      extract_nqf(data$education_level)

  )


  # ----------------------------------------------------------
  # Missingness indicators
  # ----------------------------------------------------------

  x$has_previous_observation <-
    as.numeric(!is.na(data$lag_round))

  x$has_employed_history <-
    as.numeric(!is.na(data$employed_lag))

  x$has_tenure_history <-
    as.numeric(!is.na(data$tenure_lag))

  x$has_work_readiness <-
    as.numeric(!is.na(data$work_readiness_score))

  x$has_days_since_observation <-
    as.numeric(!is.na(data$days_since_last_obs))


  # ----------------------------------------------------------
  # Historical employment strength
  # ----------------------------------------------------------

  employed_lag_num <-
    safe_numeric(data$employed_lag)

  rounds <-
    safe_numeric(data$total_historical_rounds)

  x$employment_history_strength <-
    ifelse(
      is.na(employed_lag_num),
      0,
      employed_lag_num *
        pmin(
          ifelse(is.na(rounds), 0, rounds),
          5
        )
    )


  # ----------------------------------------------------------
  # Recent observation
  # ----------------------------------------------------------

  days <-
    safe_numeric(data$days_since_last_obs)

  x$recent_observation <-
    ifelse(
      is.na(days),
      0,
      as.numeric(days <= 180)
    )


  # ----------------------------------------------------------
  # Long history
  # ----------------------------------------------------------

  x$long_history <-
    ifelse(
      is.na(rounds),
      0,
      as.numeric(rounds >= 3)
    )


  # ----------------------------------------------------------
  # Age nonlinear features
  # ----------------------------------------------------------

  age <-
    safe_numeric(data$age)

  x$age_squared <-
    ifelse(
      is.na(age),
      NA,
      age^2
    )


  # ----------------------------------------------------------
  # Work readiness nonlinear feature
  # ----------------------------------------------------------

  wr <-
    safe_numeric(data$work_readiness_score)

  x$work_readiness_squared <-
    ifelse(
      is.na(wr),
      NA,
      wr^2
    )


  # ----------------------------------------------------------
  # Clean impossible values
  # ----------------------------------------------------------

  x$age[
    x$age < 15 |
      x$age > 80
  ] <- NA

  x$school_quintile[
    x$school_quintile < 0 |
      x$school_quintile > 5
  ] <- NA

  x$work_readiness_score[
    x$work_readiness_score < 0 |
      x$work_readiness_score > 1
  ] <- NA


  x

}


# ------------------------------------------------------------
# 7. CREATE FEATURES
# ------------------------------------------------------------

x_train_df <- create_features(train_data)
x_test_df <- create_features(test_data)

cat(
  "Number of features:",
  ncol(x_train_df),
  "\n\n"
)


# ------------------------------------------------------------
# 8. CHECK STRUCTURE
# ------------------------------------------------------------

if (
  !identical(
    names(x_train_df),
    names(x_test_df)
  )
) {

  stop(
    "Training and testing feature names do not match."
  )

}


# ------------------------------------------------------------
# 9. IMPUTE USING TRAINING MEDIANS
# ------------------------------------------------------------

for (column in names(x_train_df)) {

  median_value <- median(
    x_train_df[[column]],
    na.rm = TRUE
  )

  if (
    is.na(median_value) ||
    is.infinite(median_value)
  ) {

    median_value <- 0

  }

  x_train_df[[column]][
    is.na(x_train_df[[column]])
  ] <- median_value

  x_test_df[[column]][
    is.na(x_test_df[[column]])
  ] <- median_value

}


# ------------------------------------------------------------
# 10. MATRIX
# ------------------------------------------------------------

x_train <- as.matrix(x_train_df)
x_test <- as.matrix(x_test_df)


cat(
  "Train feature rows:",
  nrow(x_train),
  "\n"
)

cat(
  "Test feature rows:",
  nrow(x_test),
  "\n\n"
)


# ------------------------------------------------------------
# 11. TARGET
# ------------------------------------------------------------

y_train <- safe_numeric(
  train_data$employed_status
)

y_train <- ifelse(
  y_train > 0,
  1,
  0
)

cat("Target distribution:\n")

print(
  table(y_train)
)

cat("\n")


# ------------------------------------------------------------
# 12. XGBOOST MATRICES
# ------------------------------------------------------------

dtrain <- xgb.DMatrix(
  data = x_train,
  label = y_train
)

dtest <- xgb.DMatrix(
  data = x_test
)


# ------------------------------------------------------------
# 13. MODEL PARAMETERS
# ------------------------------------------------------------

params <- list(

  objective = "binary:logistic",

  eval_metric = "auc",

  max_depth = 3,

  learning_rate = 0.025,

  subsample = 0.85,

  colsample_bytree = 0.85,

  min_child_weight = 8,

  gamma = 0.1,

  reg_lambda = 5,

  reg_alpha = 0.25

)


# ------------------------------------------------------------
# 14. TRAIN
# ------------------------------------------------------------

cat("============================================\n")
cat("TRAINING XGBOOST\n")
cat("============================================\n\n")

set.seed(42)

model_xgb <- xgb.train(

  params = params,

  data = dtrain,

  nrounds = 500,

  verbose = 0

)


cat(
  "XGBoost model trained successfully.\n\n"
)


# ------------------------------------------------------------
# 15. PREDICTIONS
# ------------------------------------------------------------

cat("============================================\n")
cat("GENERATING PREDICTIONS\n")
cat("============================================\n\n")

test_prob <- predict(
  model_xgb,
  dtest
)


# ------------------------------------------------------------
# 16. CHECK
# ------------------------------------------------------------

if (any(is.na(test_prob))) {

  stop(
    "Model produced NA predictions."
  )

}

if (
  length(test_prob) != nrow(test_data)
) {

  stop(
    paste(
      "Wrong number of predictions.",
      "Expected:",
      nrow(test_data),
      "Got:",
      length(test_prob)
    )
  )

}


cat("Prediction summary:\n")

print(
  summary(test_prob)
)

cat("\n")


# ------------------------------------------------------------
# 17. SUBMISSION
# ------------------------------------------------------------

submission_exp16 <- data.frame(

  anonymised_id =
    test_data$anonymised_id,

  employed_status =
    test_prob

)


# ------------------------------------------------------------
# 18. SAVE
# ------------------------------------------------------------

output_file <-
  "submission_exp16_xgboost_boosted.csv"

write.csv(
  submission_exp16,
  output_file,
  row.names = FALSE,
  quote = FALSE
)


# ------------------------------------------------------------
# 19. FINAL CHECK
# ------------------------------------------------------------

cat("============================================\n")
cat("EXPERIMENT 16 COMPLETE\n")
cat("============================================\n\n")

cat(
  "Saved:",
  output_file,
  "\n"
)

cat(
  "Rows:",
  nrow(submission_exp16),
  "\n\n"
)

cat("First 10 predictions:\n")

print(
  head(
    submission_exp16,
    10
  )
)

cat("\n============================================\n")
cat("READY FOR KAGGLE SUBMISSION\n")
cat("============================================\n")

EXPERIMENT 16 — XGBOOST BOOSTED TREES



Warning message:
"package 'xgboost' was built under R version 4.4.3"


Training rows: 16719 
Testing rows : 3230 

Training rows after removing missing target: 16653 

Number of features: 19 

Train feature rows: 16653 
Test feature rows: 3230 

Target distribution:
y_train
    0     1 
11375  5278 

TRAINING XGBOOST

XGBoost model trained successfully.

GENERATING PREDICTIONS

Prediction summary:
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
 0.1051  0.3037  0.3153  0.3451  0.3415  0.7718 

EXPERIMENT 16 COMPLETE

Saved: submission_exp16_xgboost_boosted.csv 
Rows: 3230 

First 10 predictions:
   anonymised_id employed_status
1          ID_10       0.3137065
2         ID_100       0.3137065
3        ID_1000       0.3132719
4        ID_1001       0.3132719
5        ID_1002       0.3132719
6        ID_1003       0.3132719
7        ID_1004       0.3132719
8        ID_1005       0.3132719
9        ID_1006       0.3132719
10       ID_1007       0.3132719

READY FOR KAGGLE SUBMISSION


: 

In [1]:
# ============================================================
# EXPERIMENT 17 — RIDGE-HEAVY ELASTIC NET
# BASED ON BEST MODEL: EXP 5
# ============================================================

cat("============================================\n")
cat("EXPERIMENT 17 — RIDGE-HEAVY ELASTIC NET\n")
cat("============================================\n\n")


# ------------------------------------------------------------
# 1. PACKAGE
# ------------------------------------------------------------

if (!requireNamespace("glmnet", quietly = TRUE)) {
  install.packages(
    "glmnet",
    repos = "https://cloud.r-project.org"
  )
}

library(glmnet)


# ------------------------------------------------------------
# 2. LOAD DATA
# ------------------------------------------------------------

data_dir <- "../../assets/dataset"

train_data <- read.csv(
  file.path(data_dir, "train.csv"),
  stringsAsFactors = FALSE
)

test_data <- read.csv(
  file.path(data_dir, "test.csv"),
  stringsAsFactors = FALSE
)

cat("Training rows:", nrow(train_data), "\n")
cat("Testing rows :", nrow(test_data), "\n\n")


# ------------------------------------------------------------
# 3. REMOVE MISSING TARGET
# ------------------------------------------------------------

train_data <- train_data[
  !is.na(train_data$employed_status),
]

cat(
  "Training rows after removing missing target:",
  nrow(train_data),
  "\n\n"
)


# ------------------------------------------------------------
# 4. SAFE NUMERIC
# ------------------------------------------------------------

safe_numeric <- function(x) {
  suppressWarnings(
    as.numeric(as.character(x))
  )
}


# ------------------------------------------------------------
# 5. NQF EXTRACTION
# ------------------------------------------------------------

extract_nqf <- function(x) {

  x <- as.character(x)

  result <- rep(
    NA_real_,
    length(x)
  )

  for (i in seq_along(x)) {

    value <- x[i]

    if (
      is.na(value) ||
      value == ""
    ) {
      next
    }

    match_nqf <- regexec(
      "NQF Level ([0-9]+)",
      value,
      ignore.case = TRUE
    )

    parts <- regmatches(
      value,
      match_nqf
    )

    if (
      length(parts[[1]]) >= 2
    ) {

      result[i] <- suppressWarnings(
        as.numeric(parts[[1]][2])
      )

      next
    }

    match_grade <- regexec(
      "Grade ([0-9]+)",
      value,
      ignore.case = TRUE
    )

    grade_parts <- regmatches(
      value,
      match_grade
    )

    if (
      length(grade_parts[[1]]) >= 2
    ) {

      grade <- suppressWarnings(
        as.numeric(grade_parts[[1]][2])
      )

      if (!is.na(grade)) {

        result[i] <- max(
          grade - 8,
          1
        )

      }
    }
  }

  result
}


# ------------------------------------------------------------
# 6. FEATURE ENGINEERING
# ------------------------------------------------------------

create_features <- function(data) {

  x <- data.frame(

    age =
      safe_numeric(data$age),

    school_quintile =
      safe_numeric(data$school_quintile),

    work_readiness_score =
      safe_numeric(
        data$work_readiness_score
      ),

    total_historical_rounds =
      safe_numeric(
        data$total_historical_rounds
      ),

    lag_round =
      safe_numeric(
        data$lag_round
      ),

    days_since_last_obs =
      safe_numeric(
        data$days_since_last_obs
      ),

    tenure_lag =
      safe_numeric(
        data$tenure_lag
      ),

    employed_lag =
      safe_numeric(
        data$employed_lag
      ),

    education_nqf =
      extract_nqf(
        data$education_level
      )
  )


  # ----------------------------------------------------------
  # Missingness
  # ----------------------------------------------------------

  x$has_previous_observation <-
    as.numeric(
      !is.na(data$lag_round)
    )

  x$has_employed_history <-
    as.numeric(
      !is.na(data$employed_lag)
    )

  x$has_tenure_history <-
    as.numeric(
      !is.na(data$tenure_lag)
    )

  x$has_work_readiness <-
    as.numeric(
      !is.na(
        data$work_readiness_score
      )
    )

  x$has_days_since_observation <-
    as.numeric(
      !is.na(
        data$days_since_last_obs
      )
    )


  # ----------------------------------------------------------
  # Historical employment strength
  # ----------------------------------------------------------

  employed_lag_num <-
    safe_numeric(
      data$employed_lag
    )

  rounds <-
    safe_numeric(
      data$total_historical_rounds
    )

  x$employment_history_strength <-
    ifelse(
      is.na(employed_lag_num),
      0,
      employed_lag_num *
        pmin(
          ifelse(
            is.na(rounds),
            0,
            rounds
          ),
          5
        )
    )


  # ----------------------------------------------------------
  # Recent observation
  # ----------------------------------------------------------

  days <-
    safe_numeric(
      data$days_since_last_obs
    )

  x$recent_observation <-
    ifelse(
      is.na(days),
      0,
      as.numeric(
        days <= 180
      )
    )


  # ----------------------------------------------------------
  # Long history
  # ----------------------------------------------------------

  x$long_history <-
    ifelse(
      is.na(rounds),
      0,
      as.numeric(
        rounds >= 3
      )
    )


  # ----------------------------------------------------------
  # Age squared
  # ----------------------------------------------------------

  age <-
    safe_numeric(
      data$age
    )

  x$age_squared <-
    ifelse(
      is.na(age),
      NA,
      age^2
    )


  # ----------------------------------------------------------
  # Work readiness squared
  # ----------------------------------------------------------

  wr <-
    safe_numeric(
      data$work_readiness_score
    )

  x$work_readiness_squared <-
    ifelse(
      is.na(wr),
      NA,
      wr^2
    )


  # ----------------------------------------------------------
  # Clean impossible values
  # ----------------------------------------------------------

  x$age[
    x$age < 15 |
      x$age > 80
  ] <- NA

  x$school_quintile[
    x$school_quintile < 0 |
      x$school_quintile > 5
  ] <- NA

  x$work_readiness_score[
    x$work_readiness_score < 0 |
      x$work_readiness_score > 1
  ] <- NA


  x
}


# ------------------------------------------------------------
# 7. CREATE FEATURES
# ------------------------------------------------------------

x_train_df <- create_features(
  train_data
)

x_test_df <- create_features(
  test_data
)

cat(
  "Number of features:",
  ncol(x_train_df),
  "\n\n"
)


# ------------------------------------------------------------
# 8. IMPUTE
# ------------------------------------------------------------

for (column in names(x_train_df)) {

  median_value <- median(
    x_train_df[[column]],
    na.rm = TRUE
  )

  if (
    is.na(median_value) ||
    is.infinite(median_value)
  ) {
    median_value <- 0
  }

  x_train_df[[column]][
    is.na(x_train_df[[column]])
  ] <- median_value

  x_test_df[[column]][
    is.na(x_test_df[[column]])
  ] <- median_value
}


# ------------------------------------------------------------
# 9. SCALE
# ------------------------------------------------------------

train_means <- sapply(
  x_train_df,
  mean
)

train_sds <- sapply(
  x_train_df,
  sd
)

train_sds[
  is.na(train_sds) |
    train_sds == 0
] <- 1


x_train_scaled <- sweep(
  as.matrix(x_train_df),
  2,
  train_means,
  "-"
)

x_train_scaled <- sweep(
  x_train_scaled,
  2,
  train_sds,
  "/"
)


x_test_scaled <- sweep(
  as.matrix(x_test_df),
  2,
  train_means,
  "-"
)

x_test_scaled <- sweep(
  x_test_scaled,
  2,
  train_sds,
  "/"
)


# ------------------------------------------------------------
# 10. TARGET
# ------------------------------------------------------------

y_train <- safe_numeric(
  train_data$employed_status
)

y_train <- ifelse(
  y_train > 0,
  1,
  0
)

cat("Target distribution:\n")

print(
  table(y_train)
)

cat("\n")


# ------------------------------------------------------------
# 11. CROSS VALIDATION
# ------------------------------------------------------------

cat("============================================\n")
cat("CROSS-VALIDATING RIDGE-HEAVY MODEL\n")
cat("============================================\n\n")

set.seed(42)

cv_model <- cv.glmnet(

  x_train_scaled,

  y_train,

  family = "binomial",

  alpha = 0.10,

  nfolds = 5,

  type.measure = "auc",

  standardize = FALSE

)


cat(
  "Best lambda:",
  cv_model$lambda.min,
  "\n"
)

cat(
  "1-SE lambda:",
  cv_model$lambda.1se,
  "\n\n"
)


# ------------------------------------------------------------
# 12. FINAL MODEL
# ------------------------------------------------------------

final_model <- glmnet(

  x_train_scaled,

  y_train,

  family = "binomial",

  alpha = 0.10,

  lambda = cv_model$lambda.min,

  standardize = FALSE

)

cat(
  "Final model trained successfully.\n\n"
)


# ------------------------------------------------------------
# 13. PREDICTIONS
# ------------------------------------------------------------

test_prob <- as.numeric(
  predict(
    final_model,
    newx = x_test_scaled,
    type = "response"
  )
)


# ------------------------------------------------------------
# 14. VALIDATION
# ------------------------------------------------------------

if (any(is.na(test_prob))) {

  stop(
    "Model produced NA predictions."
  )
}

if (
  length(test_prob) !=
    nrow(test_data)
) {

  stop(
    paste(
      "Wrong number of predictions.",
      "Expected:",
      nrow(test_data),
      "Got:",
      length(test_prob)
    )
  )
}


cat("Prediction summary:\n")

print(
  summary(test_prob)
)

cat("\n")


# ------------------------------------------------------------
# 15. SUBMISSION
# ------------------------------------------------------------

submission_exp17 <- data.frame(

  anonymised_id =
    test_data$anonymised_id,

  employed_status =
    test_prob
)


# ------------------------------------------------------------
# 16. SAVE
# ------------------------------------------------------------

output_file <-
  "submission_exp17_ridge_heavy.csv"

write.csv(
  submission_exp17,
  output_file,
  row.names = FALSE,
  quote = FALSE
)


# ------------------------------------------------------------
# 17. FINAL
# ------------------------------------------------------------

cat("============================================\n")
cat("EXPERIMENT 17 COMPLETE\n")
cat("============================================\n\n")

cat(
  "Saved:",
  output_file,
  "\n"
)

cat(
  "Rows:",
  nrow(submission_exp17),
  "\n\n"
)

cat("First 10 predictions:\n")

print(
  head(
    submission_exp17,
    10
  )
)

cat("\n============================================\n")
cat("READY FOR KAGGLE SUBMISSION\n")
cat("============================================\n")

EXPERIMENT 17 — RIDGE-HEAVY ELASTIC NET



Warning message:
"package 'glmnet' was built under R version 4.4.3"
Loading required package: Matrix

Loaded glmnet 4.1-10



Training rows: 16719 
Testing rows : 3230 

Training rows after removing missing target: 16653 

Number of features: 19 

Target distribution:
y_train
    0     1 
11375  5278 

CROSS-VALIDATING RIDGE-HEAVY MODEL

Best lambda: 0.001274537 
1-SE lambda: 0.1216607 

Final model trained successfully.

Prediction summary:
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
 0.1602  0.3000  0.3202  0.3494  0.3428  0.7110 

EXPERIMENT 17 COMPLETE

Saved: submission_exp17_ridge_heavy.csv 
Rows: 3230 

First 10 predictions:
   anonymised_id employed_status
1          ID_10       0.3093458
2         ID_100       0.3047217
3        ID_1000       0.3138539
4        ID_1001       0.3138539
5        ID_1002       0.3138539
6        ID_1003       0.3138539
7        ID_1004       0.3138539
8        ID_1005       0.3138539
9        ID_1006       0.3138539
10       ID_1007       0.3138539

READY FOR KAGGLE SUBMISSION


In [2]:
# ============================================================
# EXPERIMENT 18 — REGULARISED LOGISTIC + CATEGORICAL FEATURES
# BASED ON EXP 5
# ============================================================

cat("============================================\n")
cat("EXPERIMENT 18 — CATEGORICAL + HISTORY MODEL\n")
cat("============================================\n\n")


# ------------------------------------------------------------
# 1. PACKAGE
# ------------------------------------------------------------

if (!requireNamespace("glmnet", quietly = TRUE)) {
  install.packages(
    "glmnet",
    repos = "https://cloud.r-project.org"
  )
}

library(glmnet)


# ------------------------------------------------------------
# 2. LOAD DATA
# ------------------------------------------------------------

data_dir <- "../../assets/dataset"

train_data <- read.csv(
  file.path(data_dir, "train.csv"),
  stringsAsFactors = FALSE
)

test_data <- read.csv(
  file.path(data_dir, "test.csv"),
  stringsAsFactors = FALSE
)

cat("Training rows:", nrow(train_data), "\n")
cat("Testing rows :", nrow(test_data), "\n\n")


# ------------------------------------------------------------
# 3. REMOVE MISSING TARGET
# ------------------------------------------------------------

train_data <- train_data[
  !is.na(train_data$employed_status),
]

cat(
  "Training rows after removing missing target:",
  nrow(train_data),
  "\n\n"
)


# ------------------------------------------------------------
# 4. SAFE NUMERIC
# ------------------------------------------------------------

safe_numeric <- function(x) {

  suppressWarnings(
    as.numeric(as.character(x))
  )

}


# ------------------------------------------------------------
# 5. NQF EXTRACTION
# ------------------------------------------------------------

extract_nqf <- function(x) {

  x <- as.character(x)

  result <- rep(
    NA_real_,
    length(x)
  )

  for (i in seq_along(x)) {

    value <- x[i]

    if (
      is.na(value) ||
      value == ""
    ) {
      next
    }

    match_nqf <- regexec(
      "NQF Level ([0-9]+)",
      value,
      ignore.case = TRUE
    )

    parts <- regmatches(
      value,
      match_nqf
    )

    if (
      length(parts[[1]]) >= 2
    ) {

      result[i] <- suppressWarnings(
        as.numeric(parts[[1]][2])
      )

      next
    }

    match_grade <- regexec(
      "Grade ([0-9]+)",
      value,
      ignore.case = TRUE
    )

    grade_parts <- regmatches(
      value,
      match_grade
    )

    if (
      length(grade_parts[[1]]) >= 2
    ) {

      grade <- suppressWarnings(
        as.numeric(grade_parts[[1]][2])
      )

      if (!is.na(grade)) {

        result[i] <- max(
          grade - 8,
          1
        )

      }
    }
  }

  result
}


# ------------------------------------------------------------
# 6. NUMERIC FEATURES
# ------------------------------------------------------------

create_numeric_features <- function(data) {

  x <- data.frame(

    age =
      safe_numeric(data$age),

    school_quintile =
      safe_numeric(data$school_quintile),

    work_readiness_score =
      safe_numeric(
        data$work_readiness_score
      ),

    total_historical_rounds =
      safe_numeric(
        data$total_historical_rounds
      ),

    lag_round =
      safe_numeric(
        data$lag_round
      ),

    days_since_last_obs =
      safe_numeric(
        data$days_since_last_obs
      ),

    tenure_lag =
      safe_numeric(
        data$tenure_lag
      ),

    employed_lag =
      safe_numeric(
        data$employed_lag
      ),

    education_nqf =
      extract_nqf(
        data$education_level
      )
  )


  # Missingness indicators

  x$has_previous_observation <-
    as.numeric(
      !is.na(data$lag_round)
    )

  x$has_employed_history <-
    as.numeric(
      !is.na(data$employed_lag)
    )

  x$has_tenure_history <-
    as.numeric(
      !is.na(data$tenure_lag)
    )

  x$has_work_readiness <-
    as.numeric(
      !is.na(
        data$work_readiness_score
      )
    )

  x$has_days_since_observation <-
    as.numeric(
      !is.na(
        data$days_since_last_obs
      )
    )


  # Historical employment strength

  employed_lag_num <-
    safe_numeric(
      data$employed_lag
    )

  rounds <-
    safe_numeric(
      data$total_historical_rounds
    )

  x$employment_history_strength <-
    ifelse(
      is.na(employed_lag_num),
      0,
      employed_lag_num *
        pmin(
          ifelse(
            is.na(rounds),
            0,
            rounds
          ),
          5
        )
    )


  # Recent observation

  days <-
    safe_numeric(
      data$days_since_last_obs
    )

  x$recent_observation <-
    ifelse(
      is.na(days),
      0,
      as.numeric(
        days <= 180
      )
    )


  # Long history

  x$long_history <-
    ifelse(
      is.na(rounds),
      0,
      as.numeric(
        rounds >= 3
      )
    )


  # Age squared

  age <-
    safe_numeric(data$age)

  x$age_squared <-
    ifelse(
      is.na(age),
      NA,
      age^2
    )


  # Work readiness squared

  wr <-
    safe_numeric(
      data$work_readiness_score
    )

  x$work_readiness_squared <-
    ifelse(
      is.na(wr),
      NA,
      wr^2
    )


  # Clean impossible values

  x$age[
    x$age < 15 |
      x$age > 80
  ] <- NA

  x$school_quintile[
    x$school_quintile < 0 |
      x$school_quintile > 5
  ] <- NA

  x$work_readiness_score[
    x$work_readiness_score < 0 |
      x$work_readiness_score > 1
  ] <- NA


  x
}


# ------------------------------------------------------------
# 7. CATEGORICAL FEATURES
# ------------------------------------------------------------

categorical_columns <- c(
  "gender",
  "race",
  "province",
  "education_level",
  "institution_type",
  "education_field"
)


create_categorical_features <- function(
  train,
  test
) {

  combined <- rbind(
    train[, categorical_columns, drop = FALSE],
    test[, categorical_columns, drop = FALSE]
  )


  for (column in categorical_columns) {

    combined[[column]] <-
      as.character(
        combined[[column]]
      )

    combined[[column]][
      is.na(combined[[column]]) |
        combined[[column]] == ""
    ] <- "MISSING"

  }


  # Collapse rare categories.

  for (column in categorical_columns) {

    counts <- table(
      combined[[column]]
    )

    rare_levels <- names(
      counts[counts < 20]
    )

    combined[[column]][
      combined[[column]] %in%
        rare_levels
    ] <- "OTHER"

  }


  # Make all levels identical.

  for (column in categorical_columns) {

    combined[[column]] <-
      factor(
        combined[[column]]
      )

  }


  train_cat <- combined[
    1:nrow(train),
    ,
    drop = FALSE
  ]

  test_cat <- combined[
    (nrow(train) + 1):
      nrow(combined),
    ,
    drop = FALSE
  ]


  list(
    train = train_cat,
    test = test_cat
  )
}


# ------------------------------------------------------------
# 8. CREATE FEATURES
# ------------------------------------------------------------

numeric_train <-
  create_numeric_features(
    train_data
  )

numeric_test <-
  create_numeric_features(
    test_data
  )


categorical_features <-
  create_categorical_features(
    train_data,
    test_data
  )


train_cat <-
  categorical_features$train

test_cat <-
  categorical_features$test


cat(
  "Numeric features:",
  ncol(numeric_train),
  "\n"
)

cat(
  "Categorical features:",
  ncol(train_cat),
  "\n\n"
)


# ------------------------------------------------------------
# 9. IMPUTE NUMERIC FEATURES
# ------------------------------------------------------------

for (column in names(numeric_train)) {

  median_value <- median(
    numeric_train[[column]],
    na.rm = TRUE
  )

  if (
    is.na(median_value) ||
    is.infinite(median_value)
  ) {

    median_value <- 0

  }


  numeric_train[[column]][
    is.na(numeric_train[[column]])
  ] <- median_value


  numeric_test[[column]][
    is.na(numeric_test[[column]])
  ] <- median_value
}


# ------------------------------------------------------------
# 10. COMBINE TRAIN + TEST
# ------------------------------------------------------------

all_features <- rbind(
  numeric_train,
  numeric_test
)


all_categorical <- rbind(
  train_cat,
  test_cat
)


# ------------------------------------------------------------
# 11. ONE-HOT ENCODE
# ------------------------------------------------------------

cat(
  "Creating one-hot encoded matrix...\n"
)

combined_model_data <- cbind(
  all_features,
  all_categorical
)


feature_matrix <- model.matrix(
  ~ . - 1,
  data = combined_model_data
)


# ------------------------------------------------------------
# 12. SPLIT MATRIX
# ------------------------------------------------------------

n_train <- nrow(train_data)

x_train <- feature_matrix[
  1:n_train,
  ,
  drop = FALSE
]

x_test <- feature_matrix[
  (n_train + 1):
    nrow(feature_matrix),
  ,
  drop = FALSE
]


cat(
  "Final feature matrix:",
  ncol(x_train),
  "features\n\n"
)


# ------------------------------------------------------------
# 13. REMOVE ZERO-VARIANCE FEATURES
# ------------------------------------------------------------

feature_sd <- apply(
  x_train,
  2,
  sd
)

keep_features <-
  !is.na(feature_sd) &
  feature_sd > 0

x_train <- x_train[
  ,
  keep_features,
  drop = FALSE
]

x_test <- x_test[
  ,
  keep_features,
  drop = FALSE
]


cat(
  "Features after removing zero variance:",
  ncol(x_train),
  "\n\n"
)


# ------------------------------------------------------------
# 14. TARGET
# ------------------------------------------------------------

y_train <- safe_numeric(
  train_data$employed_status
)

y_train <- ifelse(
  y_train > 0,
  1,
  0
)


cat("Target distribution:\n")

print(
  table(y_train)
)

cat("\n")


# ------------------------------------------------------------
# 15. SCALE
# ------------------------------------------------------------

train_means <- colMeans(
  x_train
)

train_sds <- apply(
  x_train,
  2,
  sd
)

train_sds[
  is.na(train_sds) |
    train_sds == 0
] <- 1


x_train_scaled <- sweep(
  x_train,
  2,
  train_means,
  "-"
)

x_train_scaled <- sweep(
  x_train_scaled,
  2,
  train_sds,
  "/"
)


x_test_scaled <- sweep(
  x_test,
  2,
  train_means,
  "-"
)

x_test_scaled <- sweep(
  x_test_scaled,
  2,
  train_sds,
  "/"
)


# ------------------------------------------------------------
# 16. CROSS VALIDATION
# ------------------------------------------------------------

cat("============================================\n")
cat("CROSS-VALIDATING MODEL\n")
cat("============================================\n\n")

set.seed(42)

cv_model <- cv.glmnet(

  x_train_scaled,

  y_train,

  family = "binomial",

  alpha = 0.25,

  nfolds = 5,

  type.measure = "auc",

  standardize = FALSE

)


cat(
  "Best lambda:",
  cv_model$lambda.min,
  "\n"
)

cat(
  "1-SE lambda:",
  cv_model$lambda.1se,
  "\n\n"
)


# ------------------------------------------------------------
# 17. FINAL MODEL
# ------------------------------------------------------------

final_model <- glmnet(

  x_train_scaled,

  y_train,

  family = "binomial",

  alpha = 0.25,

  lambda = cv_model$lambda.min,

  standardize = FALSE
)


cat(
  "Final model trained successfully.\n\n"
)


# ------------------------------------------------------------
# 18. PREDICT
# ------------------------------------------------------------

cat("============================================\n")
cat("GENERATING ROUND 9 PREDICTIONS\n")
cat("============================================\n\n")


test_prob <- as.numeric(
  predict(
    final_model,
    newx = x_test_scaled,
    type = "response"
  )
)


# ------------------------------------------------------------
# 19. VALIDATE
# ------------------------------------------------------------

if (any(is.na(test_prob))) {

  stop(
    "Model produced NA predictions."
  )

}

if (
  length(test_prob) !=
    nrow(test_data)
) {

  stop(
    paste(
      "Wrong number of predictions.",
      "Expected:",
      nrow(test_data),
      "Got:",
      length(test_prob)
    )
  )
}


cat("Prediction summary:\n")

print(
  summary(test_prob)
)

cat("\n")


# ------------------------------------------------------------
# 20. SUBMISSION
# ------------------------------------------------------------

submission_exp18 <- data.frame(

  anonymised_id =
    test_data$anonymised_id,

  employed_status =
    test_prob
)


output_file <-
  "submission_exp18_categorical_regularised.csv"


write.csv(
  submission_exp18,
  output_file,
  row.names = FALSE,
  quote = FALSE
)


# ------------------------------------------------------------
# 21. FINAL
# ------------------------------------------------------------

cat("============================================\n")
cat("EXPERIMENT 18 COMPLETE\n")
cat("============================================\n\n")

cat(
  "Saved:",
  output_file,
  "\n"
)

cat(
  "Rows:",
  nrow(submission_exp18),
  "\n\n"
)

cat("First 10 predictions:\n")

print(
  head(
    submission_exp18,
    10
  )
)

cat("\n============================================\n")
cat("READY FOR KAGGLE SUBMISSION\n")
cat("============================================\n")

EXPERIMENT 18 — CATEGORICAL + HISTORY MODEL

Training rows: 16719 
Testing rows : 3230 

Training rows after removing missing target: 16653 

Numeric features: 19 
Categorical features: 6 

Creating one-hot encoded matrix...
Final feature matrix: 127 features

Features after removing zero variance: 127 

Target distribution:
y_train
    0     1 
11375  5278 

CROSS-VALIDATING MODEL

Best lambda: 0.0191942 
1-SE lambda: 0.0486643 

Final model trained successfully.

GENERATING ROUND 9 PREDICTIONS

Prediction summary:
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
 0.1367  0.2343  0.2570  0.3168  0.3690  0.7097 

EXPERIMENT 18 COMPLETE

Saved: submission_exp18_categorical_regularised.csv 
Rows: 3230 

First 10 predictions:
   anonymised_id employed_status
1          ID_10       0.2281346
2         ID_100       0.2251167
3        ID_1000       0.2311908
4        ID_1001       0.2311908
5        ID_1002       0.2311908
6        ID_1003       0.2311908
7        ID_1004       0.2311908
8  

In [3]:
# ============================================================
# EXPERIMENT 19 — HISTORICAL STABILITY
# REGULARISED LOGISTIC REGRESSION
# ============================================================

cat("============================================\n")
cat("EXPERIMENT 19 — HISTORICAL STABILITY MODEL\n")
cat("============================================\n\n")


# ------------------------------------------------------------
# 1. PACKAGE
# ------------------------------------------------------------

if (!requireNamespace("glmnet", quietly = TRUE)) {
  install.packages(
    "glmnet",
    repos = "https://cloud.r-project.org"
  )
}

library(glmnet)


# ------------------------------------------------------------
# 2. LOAD DATA
# ------------------------------------------------------------

data_dir <- "../../assets/dataset"

train_data <- read.csv(
  file.path(data_dir, "train.csv"),
  stringsAsFactors = FALSE
)

test_data <- read.csv(
  file.path(data_dir, "test.csv"),
  stringsAsFactors = FALSE
)

cat("Training rows:", nrow(train_data), "\n")
cat("Testing rows :", nrow(test_data), "\n\n")


# ------------------------------------------------------------
# 3. REMOVE MISSING TARGET
# ------------------------------------------------------------

train_data <- train_data[
  !is.na(train_data$employed_status),
]

cat(
  "Training rows after removing missing target:",
  nrow(train_data),
  "\n\n"
)


# ------------------------------------------------------------
# 4. SAFE NUMERIC
# ------------------------------------------------------------

safe_numeric <- function(x) {

  suppressWarnings(
    as.numeric(as.character(x))
  )

}


# ------------------------------------------------------------
# 5. NQF EXTRACTION
# ------------------------------------------------------------

extract_nqf <- function(x) {

  x <- as.character(x)

  result <- rep(
    NA_real_,
    length(x)
  )

  for (i in seq_along(x)) {

    value <- x[i]

    if (
      is.na(value) ||
      value == ""
    ) {
      next
    }

    match_nqf <- regexec(
      "NQF Level ([0-9]+)",
      value,
      ignore.case = TRUE
    )

    parts <- regmatches(
      value,
      match_nqf
    )

    if (
      length(parts[[1]]) >= 2
    ) {

      result[i] <- suppressWarnings(
        as.numeric(parts[[1]][2])
      )

      next
    }

    match_grade <- regexec(
      "Grade ([0-9]+)",
      value,
      ignore.case = TRUE
    )

    grade_parts <- regmatches(
      value,
      match_grade
    )

    if (
      length(grade_parts[[1]]) >= 2
    ) {

      grade <- suppressWarnings(
        as.numeric(grade_parts[[1]][2])
      )

      if (!is.na(grade)) {

        result[i] <- max(
          grade - 8,
          1
        )

      }
    }
  }

  result
}


# ------------------------------------------------------------
# 6. FEATURE ENGINEERING
# ------------------------------------------------------------

create_features <- function(data) {

  age <-
    safe_numeric(data$age)

  quintile <-
    safe_numeric(data$school_quintile)

  wr <-
    safe_numeric(
      data$work_readiness_score
    )

  rounds <-
    safe_numeric(
      data$total_historical_rounds
    )

  lag_round <-
    safe_numeric(
      data$lag_round
    )

  days <-
    safe_numeric(
      data$days_since_last_obs
    )

  tenure <-
    safe_numeric(
      data$tenure_lag
    )

  employed <-
    safe_numeric(
      data$employed_lag
    )


  # ----------------------------------------------------------
  # Core variables
  # ----------------------------------------------------------

  x <- data.frame(

    age = age,

    school_quintile = quintile,

    work_readiness_score = wr,

    total_historical_rounds = rounds,

    lag_round = lag_round,

    days_since_last_obs = days,

    tenure_lag = tenure,

    employed_lag = employed,

    education_nqf =
      extract_nqf(
        data$education_level
      )
  )


  # ----------------------------------------------------------
  # Missingness
  # ----------------------------------------------------------

  x$has_previous_observation <-
    as.numeric(
      !is.na(lag_round)
    )

  x$has_employed_history <-
    as.numeric(
      !is.na(employed)
    )

  x$has_tenure_history <-
    as.numeric(
      !is.na(tenure)
    )

  x$has_work_readiness <-
    as.numeric(
      !is.na(wr)
    )

  x$has_days_since_observation <-
    as.numeric(
      !is.na(days)
    )


  # ----------------------------------------------------------
  # HISTORY DEPTH
  # ----------------------------------------------------------

  x$history_depth_2plus <-
    as.numeric(
      !is.na(rounds) &
        rounds >= 2
    )

  x$history_depth_3plus <-
    as.numeric(
      !is.na(rounds) &
        rounds >= 3
    )

  x$history_depth_5plus <-
    as.numeric(
      !is.na(rounds) &
        rounds >= 5
    )


  # ----------------------------------------------------------
  # RECENCY
  # ----------------------------------------------------------

  x$recent_30 <-
    ifelse(
      is.na(days),
      0,
      as.numeric(days <= 30)
    )

  x$recent_90 <-
    ifelse(
      is.na(days),
      0,
      as.numeric(days <= 90)
    )

  x$recent_180 <-
    ifelse(
      is.na(days),
      0,
      as.numeric(days <= 180)
    )

  x$recent_365 <-
    ifelse(
      is.na(days),
      0,
      as.numeric(days <= 365)
    )


  # ----------------------------------------------------------
  # EMPLOYMENT × HISTORY
  # ----------------------------------------------------------

  x$employed_history_weighted <-
    ifelse(
      is.na(employed),
      0,
      employed *
        log1p(
          pmax(
            rounds,
            0
          )
        )
    )


  x$employment_recency_weighted <-
    ifelse(
      is.na(employed) |
        is.na(days),
      0,
      employed *
        exp(
          -days / 365
        )
    )


  # ----------------------------------------------------------
  # EMPLOYMENT × RECENT OBSERVATION
  # ----------------------------------------------------------

  x$recent_employment <-
    ifelse(
      is.na(employed) |
        is.na(days),
      0,
      employed *
        as.numeric(
          days <= 180
        )
    )


  x$very_recent_employment <-
    ifelse(
      is.na(employed) |
        is.na(days),
      0,
      employed *
        as.numeric(
          days <= 90
        )
    )


  # ----------------------------------------------------------
  # UNEMPLOYMENT HISTORY
  # ----------------------------------------------------------

  x$historical_unemployment <-
    ifelse(
      is.na(employed),
      0,
      1 - employed
    )


  x$recent_unemployment <-
    ifelse(
      is.na(employed) |
        is.na(days),
      0,
      (1 - employed) *
        as.numeric(
          days <= 180
        )
    )


  # ----------------------------------------------------------
  # TENURE × EMPLOYMENT
  # ----------------------------------------------------------

  x$employment_tenure <-
    ifelse(
      is.na(employed) |
        is.na(tenure),
      0,
      employed *
        tenure
    )


  # ----------------------------------------------------------
  # AGE NONLINEARITY
  # ----------------------------------------------------------

  x$age_squared <-
    ifelse(
      is.na(age),
      NA,
      age^2
    )


  # ----------------------------------------------------------
  # WORK READINESS NONLINEARITY
  # ----------------------------------------------------------

  x$work_readiness_squared <-
    ifelse(
      is.na(wr),
      NA,
      wr^2
    )


  # ----------------------------------------------------------
  # CLEAN IMPOSSIBLE VALUES
  # ----------------------------------------------------------

  x$age[
    x$age < 15 |
      x$age > 80
  ] <- NA


  x$school_quintile[
    x$school_quintile < 0 |
      x$school_quintile > 5
  ] <- NA


  x$work_readiness_score[
    x$work_readiness_score < 0 |
      x$work_readiness_score > 1
  ] <- NA


  x
}


# ------------------------------------------------------------
# 7. CREATE FEATURES
# ------------------------------------------------------------

x_train_df <-
  create_features(
    train_data
  )

x_test_df <-
  create_features(
    test_data
  )


cat(
  "Number of features:",
  ncol(x_train_df),
  "\n\n"
)


# ------------------------------------------------------------
# 8. CHECK STRUCTURE
# ------------------------------------------------------------

if (
  !identical(
    names(x_train_df),
    names(x_test_df)
  )
) {

  stop(
    "Training and testing feature names do not match."
  )

}


# ------------------------------------------------------------
# 9. IMPUTATION
# ------------------------------------------------------------

for (column in names(x_train_df)) {

  median_value <- median(
    x_train_df[[column]],
    na.rm = TRUE
  )

  if (
    is.na(median_value) ||
    is.infinite(median_value)
  ) {

    median_value <- 0

  }


  x_train_df[[column]][
    is.na(x_train_df[[column]])
  ] <- median_value


  x_test_df[[column]][
    is.na(x_test_df[[column]])
  ] <- median_value
}


# ------------------------------------------------------------
# 10. REMOVE ZERO-VARIANCE FEATURES
# ------------------------------------------------------------

feature_sd <- sapply(
  x_train_df,
  sd
)

keep <- !is.na(feature_sd) &
  feature_sd > 0

x_train_df <-
  x_train_df[
    ,
    keep,
    drop = FALSE
  ]

x_test_df <-
  x_test_df[
    ,
    keep,
    drop = FALSE
  ]


# ------------------------------------------------------------
# 11. SCALE
# ------------------------------------------------------------

train_means <- sapply(
  x_train_df,
  mean
)

train_sds <- sapply(
  x_train_df,
  sd
)

train_sds[
  is.na(train_sds) |
    train_sds == 0
] <- 1


x_train_scaled <- sweep(
  as.matrix(x_train_df),
  2,
  train_means,
  "-"
)

x_train_scaled <- sweep(
  x_train_scaled,
  2,
  train_sds,
  "/"
)


x_test_scaled <- sweep(
  as.matrix(x_test_df),
  2,
  train_means,
  "-"
)

x_test_scaled <- sweep(
  x_test_scaled,
  2,
  train_sds,
  "/"
)


# ------------------------------------------------------------
# 12. TARGET
# ------------------------------------------------------------

y_train <-
  safe_numeric(
    train_data$employed_status
  )

y_train <- ifelse(
  y_train > 0,
  1,
  0
)


cat("Target distribution:\n")

print(
  table(y_train)
)

cat("\n")


# ------------------------------------------------------------
# 13. CROSS-VALIDATED ELASTIC NET
# ------------------------------------------------------------

cat("============================================\n")
cat("CROSS-VALIDATING MODEL\n")
cat("============================================\n\n")


set.seed(42)

cv_model <- cv.glmnet(

  x_train_scaled,

  y_train,

  family = "binomial",

  alpha = 0.25,

  nfolds = 5,

  type.measure = "auc",

  standardize = FALSE
)


cat(
  "Best lambda:",
  cv_model$lambda.min,
  "\n"
)

cat(
  "1-SE lambda:",
  cv_model$lambda.1se,
  "\n\n"
)


# ------------------------------------------------------------
# 14. FINAL MODEL
# ------------------------------------------------------------

final_model <- glmnet(

  x_train_scaled,

  y_train,

  family = "binomial",

  alpha = 0.25,

  lambda = cv_model$lambda.min,

  standardize = FALSE
)


cat(
  "Model trained successfully.\n\n"
)


# ------------------------------------------------------------
# 15. COEFFICIENTS
# ------------------------------------------------------------

coef_matrix <-
  as.matrix(
    coef(final_model)
  )

coef_table <- data.frame(

  feature =
    rownames(coef_matrix),

  coefficient =
    as.numeric(
      coef_matrix[, 1]
    ),

  row.names = NULL
)


coef_table <-
  coef_table[
    order(
      abs(
        coef_table$coefficient
      ),
      decreasing = TRUE
    ),
  ]


cat("Top features:\n\n")

print(
  head(
    coef_table,
    20
  )
)

cat("\n")


# ------------------------------------------------------------
# 16. PREDICTIONS
# ------------------------------------------------------------

cat("============================================\n")
cat("GENERATING ROUND 9 PREDICTIONS\n")
cat("============================================\n\n")


test_prob <- as.numeric(
  predict(
    final_model,
    newx = x_test_scaled,
    type = "response"
  )
)


# ------------------------------------------------------------
# 17. VALIDATION
# ------------------------------------------------------------

if (
  any(is.na(test_prob))
) {

  stop(
    "Model produced NA predictions."
  )

}


if (
  length(test_prob) !=
    nrow(test_data)
) {

  stop(
    paste(
      "Wrong number of predictions.",
      "Expected:",
      nrow(test_data),
      "Got:",
      length(test_prob)
    )
  )

}


cat("Prediction summary:\n")

print(
  summary(test_prob)
)

cat("\n")


# ------------------------------------------------------------
# 18. SUBMISSION
# ------------------------------------------------------------

submission_exp19 <- data.frame(

  anonymised_id =
    test_data$anonymised_id,

  employed_status =
    test_prob
)


output_file <-
  "submission_exp19_historical_stability.csv"


write.csv(
  submission_exp19,
  output_file,
  row.names = FALSE,
  quote = FALSE
)


# ------------------------------------------------------------
# 19. COMPLETE
# ------------------------------------------------------------

cat("============================================\n")
cat("EXPERIMENT 19 COMPLETE\n")
cat("============================================\n\n")

cat(
  "Saved:",
  output_file,
  "\n"
)

cat(
  "Rows:",
  nrow(submission_exp19),
  "\n\n"
)

cat("First 10 predictions:\n")

print(
  head(
    submission_exp19,
    10
  )
)

cat("\n")

cat("Prediction summary:\n")

print(
  summary(
    submission_exp19$employed_status
  )
)

cat("\n============================================\n")
cat("READY FOR KAGGLE SUBMISSION\n")
cat("============================================\n")

EXPERIMENT 19 — HISTORICAL STABILITY MODEL

Training rows: 16719 
Testing rows : 3230 

Training rows after removing missing target: 16653 

Number of features: 30 

Target distribution:
y_train
    0     1 
11375  5278 

CROSS-VALIDATING MODEL

Best lambda: 0.00335348 
1-SE lambda: 0.03767044 

Model trained successfully.

Top features:

                       feature coefficient
1                  (Intercept) -0.78756754
22 employment_recency_weighted  0.24538559
4         work_readiness_score  0.13021837
2                          age  0.08442832
3              school_quintile  0.06990341
8                   tenure_lag -0.05900281
26           employment_tenure  0.05232767
25         recent_unemployment -0.04940818
21   employed_history_weighted  0.04204397
6                    lag_round  0.04084550
13          has_tenure_history  0.04048863
20                  recent_365 -0.04017081
14          has_work_readiness -0.02104374
10               education_nqf  0.01490263
7          day